<a href="https://colab.research.google.com/github/fred-ykv/Value-Investing-In-Python/blob/master/Revis%C3%A3o%20-%20TOP_NEW_FINVIZ_2026_Atualizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# CÉLULA 1 — Setup & Utils
# =========================
import re, math, time, warnings
warnings.filterwarnings("ignore")

import pandas as pd
from bs4 import BeautifulSoup as BS
from urllib.request import Request, urlopen

# --- Memória simples do último ticker digitado ---
LAST = {"ticker": None}

def get_or_ask_ticker(prompt="Digite o ticker da ação: "):
    """
    Retorna o ticker já armazenado em LAST['ticker'].
    Se ainda não houver, pergunta ao usuário uma única vez.
    """
    if LAST["ticker"]:
        return LAST["ticker"]
    t = input(prompt).strip().upper()
    LAST["ticker"] = t
    return t

def finviz_url(ticker:str) -> str:
    return f"https://finviz.com/quote.ashx?t={str(ticker).lower()}"

def fetch_html(url:str, retries:int=2, sleep_s:float=1.0):
    """
    Baixa HTML com cabeçalho de navegador e pequenas tentativas.
    Retorna BeautifulSoup ou None.
    """
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    for i in range(retries+1):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=20) as resp:
                html = resp.read()
            return BS(html, "html.parser")
        except Exception as e:
            if i == retries:
                print(f"Erro ao baixar {url}: {e}")
                return None
            time.sleep(sleep_s)

# --------- Conversores úteis (finanças) ----------
def percent_to_float(s):
    """'12.3%' -> 0.123 ; '-', '' -> None"""
    if s is None: return None
    s = str(s).strip()
    if not s or s == '-': return None
    try:
        return float(s.replace('%','').replace(',',''))/100.0
    except:
        return None

def money_to_number(s):
    """
    '1.2B'->1_200_000_000 ; '750M'->750_000_000 ; '45.6K'->45_600 ; '3.4T'->3_400_000_000_000
    '$1,234.56'->1234.56 ; '-' -> None
    """
    if s is None: return None
    s = str(s).strip().replace('$','').replace(',','')
    if not s or s == '-': return None
    m = re.match(r"^(-?\d+(\.\d+)?)([KMBT])?$", s, re.IGNORECASE)
    if m:
        val = float(m.group(1))
        suf = (m.group(3) or '').upper()
        mult = {'K':1e3,'M':1e6,'B':1e9,'T':1e12}.get(suf,1.0)
        return val*mult
    try:
        return float(s)
    except:
        return None

def to_number(s):
    """Genérico: remove vírgulas e tenta float; '-' -> None"""
    if s is None: return None
    s = str(s).strip().replace(',','')
    if not s or s == '-': return None
    try:
        return float(s)
    except:
        return None

# --------- Helpers de parsing Finviz ----------
def table_2col_to_pairs(table_bs):
    """Converte uma tabela 2-colunas do Finviz em lista [(attr,val)]."""
    out = []
    for tr in table_bs.find_all('tr'):
        tds = tr.find_all('td')
        if len(tds) == 2:
            a = tds[0].get_text(strip=True)
            v = tds[1].get_text(strip=True)
            if a:
                out.append((a, v))
    return out

def robust_find_table_by_headers(soup, headers_expected):
    """
    Busca a tabela que contenha pelo menos 2 headers esperados (ex.: ['P/E','Market Cap']).
    Retorna o elemento <table> ou None.
    """
    tables = soup.find_all('table')
    best, best_hits = None, 0
    for t in tables:
        text = t.get_text(" ", strip=True)
        hits = sum(1 for h in headers_expected if h in text)
        if hits > best_hits:
            best, best_hits = t, hits
    return best if best_hits >= 2 else None

def clean_insiders_df(df):
    """Normaliza colunas comuns da tabela de insiders do Finviz (se existirem)."""
    if df is None or df.empty: return df
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    # tenta setar índice como Date se existir
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.set_index('Date')
    # normalizações numéricas
    for c in ['# Shares','Value ($)','Cost','# Shares Total']:
        if c in df.columns:
            df[c] = df[c].astype(str).str.replace(',','', regex=False)
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def pairs_to_df(pairs, col1='Atributo', col2='Valor'):
    return pd.DataFrame(pairs, columns=[col1, col2])


In [2]:
# =========================
# CÉLULA 2 — Dependências
# =========================
import sys, importlib, subprocess

def ensure(pkg_import, pip_spec=None):
    """
    Tenta importar. Se falhar, instala via pip e reimporta.
    pkg_import: nome do pacote para import (ex.: 'yfinance')
    pip_spec: especificação para pip (ex.: 'yfinance>=0.2.40')
    """
    try:
        return importlib.import_module(pkg_import)
    except ImportError:
        if pip_spec is None:
            pip_spec = pkg_import
        print(f"Instalando {pip_spec} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_spec])
        return importlib.import_module(pkg_import)

# Pacotes principais que vamos usar
yf         = ensure("yfinance", "yfinance>=0.2.40")
# pyfolio original (Quantopian) está abandonado. Usaremos o fork mantido:
try:
    pyfolio = ensure("pyfolio", "pyfolio-reloaded>=0.9.8")
except Exception as e:
    print("Aviso: não foi possível instalar pyfolio-reloaded agora. Seguimos sem pyfolio.")
    pyfolio = None

# (Opcional) útil em alguns fluxos financeiros
pdr        = ensure("pandas_datareader", "pandas-datareader>=0.10.0")

# Verificação rápida de versões
print("Versões carregadas:")
print(" - yfinance:", getattr(yf, "__version__", ""))
print(" - pandas-datareader:", getattr(pdr, "__version__", ""))
print(" - pyfolio:", getattr(pyfolio, "__version__", "N/D") if pyfolio else "N/D")


Instalando pyfolio-reloaded>=0.9.8 ...
Versões carregadas:
 - yfinance: 0.2.66
 - pandas-datareader: 0.10.0
 - pyfolio: 0.9.9


In [3]:
# =========================
# CÉLULA 3 — Config & Compat
# =========================
import matplotlib.pyplot as plt

# Já importamos pandas (pd) e BeautifulSoup (BS) na CÉLULA 1.
# Se você tiver código antigo usando `soup`, criamos um alias:
from bs4 import BeautifulSoup as BS  # garantia de estar no namespace
soup = BS  # alias de compatibilidade p/ código legado: soup(...) == BS(...)

# pandas-datareader: já garantimos a instalação na CÉLULA 2 como `pdr`
# Se quiser usar padrão "web = data", criamos um alias de comodidade:
try:
    from pandas_datareader import data as web
except Exception:
    # fallback usando o módulo já carregado via ensure()
    web = None
    print("Aviso: pandas_datareader.data não pôde ser importado; use `pdr.DataReader`")

# seaborn é opcional; se quiser estilo visual, pode ativar:
try:
    import seaborn as sns
    sns.set_theme()  # opcional: deixa os gráficos com tema agradável
except Exception:
    sns = None
    print("Aviso: seaborn não disponível; seguiremos apenas com matplotlib")

# Opções de exibição do pandas
import pandas as pd
pd.set_option('display.max_colwidth', 25)
pd.set_option('display.width', 120)
pd.set_option('display.max_rows', 200)

# Grid padrão nos gráficos do matplotlib (pode alterar depois por gráfico)
plt.rcParams['axes.grid'] = True
plt.rcParams['figure.figsize'] = (10, 6)

print("Config pronta: `soup` alias criado, `web` disponível (se importado), pandas options set.")


Config pronta: `soup` alias criado, `web` disponível (se importado), pandas options set.


In [4]:
# =========================
# CÉLULA 4 — Finance & PyFolio (ajustada)
# =========================
import warnings
warnings.filterwarnings("ignore")

# yfinance e pyfolio foram instalados na CÉLULA 2.
try:
    import yfinance as yf
except ImportError:
    print("Aviso: yfinance não importado; use ensure() da CÉLULA 2 se precisar reinstalar.")

try:
    import pyfolio as pf
except ImportError:
    pf = None
    print("Aviso: pyfolio não disponível neste ambiente. Seguiremos sem ele.")

# --- Compatibilidade pdr_override ---
try:
    # Versões novas do yfinance movem o método para submódulo pdr
    if hasattr(yf, "pdr_override"):
        yf.pdr_override()
    else:
        from yfinance import pdr
        pdr.pdr_override()
    print("✅ Integração com pandas_datareader ativada via yfinance.")
except Exception as e:
    print(f"Aviso: não foi possível aplicar pdr_override: {e}")

print("✅ yfinance e pyfolio prontos (ou ignorados se ausentes).")


Aviso: não foi possível aplicar pdr_override: cannot import name 'pdr' from 'yfinance' (/usr/local/lib/python3.12/dist-packages/yfinance/__init__.py)
✅ yfinance e pyfolio prontos (ou ignorados se ausentes).


In [5]:
# =========================
# CÉLULA 5 — Entrada do ticker & HTML bootstrap
# =========================

# 1) Entrada manual (como você quer), com fallback seguro
try:
    symbol = input("Enter a ticker: ").strip().upper()
    if not symbol:
        raise ValueError("Ticker vazio")
except Exception as e:
    print("Erro ao capturar ticker. Definindo MLI como padrão.")
    symbol = "MLI"

# 2) Salva na 'memória' para uso nas próximas células sem perguntar de novo
try:
    LAST["ticker"] = symbol
except NameError:
    # se alguém pulou a CÉLULA 1, evita crash
    LAST = {"ticker": symbol}

print(f"Ticker selecionado: {symbol}")

# 3) Monta URL do Finviz e baixa o HTML agora (para reuso nas próximas células)
try:
    url = finviz_url(symbol)      # da CÉLULA 1
    html = fetch_html(url)        # da CÉLULA 1
    if html is None:
        print("⚠️ Não foi possível obter o HTML do Finviz. Verifique o ticker/conexão.")
    else:
        print(f"HTML do Finviz carregado com sucesso: {url}")
except Exception as e:
    html = None
    print(f"Erro ao inicializar HTML do Finviz: {e}")


Enter a ticker: MLI
Ticker selecionado: MLI
HTML do Finviz carregado com sucesso: https://finviz.com/quote.ashx?t=mli


In [6]:
# =========================
# CÉLULA 6 — Validar/Recarregar HTML do Finviz
# =========================

# Garante que temos symbol e html disponíveis
try:
    symbol
except NameError:
    # Se alguém pulou a CÉLULA 5, pergunta uma vez e salva
    symbol = get_or_ask_ticker("Enter a ticker: ")
    LAST["ticker"] = symbol

# Se html não existe ou falhou, recarrega com nossas helpers
try:
    html
except NameError:
    html = None

if html is None:
    url = finviz_url(symbol)     # https + construção padronizada
    html = fetch_html(url)
else:
    # ainda assim define url para referência/prints
    url = finviz_url(symbol)

# Checagens rápidas de sanidade
if html is None:
    raise RuntimeError("Não foi possível obter o HTML do Finviz. Verifique conexão/ticker e tente novamente.")

title = html.title.get_text(strip=True) if html.title else "(sem <title>)"
tables = html.find_all("table")
news_tbl = html.find("table", class_="fullview-news-outer")

print("=== Finviz HTML OK ===")
print(f"URL: {url}")
print(f"Title: {title}")
print(f"Nº de <table>: {len(tables)}")
print(f"Tem tabela de notícias? {'sim' if news_tbl else 'não'}")


=== Finviz HTML OK ===
URL: https://finviz.com/quote.ashx?t=mli
Title: MLI - Mueller Industries Inc Stock Price and Quote
Nº de <table>: 16
Tem tabela de notícias? sim


In [7]:
# =========================
# CÉLULA 7 — Coleta de dados históricos do Yahoo Finance
# =========================
import datetime as dt

print("\n📈 Coletando dados históricos do Yahoo Finance...\n")

# ✅ Usa o ticker atual da sessão (CÉLULA 5)
try:
    symbol
except NameError:
    symbol = get_or_ask_ticker()
    LAST["ticker"] = symbol

# 🔧 Define período padrão (pode ajustar depois)
start_date = dt.date(2020, 1, 1)
end_date = dt.date.today()

try:
    stock_data = yf.download(symbol, start=start_date, end=end_date)

    if stock_data.empty:
        raise ValueError("Nenhum dado retornado. Verifique o ticker ou o intervalo.")

    print(f"✅ Dados do Yahoo Finance para {symbol}:")
    print(stock_data.head(5))
    print(f"\nPeríodo: {start_date} → {end_date}")
    print(f"Total de registros: {len(stock_data)}")

except Exception as e:
    stock_data = pd.DataFrame()
    print(f"⚠️ Erro ao baixar dados de {symbol}: {e}")



📈 Coletando dados históricos do Yahoo Finance...



[*********************100%***********************]  1 of 1 completed

✅ Dados do Yahoo Finance para MLI:
Price           Close       High        Low       Open  Volume
Ticker            MLI        MLI        MLI        MLI     MLI
Date                                                          
2020-01-02  14.493054  14.639449  14.378684  14.598275  297600
2020-01-03  14.488478  14.497628  14.241437  14.255161  421200
2020-01-06  14.332937  14.424434  14.300914  14.410709  318400
2020-01-07  14.227713  14.351233  14.159090  14.314635  259200
2020-01-08  14.181968  14.310063  14.159094  14.195693  209800

Período: 2020-01-01 → 2026-06-23
Total de registros: 1625


In [8]:
# =========================
# CÉLULA 8 — Coleta e limpeza de transações de insiders (Finviz)
# =========================
print("\n👔 Coletando transações de insiders...\n")

def get_insider_from_html(html_soup):
    """
    Extrai e normaliza a tabela de transações de insiders do Finviz.
    Retorna DataFrame limpo ou None.
    """
    try:
        # Encontra todas as tabelas com classe 'body-table'
        tables = pd.read_html(str(html_soup), attrs={'class': 'body-table'})
        if not tables:
            print("⚠️ Nenhuma tabela 'body-table' encontrada.")
            return None

        insider_df = tables[0].copy()

        # Remove a primeira linha se for cabeçalho duplicado
        if insider_df.iloc[0, 0].lower().startswith("date"):
            insider_df = insider_df.iloc[1:]

        # Define colunas padronizadas (caso batam em número)
        if len(insider_df.columns) == 9:
            insider_df.columns = [
                "Trader", "Relationship", "Date", "Transaction",
                "Cost", "# Shares", "Value ($)", "# Shares Total", "SEC Form 4"
            ]

        # Reorganiza a ordem
        insider_df = insider_df[
            ["Date", "Trader", "Relationship", "Transaction", "Cost",
             "# Shares", "Value ($)", "# Shares Total", "SEC Form 4"]
        ]

        # Usa utilitário da CÉLULA 1 para normalizar
        insider_df = clean_insiders_df(insider_df)

        return insider_df

    except Exception as e:
        print(f"⚠️ Erro ao extrair insiders: {e}")
        return None


# Executa extração
insider = get_insider_from_html(html)

if insider is not None and not insider.empty:
    print("✅ Transações de insiders encontradas:\n")
    display(insider.head(10))
else:
    print("⚠️ Nenhuma transação de insider encontrada.")



👔 Coletando transações de insiders...

✅ Transações de insiders encontradas:



,Trader,Relationship,Transaction,Cost,# Shares,Value ($),# Shares Total,SEC Form 4
Date,,,,,,,,
NaT,Insider Trading,Relationship,Transaction,NaN,NaN,NaN,NaN,SEC Form 4
2026-05-29,GOLDMAN SCOTT JAY,Director,Sale,127.91,2000.0,255820.0,40867.0,May 29 03:22 PM
2026-05-29,GOLDMAN SCOTT JAY,Director,Proposed Sale,127.91,2000.0,255820.0,NaN,May 29 10:06 AM
2026-05-01,GLADSTEIN GARY S,Director,Option Exercise,12.62,9778.0,123447.0,31797.0,May 04 03:44 PM
2026-04-27,Christopher Gregory L.,Chairman of the Board...,Sale,137.29,103266.0,14177389.0,804911.0,Apr 29 11:29 AM
2026-04-27,GREGORY CHRISTOPHER,PRESIDENT/ CEO,Proposed Sale,138.00,200000.0,27600000.0,NaN,Apr 27 02:07 PM
2026-02-13,GOLDMAN SCOTT JAY,Director,Sale,118.97,4430.0,527037.0,41645.0,Feb 13 02:26 PM
2026-02-13,GOLDMAN SCOTT JAY,Director,Proposed Sale,119.97,4430.0,531467.0,NaN,Feb 13 10:43 AM
2026-02-09,HANSEN JOHN B,Director,Proposed Sale,116.40,1000.0,116404.0,NaN,Feb 09 02:15 PM


In [9]:
# =========================
# CÉLULA 9 — Coleta de notícias (Finviz)
# =========================
import pandas as pd
from datetime import datetime

print("\n📰 Coletando notícias do Finviz...\n")

def _normalize_finviz_news_datetime(series):
    """
    Finviz mistura linhas com 'MMM-DD-YY HH:MMAM' e outras só com 'HH:MMAM'.
    Esta função faz forward-fill da DATA para as linhas que só têm hora.
    Retorna pd.Series com strings de datetime completas.
    """
    out = []
    last_date_str = None
    for s in series:
        s = (s or "").strip()
        # Se contém letras do mês, assume que traz a data completa (ex.: 'May-12-25 07:00AM')
        has_month_letters = any(ch.isalpha() for ch in s)
        if has_month_letters:
            # guarda parte de data (até o espaço antes da hora)
            parts = s.split()
            if len(parts) >= 2:
                last_date_str = parts[0]  # 'May-12-25'
                out.append(s)
            else:
                # algo inesperado; passa adiante
                out.append(s)
        else:
            # só horário (ex.: '07:00AM'), então anexa a última data conhecida
            if last_date_str:
                out.append(f"{last_date_str} {s}")
            else:
                out.append(s)  # sem contexto; mantém como está
    return pd.Series(out)

def get_news_from_html(html_soup, base_url="https://finviz.com"):
    """
    Extrai notícias do bloco 'fullview-news-outer' no Finviz.
    Retorna DataFrame com colunas: ['Datetime','Headline','Link'] (index em Datetime).
    """
    try:
        news_table = html_soup.find('table', class_='fullview-news-outer')
        if news_table is None:
            print("⚠️ Tabela de notícias ('fullview-news-outer') não encontrada.")
            return None

        items = []
        rows = news_table.find_all('tr')
        for r in rows:
            date_td = r.find('td', class_='nn-date')
            a = r.find('a', class_='tab-link-news')
            if not a:
                continue
            dt_text = date_td.get_text(strip=True) if date_td else ""
            headline = a.get_text(strip=True)
            href = a.get('href', '')
            # completa link relativo
            if href.startswith('/'):
                href = base_url + href
            items.append({"DateText": dt_text, "Headline": headline, "Link": href})

        if not items:
            print("⚠️ Nenhum item de notícia encontrado.")
            return None

        df = pd.DataFrame(items)

        # Normaliza datas: forward-fill da data para linhas com apenas horário
        df["DateText"] = _normalize_finviz_news_datetime(df["DateText"])

        # Converte para datetime (formato Finviz costuma ser 'May-12-25 07:00AM')
        # Tenta com o formato conhecido; se falhar, usa to_datetime flexível
        def parse_dt(s):
            s = (s or "").strip()
            for fmt in ("%b-%d-%y %I:%M%p", "%b-%d-%y %H:%M", "%b-%d-%Y %I:%M%p"):
                try:
                    return datetime.strptime(s, fmt)
                except Exception:
                    pass
            # fallback flexível
            try:
                return pd.to_datetime(s, errors='coerce')
            except Exception:
                return pd.NaT

        df["Datetime"] = df["DateText"].apply(parse_dt)
        df = df.dropna(subset=["Datetime"]).copy()
        df = df.sort_values("Datetime").reset_index(drop=True)

        # Organiza colunas finais
        df = df[["Datetime", "Headline", "Link"]].set_index("Datetime")

        return df

    except Exception as e:
        print(f"⚠️ Erro ao extrair notícias: {e}")
        return None


# Executa com o `html` já carregado nas células anteriores
news_df = get_news_from_html(html)

if news_df is not None and not news_df.empty:
    print("✅ Notícias extraídas:")
    display(news_df.tail(15))
else:
    print("⚠️ Nenhuma notícia disponível para este ticker no momento.")



📰 Coletando notícias do Finviz...

⚠️ Nenhuma notícia disponível para este ticker no momento.


In [10]:
# =========================
# CÉLULA 10 — Extração e limpeza dos fundamentos (Finviz) — REVISADA
# =========================
print("\n🏛️ Extraindo fundamentos do Finviz (snapshot)...\n")

def _pairs_from_snapshot_table(table_bs):
    """
    Converte uma tabela 'snapshot' do Finviz em lista [(Atributo, Valor)].
    A snapshot tem várias <td> por linha; extraímos em pares (0,1), (2,3)...
    """
    pairs = []
    for tr in table_bs.find_all("tr"):
        tds = tr.find_all("td")
        # anda de 2 em 2
        for i in range(0, len(tds)-1, 2):
            a = tds[i].get_text(strip=True)
            v = tds[i+1].get_text(strip=True)
            if a and v:
                pairs.append((a, v))
    return pairs

def _smart_convert(v):
    v = str(v).strip()
    if "%" in v:
        return percent_to_float(v)
    num = money_to_number(v)  # $, K/M/B/T
    if num is not None:
        return num
    return to_number(v)

def get_fundamentals_from_html(html_soup, preview='full'):
    """
    Prioriza tabelas 'snapshot-table2' ou 'snapshot-table'.
    Se não achar, procura por tabela com termos típicos (P/E, Market Cap, EPS),
    ignorando a de notícias ('fullview-news-outer') e insiders ('body-table').
    Retorna DataFrame com ['Atributo','Valor','ValorNumérico'].
    """
    # 1) tentativa direta por classe
    table = html_soup.find('table', class_='snapshot-table2')
    if table is None:
        table = html_soup.find('table', class_='snapshot-table')

    # 2) heurística, se necessário
    if table is None:
        expected = ["P/E", "Market Cap", "EPS", "Dividend", "ROE", "Debt/Eq"]
        candidates = []
        for t in html_soup.find_all("table"):
            cls = (t.get("class") or [])
            # pular explicitamente notícias e insiders
            if "fullview-news-outer" in cls or "body-table" in cls:
                continue
            text = t.get_text(" ", strip=True)
            hits = sum(1 for h in expected if h in text)
            if hits >= 2:
                candidates.append((hits, t))
        if candidates:
            candidates.sort(key=lambda x: x[0], reverse=True)
            table = candidates[0][1]

    if table is None:
        print("❌ Não foi possível localizar a tabela de snapshot do Finviz.")
        return None

    # 3) extrai pares atributo/valor da snapshot
    pairs = _pairs_from_snapshot_table(table)
    if not pairs:
        print("❌ Snapshot encontrada, mas não foi possível extrair pares atributo/valor.")
        return None

    df = pd.DataFrame(pairs, columns=["Atributo", "Valor"])
    # remove duplicatas preservando a primeira ocorrência (mantém a ordem)
    df = df[~df["Atributo"].duplicated(keep="first")].reset_index(drop=True)

    # 4) coluna numérica
    df["ValorNumérico"] = df["Valor"].apply(_smart_convert)

    # 5) pré-visualização
    if preview == 'head':
        print(f"✅ {len(df)} atributos extraídos (preview):")
        display(df.head(15))
    elif preview == 'full':
        print(f"✅ {len(df)} atributos extraídos (visual clássico):")
        with pd.option_context('display.max_rows', None, 'display.max_colwidth', 60, 'display.width', 140):
            display(df[["Atributo","Valor"]])
    else:
        # sem preview
        pass

    return df

# Executa com o html já carregado
fund_df = get_fundamentals_from_html(html, preview='full')

if fund_df is not None and not fund_df.empty:
    print("\n✅ Fundamentais prontos (coluna ValorNumérico disponível para cálculos).")
else:
    print("\n⚠️ Nenhum dado fundamental encontrado.")




🏛️ Extraindo fundamentos do Finviz (snapshot)...

✅ 82 atributos extraídos (visual clássico):


,Atributo,Valor
0,Index,-
1,P/E,18.28
2,EPS (ttm),7.63
3,Insider Own,2.10%
4,Shs Outstand,110.56M
5,Perf Week,1.03%
6,Market Cap,15.43B
7,Forward P/E,16.07
8,EPS next Y,8.68
9,Insider Trans,-4.55%



✅ Fundamentais prontos (coluna ValorNumérico disponível para cálculos).


# **Coletor de Métricas para Valuation**


In [11]:
# =========================
# CÉLULA — Coletor de Métricas p/ Valuation (a partir do fund_df)
# =========================

import re
import pandas as pd

# -- Helpers (usam fund_df) -----------------------
def get_metric(df, key, default=None, like=False):
    """
    Busca o valor de um atributo em fund_df.
    Retorna ValorNumérico (se existir), senão o texto em 'Valor'.
    """
    if df is None or df.empty:
        return default
    mask = df["Atributo"].str.contains(key, case=False, regex=False) if like else (df["Atributo"] == key)
    subset = df.loc[mask]
    if subset.empty:
        return default
    # prioriza número
    val = subset["ValorNumérico"].iloc[0]
    if pd.isna(val):
        val = subset["Valor"].iloc[0]
    return val

def get_percent_metric(df, keys, default=None):
    """
    Procura a primeira métrica (entre 'keys') e retorna fração (0..1).
    Se vier como '12.3%', converte para 0.123.
    Se vier como número >1 e o texto tinha '%', divide por 100.
    """
    if df is None or df.empty:
        return default
    if isinstance(keys, str):
        keys = [keys]

    mask = False
    for k in keys:
        mask = mask | df["Atributo"].str.contains(k, case=False, regex=False)
    subset = df.loc[mask]
    if subset.empty:
        return default

    row = subset.iloc[0]
    # tenta numérico primeiro
    val_num = row.get("ValorNumérico", None)
    txt = str(row.get("Valor", "")).strip()
    if pd.notna(val_num):
        try:
            val_num = float(val_num)
            if "%" in txt and val_num > 1:
                return val_num/100.0
            # se já parece fração, retorna como está
            return val_num if val_num <= 1 else val_num/100.0
        except:
            pass

    # tenta converter do texto
    if "%" in txt:
        try:
            return float(txt.replace('%','').replace(',',''))/100.0
        except:
            return default
    try:
        v = float(txt.replace(',',''))
        return v if v <= 1 else v/100.0
    except:
        return default

def parse_dividend_ttm(df):
    """
    Lê 'Dividend TTM' no formato '0.85 (1.07%)' e retorna:
      (dividend_amount, dividend_yield_frac)
    """
    raw = get_metric(df, "Dividend TTM", default=None, like=True)
    if raw is None:
        # alternativa: 'Dividend' simples (alguns tickers)
        raw = get_metric(df, "Dividend", default=None, like=False)

    div_amount, div_yield = None, None
    if raw is None:
        return div_amount, div_yield

    if isinstance(raw, (int, float)):
        # se já veio como número, assumimos ser o valor do dividendo TTM
        return float(raw), None

    s = str(raw)
    # procura padrão 'valor (x.xx%)'
    m = re.search(r"([0-9]*\.?[0-9]+)\s*\(([-+]?[0-9]*\.?[0-9]+)%\)", s)
    if m:
        try:
            div_amount = float(m.group(1))
        except:
            div_amount = None
        try:
            div_yield = float(m.group(2))/100.0
        except:
            div_yield = None
        return div_amount, div_yield

    # se não tinha parênteses, tenta só número simples para amount
    try:
        div_amount = float(s.replace(',','').replace('$',''))
    except:
        div_amount = None
    return div_amount, None

# -- Coleta das métricas --------------------------
metrics = {}

# Tamanho/Preço
metrics["MarketCap"]     = get_metric(fund_df, "Market Cap", default=None, like=True)
metrics["Price"]         = get_metric(fund_df, "Price", default=None, like=False)
metrics["TargetPrice"]   = get_metric(fund_df, "Target Price", default=None, like=False)
metrics["Beta"]          = get_metric(fund_df, "Beta", default=None, like=False)

# Resultados / Múltiplos / Crescimento
metrics["PE"]            = get_metric(fund_df, "P/E", default=None, like=False)
metrics["ForwardPE"]     = get_metric(fund_df, "Forward P/E", default=None, like=False)
metrics["EPS_ttm"]       = get_metric(fund_df, "EPS (ttm)", default=None, like=False)
metrics["EPS_nextY"]     = get_metric(fund_df, "EPS next Y", default=None, like=False)
metrics["EPS_nextQ"]     = get_metric(fund_df, "EPS next Q", default=None, like=False)
metrics["PEG"]           = get_metric(fund_df, "PEG", default=None, like=False)

# Alavancagem e Liquidez
metrics["DebtEq"]        = get_metric(fund_df, "Debt/Eq", default=None, like=False)
metrics["LT_DebtEq"]     = get_metric(fund_df, "LT Debt/Eq", default=None, like=False)
metrics["CurrentRatio"]  = get_metric(fund_df, "Current Ratio", default=None, like=False)
metrics["QuickRatio"]    = get_metric(fund_df, "Quick Ratio", default=None, like=False)

# Lucratividade / Eficiência (percentuais → fração 0..1)
metrics["ROE"]           = get_percent_metric(fund_df, ["ROE","Return on Equity"], default=None)
metrics["ROI"]           = get_percent_metric(fund_df, ["ROI","Return on Investment"], default=None)
metrics["ROA"]           = get_percent_metric(fund_df, ["ROA","Return on Assets"], default=None)
metrics["ProfitMargin"]  = get_percent_metric(fund_df, ["Profit Margin"], default=None)
metrics["OperMargin"]    = get_percent_metric(fund_df, ["Oper. Margin","Operating Margin"], default=None)
metrics["GrossMargin"]   = get_percent_metric(fund_df, ["Gross Margin"], default=None)

# Dividendos
div_amount, div_yield = parse_dividend_ttm(fund_df)
metrics["DividendTTM"]   = div_amount                          # valor em $ por ação (TTM)
metrics["DividendYield"] = div_yield if div_yield is not None else get_percent_metric(fund_df, ["Dividend"], default=None)
metrics["Payout"]        = get_percent_metric(fund_df, ["Payout","Payout Ratio"], default=None)

# Livros / Fluxos
metrics["BookValuePS"]   = get_metric(fund_df, "Book/sh", default=None, like=False)
metrics["CashPS"]        = get_metric(fund_df, "Cash/sh", default=None, like=False)
metrics["FCF_multiple"]  = get_metric(fund_df, "P/FCF", default=None, like=False)

# Vendas e Receita
metrics["Sales"]         = get_metric(fund_df, "Sales", default=None, like=False)            # pode ser '3.92B'
metrics["PS"]            = get_metric(fund_df, "P/S", default=None, like=False)
metrics["SalesPast5Y"]   = get_percent_metric(fund_df, ["Sales past 5Y"], default=None)
metrics["SalesYY_TTM"]   = get_percent_metric(fund_df, ["Sales Y/Y TTM"], default=None)
metrics["SalesQQ"]       = get_percent_metric(fund_df, ["Sales Q/Q"], default=None)

# EPS growth
metrics["EPS_thisY"]     = get_percent_metric(fund_df, ["EPS this Y"], default=None)
metrics["EPS_past5Y"]    = get_percent_metric(fund_df, ["EPS past 5Y"], default=None)
metrics["EPS_QQ"]        = get_percent_metric(fund_df, ["EPS Q/Q"], default=None)
metrics["EPS_YY_TTM"]    = get_percent_metric(fund_df, ["EPS Y/Y TTM"], default=None)
metrics["EPS_next5Y"]    = get_percent_metric(fund_df, ["EPS next 5Y"], default=None)

# Outras úteis
metrics["InstOwn"]       = get_percent_metric(fund_df, ["Inst Own"], default=None)
metrics["InsiderOwn"]    = get_percent_metric(fund_df, ["Insider Own"], default=None)
metrics["ShortFloat"]    = get_percent_metric(fund_df, ["Short Float"], default=None)
metrics["ShortRatio"]    = get_metric(fund_df, "Short Ratio", default=None, like=False)

# -- Saída organizada ----------------------------
# Converte para DataFrame “vertical”
met_df = (
    pd.DataFrame.from_dict(metrics, orient='index', columns=['Value'])
      .reset_index()
      .rename(columns={'index':'Metric'})
)

# Coluna amigável (formatada): % e $ onde fizer sentido
def fmt_row(row):
    m, v = row['Metric'], row['Value']
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "-"
    # percentuais (frações)
    if m in ["ROE","ROI","ROA","ProfitMargin","OperMargin","GrossMargin",
             "DividendYield","Payout","SalesPast5Y","SalesYY_TTM","SalesQQ",
             "EPS_thisY","EPS_past5Y","EPS_QQ","EPS_YY_TTM","EPS_next5Y",
             "InstOwn","InsiderOwn","ShortFloat"]:
        return f"{v*100:.2f}%"
    # valores grandes em dinheiro
    if m in ["MarketCap","Sales"]:
        try:
            return f"${v:,.0f}"
        except:
            return str(v)
    # price-like
    if m in ["Price","TargetPrice","DividendTTM","BookValuePS","CashPS"]:
        try:
            return f"${v:,.2f}"
        except:
            return str(v)
    # genéricos numéricos
    if isinstance(v, (int,float)):
        return f"{v:,.4f}" if abs(v) < 100 else f"{v:,.2f}"
    return str(v)

met_df["Pretty"] = met_df.apply(fmt_row, axis=1)

print(f"📦 Métricas coletadas para {LAST.get('ticker','?')}:")
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 60, 'display.width', 130):
    display(met_df.sort_values("Metric").reset_index(drop=True))


📦 Métricas coletadas para MLI:


,Metric,Value,Pretty
0,Beta,1.090000e+00,1.0900
1,BookValuePS,3.016000e+01,$30.16
2,CashPS,1.273000e+01,$12.73
3,CurrentRatio,5.350000e+00,5.3500
4,DebtEq,1.000000e-02,0.0100
5,DividendTTM,1.200000e+00,$1.20
6,DividendYield,8.600000e-03,0.86%
7,EPS_QQ,5.479000e-01,54.79%
8,EPS_YY_TTM,3.842000e-01,38.42%
9,EPS_next5Y,1.024000e-01,10.24%


# **Cálculos direcionados**

In [12]:
# =========================
# CÉLULA — bLL, gLL, Ganho de Alavancagem e GAF (DFL)
# =========================

import pandas as pd
import math

def _as_float(x):
    try:
        return float(x)
    except Exception:
        return None

def _pct(x):
    """Converte fração (0..1) para string percentual; lida com None."""
    if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
        return "-"
    return f"{x*100:.2f}%"

# --- Lê métricas base (todas frações) ---
payout = _as_float(metrics.get("Payout"))          # fração 0..1
roe    = _as_float(metrics.get("ROE"))             # fração 0..1
roi    = _as_float(metrics.get("ROI"))             # fração 0..1

# --- Cálculos com salvaguardas ---
# bLL = 1 - payout
bLL = None if payout is None else (1 - payout)

# gLL = bLL * ROE
gLL = None
if (bLL is not None) and (roe is not None):
    gLL = bLL * roe

# Ganho de Alavancagem = ROE - ROI
leverage_gain = None
if (roe is not None) and (roi is not None):
    leverage_gain = roe - roi

# GAF (DFL) = ROE / ROI
DFL = None
if (roe is not None) and (roi is not None) and (roi != 0):
    DFL = roe / roi

# --- Saída organizada ---
result = pd.DataFrame(
    [
        ["Payout (entrada)",    payout,         _pct(payout)],
        ["bLL (1 - payout)",    bLL,            _pct(bLL)],
        ["ROE (entrada)",       roe,            _pct(roe)],
        ["ROI (entrada)",       roi,            _pct(roi)],
        ["gLL = bLL * ROE",     gLL,            _pct(gLL)],
        ["Ganho de Alavancagem (ROE - ROI)", leverage_gain, _pct(leverage_gain)],
        ["GAF / DFL = ROE / ROI", DFL,          f"{DFL:.2f}" if DFL is not None else "-"],
    ],
    columns=["Métrica", "Valor (fração)", "Valor (%)"]
)

print(f"Métricas derivadas — {LAST.get('ticker','?')}")
display(result)

# --- Variáveis disponíveis para uso posterior ---
# bLL, gLL, leverage_gain, DFL


Métricas derivadas — MLI


,Métrica,Valor (fração),Valor (%)
0,Payout (entrada),0.145700,14.57%
1,bLL (1 - payout),0.854300,85.43%
2,ROE (entrada),0.282200,28.22%
3,ROI (entrada),0.252800,25.28%
4,gLL = bLL * ROE,0.241083,24.11%
5,Ganho de Alavancagem ...,0.029400,2.94%
6,GAF / DFL = ROE / ROI,1.116297,1.12


# **Taxas Macroeconômicas (Rf e Inflação EUA / Brasil)**

In [13]:
# =========================
# CÉLULA — Macro consolidado (Rf US, CPI US, CPI BR, S&P500 10y)
# =========================
import re, math
import pandas as pd
import numpy as np
from urllib.request import Request, urlopen
from bs4 import BeautifulSoup as BS
import datetime as dt

# ---------- Helpers HTTP ----------
def _fetch_html(url, timeout=15):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    req = Request(url, headers=headers)
    with urlopen(req, timeout=timeout) as resp:
        return resp.read()

def _as_frac(pct_str):
    s = str(pct_str).strip().replace('%','').replace(',','.')
    try:
        return float(s)/100.0
    except:
        return None

# ---------- YCharts: taxas genéricas ----------
def get_ycharts_rate(urls, label="Indicador"):
    """
    Busca taxa do YCharts (retorna fração 0..1 e string '%').
    Tenta a primeira URL que responder com estrutura esperada.
    """
    for url in urls:
        try:
            html = _fetch_html(url)
            soup = BS(html, "html.parser")
            value_elem = soup.find('div', {'class': 'key-stat-title'})
            if value_elem:
                val_txt = value_elem.get_text(strip=True).split(' ')[0].replace('%','').replace(',','.')
                val = float(val_txt)
                print(f"{label}: {val:.2f}% ({url})")
                return val/100.0, f"{val:.2f}%"
        except Exception as e:
            print(f"⚠️ {label} falhou em {url}: {e}")
    return None, None

# ---------- Inflação Brasil: múltiplas fontes ----------
def get_inflation_br_globalrates():
    url = "https://www.global-rates.com/en/inflation/cpi/67/brazil/"
    try:
        soup = BS(_fetch_html(url), "html.parser")
        text = soup.get_text(" ", strip=True)
        m = re.search(r"Inflation[^0-9]*([0-9]+[.,]?[0-9]*)\s*%", text, flags=re.IGNORECASE)
        if m:
            val = _as_frac(m.group(1))
            if val is not None:
                print(f"Inflação Brasil (Global-Rates): {val*100:.2f}%")
                return val, f"{val*100:.2f}%"
        percents = re.findall(r"([0-9]+[.,]?[0-9]*)\s*%", text)
        if percents:
            val = _as_frac(percents[0])
            if val is not None and 0 < val < 1:
                print(f"Inflação Brasil (Global-Rates*): {val*100:.2f}%")
                return val, f"{val*100:.2f}%"
    except Exception as e:
        print(f"⚠️ Global-Rates falhou: {e}")
    return None, None

def get_inflation_br_rateinflation():
    url = "https://www.rateinflation.com/inflation-rate/brazil-inflation-rate/"
    try:
        soup = BS(_fetch_html(url), "html.parser")
        text = soup.get_text(" ", strip=True)
        m = re.search(r"Latest Inflation Rate.*?([0-9]+[.,]?[0-9]*)\s*%", text, flags=re.IGNORECASE)
        if m:
            val = _as_frac(m.group(1))
            if val is not None:
                print(f"Inflação Brasil (RateInflation): {val*100:.2f}%")
                return val, f"{val*100:.2f}%"
        percents = re.findall(r"([0-9]+[.,]?[0-9]*)\s*%", text)
        if percents:
            val = _as_frac(percents[0])
            if val is not None and 0 < val < 1:
                print(f"Inflação Brasil (RateInflation*): {val*100:.2f}%")
                return val, f"{val*100:.2f}%"
    except Exception as e:
        print(f"⚠️ RateInflation falhou: {e}")
    return None, None

def get_inflation_br_tradingeconomics():
    url = "https://tradingeconomics.com/brazil/inflation-cpi"
    try:
        soup = BS(_fetch_html(url), "html.parser")
        elem = soup.find('span', {'class': 'indicator-value'}) or soup.find('div', {'class': 'indicator-value'})
        if elem:
            val = _as_frac(elem.get_text(strip=True))
            if val is not None:
                print(f"Inflação Brasil (TradingEconomics): {val*100:.2f}%")
                return val, f"{val*100:.2f}%"
        text = soup.get_text(" ", strip=True)
        m = re.search(r"([0-9]+[.,]?[0-9]*)\s*%", text)
        if m:
            val = _as_frac(m.group(1))
            if val is not None and 0 < val < 1:
                print(f"Inflação Brasil (TradingEconomics*): {val*100:.2f}%")
                return val, f"{val*100:.2f}%"
    except Exception as e:
        print(f"⚠️ TradingEconomics falhou: {e}")
    return None, None

def get_inflation_br_investing():
    url = "https://www.investing.com/economic-calendar/brazil-consumer-price-index-%28cpi%29-yoy-410"
    try:
        soup = BS(_fetch_html(url), "html.parser")
        text = soup.get_text(" ", strip=True)
        m = re.search(r"Actual\s*:\s*([0-9]+[.,]?[0-9]*)\s*%", text, flags=re.IGNORECASE)
        if m:
            val = _as_frac(m.group(1))
            if val is not None:
                print(f"Inflação Brasil (Investing): {val*100:.2f}%")
                return val, f"{val*100:.2f}%"
        m = re.search(r"([0-9]+[.,]?[0-9]*)\s*%", text)
        if m:
            val = _as_frac(m.group(1))
            if val is not None and 0 < val < 1:
                print(f"Inflação Brasil (Investing*): {val*100:.2f}%")
                return val, f"{val*100:.2f}%"
    except Exception as e:
        print(f"⚠️ Investing falhou: {e}")
    return None, None

def get_inflation_br():
    for fn in (get_inflation_br_globalrates,
               get_inflation_br_rateinflation,
               get_inflation_br_tradingeconomics,
               get_inflation_br_investing):
        val, s = fn()
        if val is not None:
            return val, s
    return None, None

# ---------- S&P500: 10 anos ----------
def sp500_returns_10y():
    """
    Retorna (acumulado %, CAGR %, df).
    Usa yf.download("^GSPC", auto_adjust=True) e janela ~10 anos.
    """
    try:
        end = pd.Timestamp.today().normalize()
        start = end - pd.DateOffset(years=10)
        sp = yf.download("^GSPC", start=start, end=end, progress=False, auto_adjust=True)
        if sp is None or sp.empty:
            raise ValueError("Sem dados retornados do ^GSPC.")
        if "Close" not in sp.columns:
            sp["Close"] = sp.iloc[:, 0]
        sp = sp[["Close"]].dropna().asfreq("B").ffill()
        first, last = float(sp["Close"].iloc[0]), float(sp["Close"].iloc[-1])
        years = (sp.index[-1] - sp.index[0]).days / 365.25
        total = (last / first - 1.0) * 100.0
        cagr = ((last / first) ** (1.0 / years) - 1.0) * 100.0
        return total, cagr, sp
    except Exception as e:
        print(f"⚠️ Erro S&P500 10y: {e}")
        return None, None, pd.DataFrame()

# ---------- Orquestração ----------
# 1) Rf EUA (10Y) e Inflação EUA
Rf_val, Rf_str           = get_ycharts_rate(
    ["https://ycharts.com/indicators/10_year_treasury_rate",
     "https://ycharts.com/indicators/us_10year_government_bond_interest_rate"],
    label="Rf EUA (10Y)"
)
infl_US_val, infl_US_str = get_ycharts_rate(
    ["https://ycharts.com/indicators/us_consumer_price_index_yoy"],
    label="Inflação EUA (CPI YoY)"
)

# 2) Inflação Brasil
infl_BR_val, infl_BR_str = get_inflation_br()
if infl_BR_val is None:
    infl_BR_str = "-"

# 3) S&P 500 — 10y
Ten_year_return, mean_yearly_return, sp500_series = sp500_returns_10y()

# ---------- Consolidação ----------
macro_rows = [
    ("Rf EUA (10Y Treasury)",       Rf_val,           (f"{Rf_val*100:.2f}%" if Rf_val is not None else "-")),
    ("Inflação EUA (CPI YoY)",      infl_US_val,      (f"{infl_US_val*100:.2f}%" if infl_US_val is not None else "-")),
    ("Inflação Brasil (IPCA YoY)",  infl_BR_val,      infl_BR_str),
    ("S&P 500 — Retorno 10y (acum.)", Ten_year_return/100.0 if Ten_year_return is not None else None,
                                      (f"{Ten_year_return:.2f}%" if Ten_year_return is not None else "-")),
    ("S&P 500 — CAGR 10y (a.a.)",  mean_yearly_return/100.0 if mean_yearly_return is not None else None,
                                      (f"{mean_yearly_return:.2f}%" if mean_yearly_return is not None else "-")),
]

macro_df = pd.DataFrame(macro_rows, columns=["Indicador","Valor (fração)","Valor (%)"])
print("\n📊 Macro consolidado:")
display(macro_df)

# Dicionário para uso no valuation
macro = {
    "Rf_US": Rf_val,                 # fração (0..1)
    "Infl_US": infl_US_val,          # fração
    "Infl_BR": infl_BR_val,          # fração
    "SP500_10y_total_pct": Ten_year_return,     # %
    "SP500_10y_cagr_pct": mean_yearly_return,   # %
}


Rf EUA (10Y): 4.51% (https://ycharts.com/indicators/10_year_treasury_rate)
Inflação EUA (CPI YoY): 4.20% (https://ycharts.com/indicators/us_consumer_price_index_yoy)
Inflação Brasil (Global-Rates*): 4.39%

📊 Macro consolidado:


,Indicador,Valor (fração),Valor (%)
0,Rf EUA (10Y Treasury),0.045100,4.51%
1,Inflação EUA (CPI YoY),0.042000,4.20%
2,Inflação Brasil (IPCA...,0.043900,4.39%
3,S&P 500 — Retorno 10y...,2.536043,253.60%
4,S&P 500 — CAGR 10y (a...,0.134682,13.47%


In [14]:
# =========================
# CÉLULA — Balanço Patrimonial (Zacks) com fallback yfinance
# =========================
import re, math, requests
import pandas as pd
from bs4 import BeautifulSoup as BS

def _parse_number(s):
    """
    Converte strings como '1,234', '(456)', '-', 'N/A', '' -> float ou None.
    Zacks costuma usar números em USD (milhares/milhões) sem sufixos.
    """
    if s is None:
        return None
    s = str(s).strip()
    if not s or s in {'-', '—', '–', 'N/A', 'NA'}:
        return None
    # Parênteses indicam negativo
    neg = False
    if s.startswith('(') and s.endswith(')'):
        neg = True
        s = s[1:-1]
    # Remove símbolos e separadores
    s = s.replace(',', '').replace('$', '')
    # Alguns itens podem ter notas tipo '1234*' ou '1234 †'
    s = re.sub(r'[^0-9.\-]', '', s)
    try:
        v = float(s)
        return -v if neg else v
    except:
        return None

def fetch_zacks_balance_sheet(symbol, timeout=15):
    """
    Tenta extrair a tabela de balanço patrimonial anual do Zacks.
    Retorna (bs_wide, bs_long) ou (None, None) se falhar.
    """
    url = f"https://www.zacks.com/stock/quote/{symbol}/balance-sheet"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    try:
        r = requests.get(url, headers=headers, timeout=timeout)
        r.raise_for_status()
    except Exception as e:
        print(f"⚠️ Falha HTTP ao acessar Zacks: {e}")
        return None, None

    soup = BS(r.content, "html.parser")

    # Procura por tabelas com cabeçalhos de anos (ex.: 2024, 2023...)
    tables = soup.find_all('table')
    candidate = None
    best_hits = 0
    year_pat = re.compile(r'20\d{2}')  # '2020'..'2099'
    for t in tables:
        ths = [th.get_text(" ", strip=True) for th in t.find_all('th')]
        hits = sum(1 for h in ths if year_pat.search(h))
        # tabela boa costuma ter vários anos no cabeçalho
        if hits >= best_hits and hits >= 2:
            candidate, best_hits = t, hits

    if candidate is None:
        print("⚠️ Não encontrei uma tabela com cabeçalhos de anos no Zacks.")
        return None, None

    # Extrai cabeçalhos
    headers = [th.get_text(" ", strip=True) for th in candidate.find_all('th')]
    # Normaliza cabeçalhos: 1ª coluna é a descrição (LineItem), restantes são períodos
    # Alguns cabeçalhos vêm como 'Fiscal Year Ended 12/31/2024' — guardamos apenas a parte com ano/mes/dia
    def _clean_period(h):
        # tenta extrair AAAA-MM ou AAAA-MM-DD ou AAAA
        m = re.search(r'(20\d{2}(?:[-/]\d{1,2}(?:[-/]\d{1,2})?)?)', h)
        return m.group(1) if m else h

    if len(headers) >= 2:
        periods = [_clean_period(h) for h in headers[1:]]
    else:
        print("⚠️ Cabeçalhos insuficientes no Zacks.")
        return None, None

    rows = []
    for tr in candidate.find_all('tr'):
        tds = tr.find_all('td')
        if not tds:
            continue
        line = [td.get_text(" ", strip=True) for td in tds]
        # algumas tabelas têm a descrição em <th> na linha
        if tr.find('th') and not line:
            continue
        # Esperado: [Description, v1, v2, v3, ...]
        if len(line) >= 2:
            desc = line[0]
            vals = [ _parse_number(x) for x in line[1:1+len(periods)] ]
            # completa lista se vier menor
            while len(vals) < len(periods):
                vals.append(None)
            rows.append([desc, *vals])

    if not rows:
        print("⚠️ Linhas não encontradas/parseadas no Zacks.")
        return None, None

    cols = ["LineItem"] + periods
    bs_wide = pd.DataFrame(rows, columns=cols)

    # Remove linhas completamente vazias
    value_cols = cols[1:]
    bs_wide = bs_wide[~bs_wide[value_cols].isna().all(axis=1)].reset_index(drop=True)

    # Formato tidy
    bs_long = (
        bs_wide
        .melt(id_vars=["LineItem"], value_vars=value_cols, var_name="Period", value_name="Value")
        .dropna(subset=["Value"])
        .reset_index(drop=True)
    )

    print(f"✅ Zacks — balanço extraído para {symbol} ({len(bs_wide)} linhas, {len(value_cols)} períodos).")
    return bs_wide, bs_long

def fetch_yf_balance_sheet(symbol, quarterly=False):
    """
    Fallback via yfinance:
      quarterly=False -> Ticker(symbol).balance_sheet (anual)
      quarterly=True  -> quarterly_balance_sheet
    Retorna (bs_wide, bs_long) com Period em ISO 'YYYY-MM-DD'.
    """
    try:
        t = yf.Ticker(symbol)
        df = t.quarterly_balance_sheet if quarterly else t.balance_sheet
        if df is None or df.empty:
            return None, None
        # df vem com index=LineItem e colunas=Datetime (ou pandas Timestamp)
        df = df.copy()
        # transforma colunas para string data
        periods = []
        for c in df.columns:
            try:
                d = pd.to_datetime(c)
                periods.append(d.strftime("%Y-%m-%d"))
            except:
                periods.append(str(c))
        df.columns = periods
        df.reset_index(inplace=True)
        df.rename(columns={'index':'LineItem'}, inplace=True)
        bs_wide = df

        bs_long = (
            bs_wide
            .melt(id_vars=["LineItem"], var_name="Period", value_name="Value")
            .dropna(subset=["Value"])
            .reset_index(drop=True)
        )
        src = "yfinance quarterly" if quarterly else "yfinance annual"
        print(f"✅ Fallback {src} — balanço extraído para {symbol} ({len(bs_wide)} linhas, {len(periods)} períodos).")
        return bs_wide, bs_long
    except Exception as e:
        print(f"⚠️ Fallback yfinance falhou: {e}")
        return None, None

# --------- Orquestração (Zacks -> fallback yfinance) ----------
try:
    symbol
except NameError:
    symbol = get_or_ask_ticker()
    LAST["ticker"] = symbol

bs_wide, bs_long = fetch_zacks_balance_sheet(symbol)

if bs_wide is None:
    print("⚠️ Usando fallback para yfinance (anual)...")
    bs_wide, bs_long = fetch_yf_balance_sheet(symbol, quarterly=False)

if bs_wide is None:
    print("⚠️ Usando fallback para yfinance (trimestral)...")
    bs_wide, bs_long = fetch_yf_balance_sheet(symbol, quarterly=True)

# Saídas finais organizadas
if bs_wide is not None:
    print("\n📘 Balanço — formato LARGO (algumas linhas):")
    display(bs_wide.head(12))

if bs_long is not None:
    print("\n📗 Balanço — formato LONGO (algumas linhas):")
    display(bs_long.head(12))

if bs_wide is None and bs_long is None:
    print("❌ Não foi possível obter o balanço patrimonial por nenhuma fonte.")


⚠️ Falha HTTP ao acessar Zacks: 403 Client Error: Forbidden for url: https://www.zacks.com/stock/quote/MLI/balance-sheet
⚠️ Usando fallback para yfinance (anual)...
✅ Fallback yfinance annual — balanço extraído para MLI (75 linhas, 5 períodos).

📘 Balanço — formato LARGO (algumas linhas):


,LineItem,2025-12-31,2024-12-31,2023-12-31,2022-12-31,2021-12-31
0,Treasury Shares Number,4.918626e+07,4.661488e+07,4.620809e+07,4.636277e+07,NaN
1,Ordinary Shares Number,1.111798e+08,1.137511e+08,1.141579e+08,1.140032e+08,NaN
2,Share Issued,1.603660e+08,1.603660e+08,1.603660e+08,1.603660e+08,NaN
3,Total Debt,2.749000e+07,3.375800e+07,3.555700e+07,2.385100e+07,NaN
4,Tangible Book Value,2.624698e+09,2.155643e+09,2.139417e+09,1.578541e+09,NaN
5,Invested Capital,3.209966e+09,2.774259e+09,2.338426e+09,1.792943e+09,NaN
6,Working Capital,2.032611e+09,1.614242e+09,1.722883e+09,1.186358e+09,NaN
7,Net Tangible Assets,2.624698e+09,2.155643e+09,2.139417e+09,1.578541e+09,NaN
8,Capital Lease Obligat...,2.749000e+07,3.266400e+07,3.457600e+07,2.182200e+07,NaN
9,Common Stock Equity,3.209966e+09,2.773165e+09,2.337445e+09,1.790914e+09,NaN



📗 Balanço — formato LONGO (algumas linhas):


,LineItem,Period,Value
0,Treasury Shares Number,2025-12-31,4.918626e+07
1,Ordinary Shares Number,2025-12-31,1.111798e+08
2,Share Issued,2025-12-31,1.603660e+08
3,Total Debt,2025-12-31,2.749000e+07
4,Tangible Book Value,2025-12-31,2.624698e+09
5,Invested Capital,2025-12-31,3.209966e+09
6,Working Capital,2025-12-31,2.032611e+09
7,Net Tangible Assets,2025-12-31,2.624698e+09
8,Capital Lease Obligat...,2025-12-31,2.749000e+07
9,Common Stock Equity,2025-12-31,3.209966e+09


**Patrimônio Líquido (PL)**

In [15]:
# =========================
# CÉLULA — Patrimônio Líquido (PL) a partir de bs_wide/bs_long (yfinance) + fallback Finviz
# =========================
import math
import pandas as pd

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return str(x)

def _latest_nonnull(series):
    """Retorna o último valor não-nulo em uma Series (ou None)."""
    try:
        s = series.dropna()
        return None if s.empty else s.iloc[-1]
    except:
        return None

def _get_equity_from_bs(bs_wide=None, bs_long=None):
    """
    Procura o PL em nomes comuns. Prioridades:
      1) 'Total Stockholders Equity'
      2) 'Total Shareholder's Equity'
      3) 'Common Stock Equity'
      4) 'Total Equity'
      5) 'Total Equity Gross Minority Interest' (se nada mais houver)
    Retorna (valor, periodo_str, nome_usado) ou (None, None, None).
    """
    if bs_long is None or bs_long.empty:
        if bs_wide is None or bs_wide.empty:
            return None, None, None
        # converte wide -> long para uniformizar
        value_cols = [c for c in bs_wide.columns if c != "LineItem"]
        bs_long = (
            bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                         var_name="Period", value_name="Value")
                    .dropna(subset=["Value"])
        )

    # normaliza nomes
    bs_long = bs_long.copy()
    bs_long["LineItem_norm"] = bs_long["LineItem"].str.strip().str.lower()

    # lista de aliases por prioridade
    candidates = [
        ["total stockholders equity", "total stockholders' equity"],
        ["total shareholder's equity", "total shareholders' equity", "total shareholders equity"],
        ["common stock equity"],
        ["total equity"],
        ["total equity gross minority interest"],  # última opção
    ]

    # ordena períodos
    try:
        bs_long["Period_dt"] = pd.to_datetime(bs_long["Period"], errors="coerce")
    except:
        bs_long["Period_dt"] = pd.NaT

    for names in candidates:
        mask = False
        for n in names:
            mask = mask | bs_long["LineItem_norm"].eq(n)
        sub = bs_long.loc[mask].dropna(subset=["Value"]).copy()
        if sub.empty:
            continue
        # escolhe o período mais recente (com dt válido; se não houver, usa ordem original)
        sub = sub.sort_values(["Period_dt", "Period"], ascending=[True, True])
        val = sub["Value"].iloc[-1]
        per = sub["Period"].iloc[-1]
        used = sub["LineItem"].iloc[-1]
        return float(val), str(per), str(used)

    return None, None, None

def _get_equity_from_finviz(fund_df):
    """
    Estima PL = Book/sh * Shs Outstand, se ambos existirem.
    Finviz: 'Book/sh' e 'Shs Outstand' (ex.: '110.65M').
    """
    if fund_df is None or fund_df.empty:
        return None
    try:
        # helpers de coleta que você já tem:
        def _get(df, key, like=False):
            m = df["Atributo"].str.contains(key, case=False, regex=False) if like else (df["Atributo"]==key)
            sub = df.loc[m]
            if sub.empty: return None
            val = sub["ValorNumérico"].iloc[0]
            return None if pd.isna(val) else float(val)

        book_ps = _get(fund_df, "Book/sh", like=False)         # valor numérico por ação
        shs_out = _get(fund_df, "Shs Outstand", like=True)     # número de ações (já convertido de M/B)
        if book_ps is None or shs_out is None:
            return None
        return float(book_ps) * float(shs_out)
    except Exception:
        return None

# ---- Execução principal ----
pl_val, pl_period, pl_name = _get_equity_from_bs(bs_wide=bs_wide, bs_long=bs_long)

source = "yfinance balance_sheet"
if pl_val is None:
    # tenta via Finviz: Book/sh × Shs Outstand
    pl_est = _get_equity_from_finviz(fund_df if "fund_df" in globals() else None)
    if pl_est is not None:
        pl_val = pl_est
        pl_period = "-"
        pl_name = "Book/sh × Shs Outstand (Finviz, estimado)"
        source = "Finviz (estimado)"
    else:
        print("❌ Não foi possível obter o Patrimônio Líquido por nenhuma fonte.")
        pl_val = None

# ---- Saída ----
if pl_val is not None:
    print(f"✅ Patrimônio Líquido (PL) — {LAST.get('ticker','?')}")
    print(f"• Valor:  {_fmt_money(pl_val)}")
    print(f"• Período: {pl_period}")
    print(f"• Fonte:   {pl_name} [{source}]")


✅ Patrimônio Líquido (PL) — MLI
• Valor:  $3,209,966,000.00
• Período: 2025-12-31
• Fonte:   Common Stock Equity [yfinance balance_sheet]


**Empréstimos e Financiamentos**

In [16]:
# =========================
# CÉLULA — Empréstimos e Financiamentos (curto prazo + dívidas) a partir do balanço
# =========================
import math
import pandas as pd

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return str(x)

def _ensure_long(bs_wide, bs_long):
    """Garante DataFrame longo (LineItem, Period, Value) a partir do wide se preciso."""
    if bs_long is not None and not bs_long.empty:
        return bs_long.copy()
    if bs_wide is None or bs_wide.empty:
        return None
    value_cols = [c for c in bs_wide.columns if c != "LineItem"]
    return (
        bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                     var_name="Period", value_name="Value")
               .dropna(subset=["Value"])
               .reset_index(drop=True)
    )

def _latest_value(bs_long, aliases):
    """
    Procura a 1ª linha cujo LineItem case com algum alias (case-insensitive, igualdade).
    Retorna (valor_float, periodo_str, nome_usado) ou (None, None, None).
    """
    if bs_long is None or bs_long.empty:
        return None, None, None

    df = bs_long.copy()
    df["LineItem_norm"] = df["LineItem"].str.strip().str.lower()
    try:
        df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")
    except:
        df["Period_dt"] = pd.NaT

    mask = False
    for a in aliases:
        mask = mask | df["LineItem_norm"].eq(a.strip().lower())
    sub = df.loc[mask].dropna(subset=["Value"]).copy()
    if sub.empty:
        return None, None, None

    sub = sub.sort_values(["Period_dt", "Period"], ascending=[True, True])
    val = float(sub["Value"].iloc[-1])
    per = str(sub["Period"].iloc[-1])
    used = str(sub["LineItem"].iloc[-1])
    return val, per, used

def _get_metric_from_finviz(fund_df, key, like=True):
    """Fallback: coleta um número do snapshot do Finviz (ex.: 'Total Debt')."""
    try:
        if fund_df is None or fund_df.empty:
            return None
        m = fund_df.copy()
        cond = m["Atributo"].str.contains(key, case=False, regex=False) if like else (m["Atributo"] == key)
        sub = m.loc[cond]
        if sub.empty:
            return None
        val = sub["ValorNumérico"].iloc[0]
        return None if pd.isna(val) else float(val)
    except:
        return None

# ------- Execução -------
bsL = _ensure_long(bs_wide if "bs_wide" in globals() else None,
                   bs_long if "bs_long" in globals() else None)

# Aliases mais comuns em yfinance para as linhas relevantes:
ALIASES_CURR_PORTION_LTD = [
    "current portion of long term debt",
    "current portion of long-term debt",
    "current maturities of long-term debt",
    "current portion long-term debt",
    "current portion long term debt",
    "current debt",  # às vezes vem consolidado
    "current portion of lt debt",
]
ALIASES_LT_DEBT = [
    "long term debt",
    "long-term debt",
    "long term borrowings",
]
ALIASES_TOTAL_DEBT = [
    "total debt",
]
ALIASES_CASH_EQ = [
    "cash and cash equivalents",
    "cash & equivalents",
    "cash and equivalents",
    "cash cash equivalents and short term investments",  # variação yfinance
    "cash and cash equivalents, at carrying value",
]

curr_portion, per_curr, name_curr = _latest_value(bsL, ALIASES_CURR_PORTION_LTD)
lt_debt, per_lt, name_lt           = _latest_value(bsL, ALIASES_LT_DEBT)
tot_debt, per_tot, name_tot        = _latest_value(bsL, ALIASES_TOTAL_DEBT)
cash_eq,  per_cash, name_cash      = _latest_value(bsL, ALIASES_CASH_EQ)

# Se não achou total debt no balanço, tenta Finviz como fallback
if tot_debt is None and "fund_df" in globals():
    finviz_total_debt = _get_metric_from_finviz(fund_df, "Total Debt", like=True)
    if finviz_total_debt is not None:
        tot_debt, per_tot, name_tot = finviz_total_debt, "-", "Total Debt (Finviz snapshot)"

# Cálculos agregados úteis
# Dívida de curto prazo (aproximação): current portion of long-term debt (se existir)
short_term_debt = curr_portion  # se quiser, depois somamos "Short Term Borrowings" se existir no seu balanço

# Dívida líquida (se tivermos caixa)
net_debt = None
if (tot_debt is not None) and (cash_eq is not None):
    net_debt = tot_debt - cash_eq

# ---- Saída organizada ----
rows = [
    ("Empréstimos/Financiamentos CP (Current Portion LTD)", _fmt_money(short_term_debt), per_curr or "-",
     name_curr or "-"),
    ("Dívida de Longo Prazo", _fmt_money(lt_debt), per_lt or "-", name_lt or "-"),
    ("Dívida Total", _fmt_money(tot_debt), per_tot or "-", name_tot or "-"),
    ("Caixa & Equivalentes", _fmt_money(cash_eq), per_cash or "-", name_cash or "-"),
    ("Dívida Líquida (Total - Caixa)", _fmt_money(net_debt), per_tot or "-", "cálculo"),
]

debt_df = pd.DataFrame(rows, columns=["Indicador", "Valor", "Período", "Fonte"])
print(f"💳 Empréstimos e Financiamentos — {LAST.get('ticker','?')}")
with pd.option_context('display.max_colwidth', 70, 'display.width', 130):
    display(debt_df)


💳 Empréstimos e Financiamentos — MLI


,Indicador,Valor,Período,Fonte
0,Empréstimos/Financiamentos CP (Current Portion LTD),"$1,094,000.00",2024-12-31,Current Debt
1,Dívida de Longo Prazo,"$185,000.00",2023-12-31,Long Term Debt
2,Dívida Total,"$27,490,000.00",2025-12-31,Total Debt
3,Caixa & Equivalentes,"$1,367,003,000.00",2025-12-31,Cash And Cash Equivalents
4,Dívida Líquida (Total - Caixa),"$-1,339,513,000.00",2025-12-31,cálculo


**Passivo Total**

In [17]:
# =========================
# CÉLULA — Passivo Total (Total Liabilities) a partir do balanço (yfinance)
# =========================
import math
import pandas as pd

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return str(x)

def _ensure_long(bs_wide, bs_long):
    """Garante DataFrame longo (LineItem, Period, Value) a partir do wide se preciso."""
    if bs_long is not None and not bs_long.empty:
        return bs_long.copy()
    if bs_wide is None or bs_wide.empty:
        return None
    value_cols = [c for c in bs_wide.columns if c != "LineItem"]
    return (
        bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                     var_name="Period", value_name="Value")
               .dropna(subset=["Value"])
               .reset_index(drop=True)
    )

def _latest_value(bs_long, aliases):
    """
    Procura a 1ª linha cujo LineItem seja igual a algum alias (case-insensitive).
    Retorna (valor_float, periodo_str, nome_usado) ou (None, None, None).
    """
    if bs_long is None or bs_long.empty:
        return None, None, None

    df = bs_long.copy()
    df["LineItem_norm"] = df["LineItem"].str.strip().str.lower()
    try:
        df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")
    except:
        df["Period_dt"] = pd.NaT

    mask = False
    for a in aliases:
        mask = mask | df["LineItem_norm"].eq(a.strip().lower())
    sub = df.loc[mask].dropna(subset=["Value"]).copy()
    if sub.empty:
        return None, None, None

    sub = sub.sort_values(["Period_dt", "Period"], ascending=[True, True])
    val = float(sub["Value"].iloc[-1])
    per = str(sub["Period"].iloc[-1])
    used = str(sub["LineItem"].iloc[-1])
    return val, per, used

# ---- exec ----
bsL = _ensure_long(bs_wide if "bs_wide" in globals() else None,
                   bs_long if "bs_long" in globals() else None)

# Aliases comuns no yfinance:
ALIASES_TLIAB = [
    "total liabilities",
    "total liabilities net minority interest",  # aparece bastante
]
ALIASES_TASSETS = [
    "total assets",
]
ALIASES_TEQUITY = [
    "total stockholders equity",
    "total stockholders' equity",
    "total shareholders' equity",
    "total shareholder's equity",
    "common stock equity",
    "total equity",
    "total equity gross minority interest",  # último fallback
]

total_liab, per_liab, name_liab = _latest_value(bsL, ALIASES_TLIAB)
total_assets, per_assets, name_assets = _latest_value(bsL, ALIASES_TASSETS)
total_equity, per_equity, name_equity = _latest_value(bsL, ALIASES_TEQUITY)

print(f"📑 Passivo Total — {LAST.get('ticker','?')}")
print(f"• Valor:   {_fmt_money(total_liab)}")
print(f"• Período: {per_liab or '-'}")
print(f"• Fonte:   {name_liab or '-'}")

# (opcional) checagem contábil A ≈ L + E quando disponíveis
if (total_assets is not None) and (total_liab is not None) and (total_equity is not None):
    diff = total_assets - (total_liab + total_equity)
    print("\n🧮 Checagem (A = L + E):")
    print(f"  Total Assets:     {_fmt_money(total_assets)}  ({name_assets or '-'})")
    print(f"  Total Liabilities:{_fmt_money(total_liab)}     ({name_liab or '-'})")
    print(f"  Total Equity:     {_fmt_money(total_equity)}   ({name_equity or '-'})")
    print(f"  Diferença (A - (L+E)): {_fmt_money(diff)}")


📑 Passivo Total — MLI
• Valor:   $497,123,000.00
• Período: 2025-12-31
• Fonte:   Total Liabilities Net Minority Interest

🧮 Checagem (A = L + E):
  Total Assets:     $3,733,029,000.00  (Total Assets)
  Total Liabilities:$497,123,000.00     (Total Liabilities Net Minority Interest)
  Total Equity:     $3,235,906,000.00   (Total Equity Gross Minority Interest)
  Diferença (A - (L+E)): $0.00


**Dívida de Longo Prazo**

In [18]:
# =========================
# CÉLULA — Dívida de Longo Prazo (a partir do balanço yfinance)
# =========================
import math
import pandas as pd

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return str(x)

def _ensure_long(bs_wide, bs_long):
    """Garante DataFrame longo (LineItem, Period, Value) a partir do wide se preciso."""
    if bs_long is not None and not bs_long.empty:
        return bs_long.copy()
    if bs_wide is None or bs_wide.empty:
        return None
    value_cols = [c for c in bs_wide.columns if c != "LineItem"]
    return (
        bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                     var_name="Period", value_name="Value")
               .dropna(subset=["Value"])
               .reset_index(drop=True)
    )

def _latest_value(bs_long, aliases):
    """
    Procura a 1ª linha cujo LineItem seja igual a algum alias (case-insensitive).
    Retorna (valor_float, periodo_str, nome_usado) ou (None, None, None).
    """
    if bs_long is None or bs_long.empty:
        return None, None, None

    df = bs_long.copy()
    df["LineItem_norm"] = df["LineItem"].str.strip().str.lower()
    try:
        df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")
    except:
        df["Period_dt"] = pd.NaT

    mask = False
    for a in aliases:
        mask = mask | df["LineItem_norm"].eq(a.strip().lower())
    sub = df.loc[mask].dropna(subset=["Value"]).copy()
    if sub.empty:
        return None, None, None

    sub = sub.sort_values(["Period_dt", "Period"], ascending=[True, True])
    val = float(sub["Value"].iloc[-1])
    per = str(sub["Period"].iloc[-1])
    used = str(sub["LineItem"].iloc[-1])
    return val, per, used

# --- prepara df longo
bsL = _ensure_long(bs_wide if "bs_wide" in globals() else None,
                   bs_long if "bs_long" in globals() else None)

# Aliases típicos no yfinance para a dívida de longo prazo
ALIASES_LT_DEBT = [
    "long term debt",
    "long-term debt",
    "long term borrowings",
]
# Total debt como apoio/alternativa
ALIASES_TOTAL_DEBT = [
    "total debt",
]
# Equity (para opcional Debt/Equity)
ALIASES_EQUITY = [
    "total stockholders equity",
    "total stockholders' equity",
    "total shareholders' equity",
    "total shareholder's equity",
    "common stock equity",
    "total equity",
    "total equity gross minority interest",
]

# Busca principal
lt_debt, per_lt, name_lt = _latest_value(bsL, ALIASES_LT_DEBT)

# Fallback informativo: Total Debt (caso LT Debt não exista separado)
tot_debt, per_tot, name_tot = _latest_value(bsL, ALIASES_TOTAL_DEBT)

print(f"🏦 Dívida de Longo Prazo — {LAST.get('ticker','?')}")
print(f"• Long-Term Debt: {_fmt_money(lt_debt)}   (Período: {per_lt or '-'}, Fonte: {name_lt or '-'})")
if lt_debt is None and tot_debt is not None:
    print(f"  ⚠️ Campo 'Long-Term Debt' não encontrado. Usando referência 'Total Debt': {_fmt_money(tot_debt)} (Período: {per_tot or '-'})")

# (Opcional) Debt/Equity se houver PL disponível
equity_val, per_eq, name_eq = _latest_value(bsL, ALIASES_EQUITY)
if (lt_debt is not None) and (equity_val is not None) and equity_val != 0:
    de_ratio = lt_debt / equity_val
    print(f"• Debt/Equity (LT Debt / Equity): {de_ratio:.3f}   (Equity: {_fmt_money(equity_val)} | {name_eq or '-'})")


🏦 Dívida de Longo Prazo — MLI
• Long-Term Debt: $185,000.00   (Período: 2023-12-31, Fonte: Long Term Debt)
• Debt/Equity (LT Debt / Equity): 0.000   (Equity: $3,235,906,000.00 | Total Equity Gross Minority Interest)


**Capital de Terceiros (Po)**

In [19]:
# =========================
# CÉLULA — Capital de Terceiros (Po) a partir do balanço (yfinance) + fallback Finviz
# =========================
import math
import pandas as pd

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return str(x)

def _ensure_long(bs_wide, bs_long):
    """Garante DataFrame longo (LineItem, Period, Value) a partir do wide se preciso."""
    if bs_long is not None and not bs_long.empty:
        return bs_long.copy()
    if bs_wide is None or bs_wide.empty:
        return None
    value_cols = [c for c in bs_wide.columns if c != "LineItem"]
    return (
        bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                     var_name="Period", value_name="Value")
               .dropna(subset=["Value"])
               .reset_index(drop=True)
    )

def _latest_value(bs_long, aliases):
    """
    Procura a 1ª linha cujo LineItem seja igual a algum alias (case-insensitive).
    Retorna (valor_float, periodo_str, nome_usado) ou (None, None, None).
    """
    if bs_long is None or bs_long.empty:
        return None, None, None

    df = bs_long.copy()
    df["LineItem_norm"] = df["LineItem"].str.strip().str.lower()
    try:
        df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")
    except:
        df["Period_dt"] = pd.NaT

    mask = False
    for a in aliases:
        mask = mask | df["LineItem_norm"].eq(a.strip().lower())
    sub = df.loc[mask].dropna(subset=["Value"]).copy()
    if sub.empty:
        return None, None, None

    sub = sub.sort_values(["Period_dt", "Period"], ascending=[True, True])
    val = float(sub["Value"].iloc[-1])
    per = str(sub["Period"].iloc[-1])
    used = str(sub["LineItem"].iloc[-1])
    return val, per, used

def _get_from_finviz(fund_df, key, like=True):
    """Fallback: coleta número do snapshot do Finviz (por ex.: 'Total Debt')."""
    try:
        if fund_df is None or fund_df.empty:
            return None
        cond = fund_df["Atributo"].str.contains(key, case=False, regex=False) if like else (fund_df["Atributo"]==key)
        sub = fund_df.loc[cond]
        if sub.empty:
            return None
        val = sub["ValorNumérico"].iloc[0]
        return None if pd.isna(val) else float(val)
    except:
        return None

# --- prepara o DF longo
bsL = _ensure_long(bs_wide if "bs_wide" in globals() else None,
                   bs_long if "bs_long" in globals() else None)

# Aliases típicos
ALIASES_CURR_LTD = [
    "current portion of long term debt",
    "current portion of long-term debt",
    "current maturities of long-term debt",
    "current portion long-term debt",
    "current portion long term debt",
    "current debt",  # às vezes vem assim
]
ALIASES_ST_BORROW = [
    "short term borrowings",
    "short-term borrowings",
    "short term debt",
    "short-term debt",
]
ALIASES_LT_DEBT = [
    "long term debt",
    "long-term debt",
    "long term borrowings",
]
ALIASES_TOTAL_DEBT = [
    "total debt",
]
ALIASES_EQUITY = [
    "total stockholders equity",
    "total stockholders' equity",
    "total shareholders' equity",
    "total shareholder's equity",
    "common stock equity",
    "total equity",
    "total equity gross minority interest",  # última opção
]

# Busca componentes
curr_ltd, per_curr, name_curr = _latest_value(bsL, ALIASES_CURR_LTD)
st_borr,  per_st,   name_st   = _latest_value(bsL, ALIASES_ST_BORROW)
lt_debt,  per_lt,   name_lt   = _latest_value(bsL, ALIASES_LT_DEBT)
tot_debt, per_tot,  name_tot  = _latest_value(bsL, ALIASES_TOTAL_DEBT)

# Estratégia para D (dívida total):
# 1) Se houver Total Debt explícito -> usa ele como D (mais confiável).
# 2) Senão, soma componentes disponíveis: LT + Current Portion + Short-Term Borrowings.
components_sum = sum(x for x in [lt_debt, curr_ltd, st_borr] if x is not None)
if tot_debt is not None:
    D_value = tot_debt
    D_period = per_tot
    D_source = name_tot
    D_alt_sum = components_sum if components_sum not in (None, 0) else None
else:
    D_value = components_sum if components_sum not in (None, 0) else None
    # período: o mais recente entre os componentes usados
    per_list = []
    if lt_debt is not None and per_lt: per_list.append(per_lt)
    if curr_ltd is not None and per_curr: per_list.append(per_curr)
    if st_borr is not None and per_st: per_list.append(per_st)
    D_period = max(per_list) if per_list else "-"
    D_source = "Soma de componentes (LT Debt + Current Portion + Short-Term)"

# Fallback Finviz para Total Debt, se ainda nada
if D_value is None and "fund_df" in globals():
    finviz_td = _get_from_finviz(fund_df, "Total Debt", like=True)
    if finviz_td is not None:
        D_value = finviz_td
        D_period = "-"
        D_source = "Total Debt (Finviz snapshot)"
        D_alt_sum = None

# Equity para calcular Po = D / (D + E)
equity_val, per_eq, name_eq = _latest_value(bsL, ALIASES_EQUITY)

# Peso da dívida no capital (Po)
Po_weight = None
if (D_value is not None) and (equity_val is not None) and (D_value + equity_val) != 0:
    Po_weight = D_value / (D_value + equity_val)

# --- Saída organizada
rows = [
    ("Current Portion of LT Debt", _fmt_money(curr_ltd), per_curr or "-", name_curr or "-"),
    ("Short-Term Borrowings",      _fmt_money(st_borr),  per_st or "-",   name_st or "-"),
    ("Long-Term Debt",             _fmt_money(lt_debt),  per_lt or "-",   name_lt or "-"),
    ("Total Debt (balanço)",       _fmt_money(tot_debt), per_tot or "-",  name_tot or "-"),
    ("Soma Componentes (ref.)",    _fmt_money(components_sum if components_sum not in (None, 0) else None),
                                   "-", "LT + Current Portion + Short-Term"),
    ("D (Capital de Terceiros)",   _fmt_money(D_value),  D_period or "-", D_source),
    ("Equity (PL)",                _fmt_money(equity_val), per_eq or "-", name_eq or "-"),
    ("Po (peso da dívida) = D/(D+E)", f"{Po_weight:.3f}" if Po_weight is not None else "-", "-", "cálculo"),
]

Po_df = pd.DataFrame(rows, columns=["Indicador", "Valor", "Período", "Fonte"])
print(f"🏗️ Capital de Terceiros (Po) — {LAST.get('ticker','?')}")
with pd.option_context('display.max_colwidth', 70, 'display.width', 140):
    display(Po_df)

# Variáveis disponíveis para o WACC etc.
Po_value  = D_value           # valor em USD
Po_share  = Po_weight         # peso da dívida no capital (0..1), se equity disponível


🏗️ Capital de Terceiros (Po) — MLI


,Indicador,Valor,Período,Fonte
0,Current Portion of LT Debt,"$1,094,000.00",2024-12-31,Current Debt
1,Short-Term Borrowings,-,-,-
2,Long-Term Debt,"$185,000.00",2023-12-31,Long Term Debt
3,Total Debt (balanço),"$27,490,000.00",2025-12-31,Total Debt
4,Soma Componentes (ref.),"$1,279,000.00",-,LT + Current Portion + Short-Term
5,D (Capital de Terceiros),"$27,490,000.00",2025-12-31,Total Debt
6,Equity (PL),"$3,235,906,000.00",2025-12-31,Total Equity Gross Minority Interest
7,Po (peso da dívida) = D/(D+E),0.008,-,cálculo


**Capital Investido (Ativos Totais)**

In [20]:
# =========================
# CÉLULA — Capital Investido (Ativos Totais) a partir do balanço (yfinance)
# =========================
import math
import pandas as pd

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return str(x)

def _ensure_long(bs_wide, bs_long):
    """Garante DataFrame longo (LineItem, Period, Value) a partir do wide se preciso."""
    if bs_long is not None and not bs_long.empty:
        return bs_long.copy()
    if bs_wide is None or bs_wide.empty:
        return None
    value_cols = [c for c in bs_wide.columns if c != "LineItem"]
    return (
        bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                     var_name="Period", value_name="Value")
               .dropna(subset=["Value"])
               .reset_index(drop=True)
    )

def _latest_value(bs_long, aliases):
    """
    Procura a 1ª linha cujo LineItem seja igual a algum alias (case-insensitive).
    Retorna (valor_float, periodo_str, nome_usado) ou (None, None, None).
    """
    if bs_long is None or bs_long.empty:
        return None, None, None

    df = bs_long.copy()
    df["LineItem_norm"] = df["LineItem"].str.strip().str.lower()
    try:
        df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")
    except:
        df["Period_dt"] = pd.NaT

    mask = False
    for a in aliases:
        mask = mask | df["LineItem_norm"].eq(a.strip().lower())
    sub = df.loc[mask].dropna(subset=["Value"]).copy()
    if sub.empty:
        return None, None, None

    sub = sub.sort_values(["Period_dt", "Period"], ascending=[True, True])
    val = float(sub["Value"].iloc[-1])
    per = str(sub["Period"].iloc[-1])
    used = str(sub["LineItem"].iloc[-1])
    return val, per, used

# --- prepara DF longo
bsL = _ensure_long(bs_wide if "bs_wide" in globals() else None,
                   bs_long if "bs_long" in globals() else None)

# Aliases comuns
ALIASES_TOTAL_ASSETS = [
    "total assets",
]
ALIASES_CASH_EQ = [
    "cash and cash equivalents",
    "cash & equivalents",
    "cash and equivalents",
    "cash cash equivalents and short term investments",
    "cash and cash equivalents, at carrying value",
]
# Passivos circulantes sem juros (NIBCL) — aproximamos por "Total Current Liabilities"
# (se quiser, depois podemos subtrair itens de juros; por ora é aproximação prática)
ALIASES_CURR_LIAB = [
    "total current liabilities",
]

# Busca principais
total_assets, per_assets, name_assets = _latest_value(bsL, ALIASES_TOTAL_ASSETS)
cash_eq,      per_cash,   name_cash   = _latest_value(bsL, ALIASES_CASH_EQ)
curr_liab,    per_cli,    name_cli    = _latest_value(bsL, ALIASES_CURR_LIAB)

# Capital Investido (aprox. operacional): TA - Caixa - NIBCL (aprox. por Total Current Liabilities)
invested_cap = None
if total_assets is not None:
    # subtrai caixa se existir
    base = total_assets - (cash_eq if cash_eq is not None else 0.0)
    # subtrai passivos correntes (aprox. NIBCL); se não quiser subtrair tudo, podemos refinar depois
    if curr_liab is not None:
        base -= curr_liab
    invested_cap = base

# ---- Saída organizada
rows = [
    ("Ativos Totais (Total Assets)", _fmt_money(total_assets), per_assets or "-", name_assets or "-"),
    ("Caixa & Equivalentes",         _fmt_money(cash_eq),      per_cash or "-",   name_cash or "-"),
    ("Passivo Circulante (aprox. NIBCL)", _fmt_money(curr_liab), per_cli or "-", name_cli or "-"),
    ("Capital Investido (aprox.)",   _fmt_money(invested_cap), per_assets or "-", "Total Assets - Cash - Current Liabilities"),
]

capinv_df = pd.DataFrame(rows, columns=["Indicador", "Valor", "Período", "Fonte/Observação"])
print(f"🏗️ Capital Investido (Ativos Totais) — {LAST.get('ticker','?')}")
with pd.option_context('display.max_colwidth', 80, 'display.width', 140):
    display(capinv_df)

# Variáveis disponíveis para usar adiante:
TotalAssets_value = total_assets
InvestedCapital_approx = invested_cap


🏗️ Capital Investido (Ativos Totais) — MLI


,Indicador,Valor,Período,Fonte/Observação
0,Ativos Totais (Total Assets),"$3,733,029,000.00",2025-12-31,Total Assets
1,Caixa & Equivalentes,"$1,367,003,000.00",2025-12-31,Cash And Cash Equivalents
2,Passivo Circulante (aprox. NIBCL),-,-,-
3,Capital Investido (aprox.),"$2,366,026,000.00",2025-12-31,Total Assets - Cash - Current Liabilities


**Caixa e Aplicações Financeiras**

In [21]:
# =========================
# CÉLULA — Caixa e Aplicações Financeiras (a partir do balanço yfinance)
# =========================
import math
import pandas as pd

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return str(x)

def _ensure_long(bs_wide, bs_long):
    """Garante DataFrame longo (LineItem, Period, Value) a partir do wide se preciso."""
    if bs_long is not None and not bs_long.empty:
        return bs_long.copy()
    if bs_wide is None or bs_wide.empty:
        return None
    value_cols = [c for c in bs_wide.columns if c != "LineItem"]
    return (
        bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                     var_name="Period", value_name="Value")
               .dropna(subset=["Value"])
               .reset_index(drop=True)
    )

def _latest_value(bs_long, aliases):
    """
    Procura a 1ª linha cujo LineItem seja igual a algum alias (case-insensitive) e retorna
    (valor_float, periodo_str, nome_usado) do período mais recente.
    """
    if bs_long is None or bs_long.empty:
        return None, None, None

    df = bs_long.copy()
    df["LineItem_norm"] = df["LineItem"].str.strip().str.lower()
    try:
        df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")
    except:
        df["Period_dt"] = pd.NaT

    mask = False
    for a in aliases:
        mask = mask | df["LineItem_norm"].eq(a.strip().lower())
    sub = df.loc[mask].dropna(subset=["Value"]).copy()
    if sub.empty:
        return None, None, None

    sub = sub.sort_values(["Period_dt", "Period"], ascending=[True, True])
    val = float(sub["Value"].iloc[-1])
    per = str(sub["Period"].iloc[-1])
    used = str(sub["LineItem"].iloc[-1])
    return val, per, used

# --- prepara DF longo
bsL = _ensure_long(bs_wide if "bs_wide" in globals() else None,
                   bs_long if "bs_long" in globals() else None)

# Aliases típicos (yfinance costuma variar nomes entre tickers)
ALIASES_CASH_EQ = [
    "cash and cash equivalents",
    "cash & equivalents",
    "cash and equivalents",
    "cash cash equivalents and short term investments",
    "cash and cash equivalents, at carrying value",
]
ALIASES_ST_INV = [
    "short term investments",
    "short-term investments",
    "marketable securities",
    "short term marketable securities",
    "short-term marketable securities",
]

cash_eq, per_cash, name_cash = _latest_value(bsL, ALIASES_CASH_EQ)
st_inv,  per_sti,  name_sti  = _latest_value(bsL, ALIASES_ST_INV)

# Total Caixa + Aplicações
cash_total = None
if (cash_eq is not None) or (st_inv is not None):
    cash_total = (cash_eq or 0.0) + (st_inv or 0.0)

# Período de referência: o mais recente entre os encontrados
per_list = [p for p in [per_cash, per_sti] if p]
period_ref = max(per_list) if per_list else "-"

rows = [
    ("Caixa & Equivalentes",      _fmt_money(cash_eq),  per_cash or "-", name_cash or "-"),
    ("Aplicações c/ curto prazo", _fmt_money(st_inv),   per_sti or "-",  name_sti or "-"),
    ("Total (Caixa + Aplicações)",_fmt_money(cash_total), period_ref,    "soma dos itens acima"),
]

cash_df = pd.DataFrame(rows, columns=["Indicador", "Valor", "Período", "Fonte"])
print(f"💵 Caixa e Aplicações Financeiras — {LAST.get('ticker','?')}")
with pd.option_context('display.max_colwidth', 80, 'display.width', 140):
    display(cash_df)

# Variáveis disponíveis para o restante do programa:
CashEquivalents_value = cash_eq                 # float em USD (pode ser None)
ShortTermInvestments_value = st_inv             # float em USD (pode ser None)
CashAndInvestments_total = cash_total           # float em USD (pode ser None)
CashAndInvestments_period = period_ref


💵 Caixa e Aplicações Financeiras — MLI


,Indicador,Valor,Período,Fonte
0,Caixa & Equivalentes,"$1,367,003,000.00",2025-12-31,Cash And Cash Equivalents
1,Aplicações c/ curto prazo,-,-,-
2,Total (Caixa + Aplicações),"$1,367,003,000.00",2025-12-31,soma dos itens acima


**Ativo Circulante**

In [22]:
# =========================
# CÉLULA — Ativo/Passivo Circulante (TCA/TCL) no Yahoo: "Current Assets" / "Current Liabilities"
# =========================
import re, math
import pandas as pd

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return str(x)

def _ensure_long(bs_wide, bs_long):
    if bs_long is not None and not bs_long.empty:
        return bs_long.copy()
    if bs_wide is None or bs_wide.empty:
        return None
    value_cols = [c for c in bs_wide.columns if c != "LineItem"]
    return (
        bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                     var_name="Period", value_name="Value")
               .dropna(subset=["Value"])
               .reset_index(drop=True)
    )

def _latest_match(bs_long, include_regexes, exclude_regexes=()):
    """
    Linha mais recente que:
      - casa QUALQUER regex de include_regexes
      - NÃO casa NENHUMA regex de exclude_regexes
    Retorna: (valor, período, nome, candidatos_df)
    """
    if bs_long is None or bs_long.empty:
        return None, None, None, pd.DataFrame()

    df = bs_long.copy()
    df["LineItem_str"] = df["LineItem"].astype(str)
    df["norm"] = df["LineItem_str"].str.strip().str.lower()
    df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")

    inc = [re.compile(p, re.I) for p in include_regexes]
    exc = [re.compile(p, re.I) for p in exclude_regexes] if exclude_regexes else []

    def ok(name):
        if not any(r.search(name) for r in inc):
            return False
        if any(r.search(name) for r in exc):
            return False
        return True

    sub = df[df["norm"].apply(ok)].dropna(subset=["Value"]).copy()
    if sub.empty:
        return None, None, None, pd.DataFrame()

    sub = sub.sort_values(["Period_dt", "Period"], ascending=[True, True])
    best = sub.iloc[-1]
    candidates = sub.tail(10)[["LineItem_str","Period","Value"]].copy()
    return float(best["Value"]), str(best["Period"]), str(best["LineItem_str"]), candidates

def _sum_components(bs_long, include_patterns, exclude_patterns=()):
    """
    Soma componentes (por regex) no ÚLTIMO período disponível.
    Retorna: (soma, período, df_componentes)
    """
    if bs_long is None or bs_long.empty:
        return None, None, pd.DataFrame()

    df = bs_long.copy()
    df["norm"] = df["LineItem"].str.strip().str.lower()
    df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")

    inc_re = [re.compile(p, re.I) for p in include_patterns]
    exc_re = [re.compile(p, re.I) for p in exclude_patterns] if exclude_patterns else []

    def match_row(name):
        if not any(r.search(name) for r in inc_re):
            return False
        if any(r.search(name) for r in exc_re):
            return False
        return True

    sub = df[df["norm"].apply(match_row)].dropna(subset=["Value"]).copy()
    if sub.empty:
        return None, None, pd.DataFrame()

    last_dt = sub["Period_dt"].max()
    last_per = sub.loc[sub["Period_dt"]==last_dt, "Period"].iloc[-1] if pd.notna(last_dt) else sub["Period"].iloc[-1]

    sub_last = sub[sub["Period"] == last_per]
    total = float(sub_last["Value"].sum())
    comps = sub_last[["LineItem","Value"]].sort_values("Value", ascending=False)
    return total, str(last_per), comps

# --- preparar DF longo
bsL = _ensure_long(bs_wide if "bs_wide" in globals() else None,
                   bs_long if "bs_long" in globals() else None)

# Referências úteis
TA_val, TA_per, _, _ = _latest_match(bsL, [r"^total\s+assets$"])  # Total Assets
WC_val, WC_per, WC_name, _ = _latest_match(bsL, [r"^working\s+capital$"])

# Regex EXATAS para TCA/TCL (aceita com ou sem "total" antes)
INC_TCA = [r"^(?:total\s+)?current\s+assets$"]
INC_TCL = [r"^(?:total\s+)?current\s+liabilities$"]
# Exclusões para evitar confusão
EXC_NON = [r"\bnon\s*-?\s*current\b", r"\bnoncurrent\b"]
EXC_NOISE_ASSETS = EXC_NON + [r"^total\s+assets$", r"^total\s+non\s*current\s+assets$"]
EXC_NOISE_LIAB   = EXC_NON + [r"^total\s+liabilities\b", r"^total\s+non\s*current\s+liabilities\b"]

# 1) Busca direta por TCA/TCL
tca, per_tca, name_tca, cand_TCA = _latest_match(bsL, INC_TCA, EXC_NOISE_ASSETS)
tcl, per_tcl, name_tcl, cand_TCL = _latest_match(bsL, INC_TCL, EXC_NOISE_LIAB)

# 2) Se faltou TCA, somar SOMENTE componentes de ativo circulante (lado de ativos)
if tca is None:
    include_TCA_assets = [
        r"\bcash\b", r"\bcash & equivalents\b", r"\bcash and equivalents\b",
        r"\bshort[- ]?term investments\b", r"\bmarketable securities\b",
        r"\baccounts? receivable\b", r"\btrade receivables?\b", r"\bnotes receivable\b",
        r"\binventory\b", r"\binventories\b",
        r"\bprepaid\b", r"\bother current assets\b", r"\bdeferred.*\bcurrent\b"
    ]
    # EXCLUI qualquer coisa não-corrente, passivos, dívida, "total assets"
    exclude_TCA_assets = EXC_NON + [r"\bliabilit", r"\bdebt\b", r"^total\s+assets$", r"^total\s+non\s*current\s+assets$"]
    tca_sum, per_sum, comps_TCA = _sum_components(bsL, include_TCA_assets, exclude_TCA_assets)
    if tca_sum is not None:
        tca, per_tca, name_tca = tca_sum, per_sum, "Soma de componentes (current assets)"

# 3) Se faltou TCL, somar componentes de passivo circulante (lado de passivos)
if tcl is None:
    include_TCL_liab = [
        r"\baccounts? payable\b", r"\btrade payables?\b",
        r"\baccrued\b", r"\bother current liabilities\b", r"\btaxes? payable\b",
        r"\bcurrent portion.*long[- ]?term debt\b", r"\bcurrent debt\b",
        r"\bshort[- ]?term borrowings?\b", r"\bshort[- ]?term debt\b",
        r"\bdeferred.*\bcurrent\b"
    ]
    exclude_TCL_liab = EXC_NON + [r"\bassets?\b"]
    tcl_sum, per_tcl_sum, comps_TCL = _sum_components(bsL, include_TCL_liab, exclude_TCL_liab)
    if tcl_sum is not None:
        tcl, per_tcl, name_tcl = tcl_sum, per_tcl_sum, "Soma de componentes (current liabilities)"

# 4) Últimos recursos de reconstrução
method = "direto"
if name_tca and name_tca.lower().startswith("soma"):
    method = "soma de componentes"
elif tca is None and (WC_val is not None) and (tcl is not None):
    # TCA = WC + TCL
    tca = WC_val + tcl
    per_tca = max([p for p in [WC_per, per_tcl] if p]) if any([WC_per, per_tcl]) else "-"
    name_tca = "Reconstruído: Working Capital + Current Liabilities"
    method = "reconstruído (WC + TCL)"
elif tca is None:
    method = "não encontrado"

# 5) Sanidade: não deixar TCA > Total Assets (se TA disponível)
if (tca is not None) and (TA_val is not None) and (tca > TA_val * 1.001):
    # limita e avisa (se quiser, apenas limite silenciosamente)
    print(f"⚠️ TCA calculado ({_fmt_money(tca)}) > Total Assets ({_fmt_money(TA_val)}). Ajustando para TA.")
    tca = TA_val
    method = method + " + ajuste≤TA"

# Current Ratio
current_ratio = None
if (tca is not None) and (tcl is not None) and tcl != 0:
    current_ratio = tca / tcl

# ---- saída
rows = [
    ("Ativo Circulante (TCA)", _fmt_money(tca), per_tca or "-", name_tca or "-"),
    ("Passivo Circulante (TCL)", _fmt_money(tcl), per_tcl or "-", name_tcl or "-"),
    ("Working Capital (ref.)", _fmt_money(WC_val), WC_per or "-", WC_name or "-"),
    ("Método TCA", method, "-", "-"),
    ("Current Ratio (TCA/TCL)", (f"{current_ratio:.2f}" if current_ratio is not None else "-"), "-", "cálculo"),
]
tca_df = pd.DataFrame(rows, columns=["Indicador","Valor","Período","Fonte/Obs."])
print(f"📦 Ativo/Passivo Circulante — {LAST.get('ticker','?')}")
with pd.option_context('display.max_colwidth', 90, 'display.width', 180):
    display(tca_df)

# Diagnóstico: mostre candidatos e componentes SOMENTE se existirem
if 'cand_TCA' in locals() and cand_TCA is not None and not cand_TCA.empty:
    print("\n🔍 Candidatos TCA (regex exata):")
    display(cand_TCA)

if 'cand_TCL' in locals() and cand_TCL is not None and not cand_TCL.empty:
    print("\n🔍 Candidatos TCL (regex exata):")
    display(cand_TCL)

if 'comps_TCA' in locals() and comps_TCA is not None and not comps_TCA.empty:
    print("\n🧮 Componentes usados para TCA (último período):")
    display(comps_TCA)

if 'comps_TCL' in locals() and comps_TCL is not None and not comps_TCL.empty:
    print("\n🧾 Componentes usados para TCL (último período):")
    display(comps_TCL)

# Variáveis globais úteis
TotalCurrentAssets_value  = tca
TotalCurrentAssets_period = per_tca
TotalCurrentLiabilities_value = tcl
CurrentRatio_value = current_ratio


📦 Ativo/Passivo Circulante — MLI


,Indicador,Valor,Período,Fonte/Obs.
0,Ativo Circulante (TCA),"$2,445,745,000.00",2025-12-31,Current Assets
1,Passivo Circulante (TCL),"$413,134,000.00",2025-12-31,Current Liabilities
2,Working Capital (ref.),"$2,032,611,000.00",2025-12-31,Working Capital
3,Método TCA,direto,-,-
4,Current Ratio (TCA/TCL),5.92,-,cálculo



🔍 Candidatos TCA (regex exata):


,LineItem_str,Period,Value
279,Current Assets,2022-12-31,1.534653e+09
204,Current Assets,2023-12-31,2.040021e+09
130,Current Assets,2024-12-31,2.012229e+09
57,Current Assets,2025-12-31,2.445745e+09



🔍 Candidatos TCL (regex exata):


,LineItem_str,Period,Value
251,Current Liabilities,2022-12-31,348295000.0
177,Current Liabilities,2023-12-31,317138000.0
103,Current Liabilities,2024-12-31,397987000.0
32,Current Liabilities,2025-12-31,413134000.0


**Ativo Circulante Operacional e o Ativo Circulante Operacional (Ano anterior)**

In [23]:
# =========================
# CÉLULA — Ativo Circulante Operacional (último e anterior) a partir do yfinance
# =========================
import re
import math
import pandas as pd

def _ensure_long(bs_wide, bs_long):
    """Garante DF longo (LineItem, Period, Value)."""
    if bs_long is not None and not bs_long.empty:
        return bs_long.copy()
    if bs_wide is None or bs_wide.empty:
        return None
    value_cols = [c for c in bs_wide.columns if c != "LineItem"]
    return (
        bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                     var_name="Period", value_name="Value")
               .dropna(subset=["Value"])
               .reset_index(drop=True)
    )

def _series_by_regex(bs_long, include_regexes, exclude_regexes=()):
    """
    Retorna uma série (Period->Value) para a 1ª linha que bater com include (e não bater com exclude).
    Period em ordem cronológica (por dt); valores float.
    """
    if bs_long is None or bs_long.empty:
        return pd.Series(dtype=float)

    df = bs_long.copy()
    df["LineItem_str"] = df["LineItem"].astype(str)
    df["norm"] = df["LineItem_str"].str.strip().str.lower()
    df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")

    inc = [re.compile(p, re.I) for p in include_regexes]
    exc = [re.compile(p, re.I) for p in exclude_regexes] if exclude_regexes else []

    def ok(name):
        if not any(r.search(name) for r in inc):
            return False
        if any(r.search(name) for r in exc):
            return False
        return True

    cand = df[df["norm"].apply(ok)].copy()
    if cand.empty:
        return pd.Series(dtype=float)

    # pega a linha com mais períodos (se houver duplicatas de nomes)
    # estratégia: agrupa por LineItem e escolhe o que tem mais valores
    grp = cand.groupby("LineItem_str")
    best_name = grp.size().sort_values().index[-1]
    sub = cand[cand["LineItem_str"] == best_name].copy()

    # ordena por data
    sub = sub.sort_values(["Period_dt","Period"], ascending=[True, True])
    s = pd.Series(sub["Value"].astype(float).values, index=sub["Period"].astype(str).values, name=best_name)
    return s

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return "-"

# --- preparar DF longo
bsL = _ensure_long(bs_wide if "bs_wide" in globals() else None,
                   bs_long if "bs_long" in globals() else None)

if bsL is None or bsL.empty:
    print("❌ Não há balanço carregado (bs_wide/bs_long). Rode a célula de balanço primeiro.")
else:
    # Regex (aceita 'Current Assets' com ou sem 'Total' antes; exclui non-current)
    INC_CA  = [r"^(?:total\s+)?current\s+assets$"]
    INC_CE  = [r"^cash(\s*&\s*equivalents| and equivalents)?$",
               r"^cash cash equivalents and short term investments$",
               r"^cash and cash equivalents, at carrying value$"]
    INC_STI = [r"^short[- ]?term investments$", r"^marketable securities$", r"^short[- ]?term marketable securities$"]

    EXC_NON = [r"\bnon\s*-?\s*current\b", r"\bnoncurrent\b"]

    # Séries por período
    s_CA  = _series_by_regex(bsL, INC_CA, EXC_NON)
    s_CE  = _series_by_regex(bsL, INC_CE, EXC_NON)
    s_STI = _series_by_regex(bsL, INC_STI, EXC_NON)

    # Consolida em um DF por período e pega os 2 mais recentes
    df = pd.DataFrame({
        "Current Assets": s_CA,
        "Cash & Equivalents": s_CE,
        "Short-Term Investments": s_STI
    })

    # mantém só períodos com pelo menos CA (para fazer sentido)
    df = df.dropna(subset=["Current Assets"], how="all")

    # Ordenar por período cronológico (tenta converter; se falhar, mantém ordenação natural)
    try:
        order = pd.to_datetime(df.index)
        df = df.iloc[order.argsort()]
    except Exception:
        pass

    # Seleciona últimos 2 períodos
    df2 = df.tail(2).copy()

    # Calcula OCA (básico e “limpo”)
    df2["OCA (CA - Cash)"] = df2["Current Assets"] - df2["Cash & Equivalents"].fillna(0.0)
    df2["OCA (CA - Cash - ST Inv.)"] = df2["OCA (CA - Cash)"] - df2["Short-Term Investments"].fillna(0.0)

    # Formatação para exibição
    pretty = df2.copy()
    for col in pretty.columns:
        pretty[col] = pretty[col].apply(_fmt_money)

    print(f"🧰 Ativo Circulante Operacional — {LAST.get('ticker','?')}")
    print("Períodos (linhas) em ordem cronológica; as duas últimas linhas são: anterior e atual.")
    display(pretty)

    # Exibir linha a linha consolidada (Atual e Anterior)
    if not df2.empty:
        current_period = df2.index[-1]
        prev_period = df2.index[-2] if len(df2) >= 2 else None

        # Valores para uso posterior
        OCA_current_basic = float(df2["OCA (CA - Cash)"].iloc[-1]) if pd.notna(df2["OCA (CA - Cash)"].iloc[-1]) else None
        OCA_prev_basic    = float(df2["OCA (CA - Cash)"].iloc[-2]) if prev_period is not None and pd.notna(df2["OCA (CA - Cash)"].iloc[-2]) else None

        OCA_current_clean = float(df2["OCA (CA - Cash - ST Inv.)"].iloc[-1]) if pd.notna(df2["OCA (CA - Cash - ST Inv.)"].iloc[-1]) else None
        OCA_prev_clean    = float(df2["OCA (CA - Cash - ST Inv.)"].iloc[-2]) if prev_period is not None and pd.notna(df2["OCA (CA - Cash - ST Inv.)"].iloc[-2]) else None

        # Impressão resumida
        print("\n✅ Resumo (valores principais):")
        print(f"• Período atual ({current_period}):")
        print(f"   - OCA (CA - Cash):            {_fmt_money(OCA_current_basic)}")
        print(f"   - OCA (CA - Cash - ST Inv.):  {_fmt_money(OCA_current_clean)}")
        if prev_period is not None:
            print(f"• Período anterior ({prev_period}):")
            print(f"   - OCA (CA - Cash):            {_fmt_money(OCA_prev_basic)}")
            print(f"   - OCA (CA - Cash - ST Inv.):  {_fmt_money(OCA_prev_clean)}")

        # Exporta para variáveis globais, se quiser usar depois no valuation
        OCA_current_basic_value = OCA_current_basic
        OCA_previous_basic_value = OCA_prev_basic
        OCA_current_clean_value = OCA_current_clean
        OCA_previous_clean_value = OCA_prev_clean
    else:
        print("⚠️ Não foi possível identificar períodos suficientes para o OCA.")


🧰 Ativo Circulante Operacional — MLI
Períodos (linhas) em ordem cronológica; as duas últimas linhas são: anterior e atual.


,Current Assets,Cash & Equivalents,Short-Term Investments,OCA (CA - Cash),OCA (CA - Cash - ST Inv.)
2024-12-31,"$2,012,229,000.00","$1,059,103,000.00",-,"$953,126,000.00","$953,126,000.00"
2025-12-31,"$2,445,745,000.00","$1,389,736,000.00",-,"$1,056,009,000.00","$1,056,009,000.00"



✅ Resumo (valores principais):
• Período atual (2025-12-31):
   - OCA (CA - Cash):            $1,056,009,000.00
   - OCA (CA - Cash - ST Inv.):  $1,056,009,000.00
• Período anterior (2024-12-31):
   - OCA (CA - Cash):            $953,126,000.00
   - OCA (CA - Cash - ST Inv.):  $953,126,000.00


**Passivo Circulante**

In [24]:
# =========================
# CÉLULA — Passivo Circulante (Total Current Liabilities) via yfinance
# =========================
import re, math
import pandas as pd

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return "-"

def _ensure_long(bs_wide, bs_long):
    """Garante DF longo (LineItem, Period, Value)."""
    if 'bs_long' in globals() and bs_long is not None and not bs_long.empty:
        return bs_long.copy()
    if 'bs_wide' in globals() and bs_wide is not None and not bs_wide.empty:
        value_cols = [c for c in bs_wide.columns if c != "LineItem"]
        return (bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                             var_name="Period", value_name="Value")
                      .dropna(subset=["Value"])
                      .reset_index(drop=True))
    return None

def _latest_match(bs_long, include_regexes, exclude_regexes=()):
    """
    Linha mais recente que:
      - casa QUALQUER include_regex
      - NÃO casa NENHUMA exclude_regex
    Retorna: (valor, período, nome, candidatos_df)
    """
    if bs_long is None or bs_long.empty:
        return None, None, None, pd.DataFrame()

    df = bs_long.copy()
    df["LineItem_str"] = df["LineItem"].astype(str)
    df["norm"] = df["LineItem_str"].str.strip().str.lower()
    df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")

    inc = [re.compile(p, re.I) for p in include_regexes]
    exc = [re.compile(p, re.I) for p in exclude_regexes] if exclude_regexes else []

    def ok(name):
        if not any(r.search(name) for r in inc):
            return False
        if any(r.search(name) for r in exc):
            return False
        return True

    sub = df[df["norm"].apply(ok)].dropna(subset=["Value"]).copy()
    if sub.empty:
        return None, None, None, pd.DataFrame()

    sub = sub.sort_values(["Period_dt","Period"], ascending=[True,True])
    best = sub.iloc[-1]
    return float(best["Value"]), str(best["Period"]), str(best["LineItem_str"]), sub.tail(10)[["LineItem_str","Period","Value"]]

def _sum_components(bs_long, include_patterns, exclude_patterns=()):
    """Soma componentes do passivo circulante no último período disponível."""
    if bs_long is None or bs_long.empty:
        return None, None, pd.DataFrame()

    df = bs_long.copy()
    df["norm"] = df["LineItem"].str.strip().str.lower()
    df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")

    inc_re = [re.compile(p, re.I) for p in include_patterns]
    exc_re = [re.compile(p, re.I) for p in exclude_patterns] if exclude_patterns else []

    def match_row(name):
        if not any(r.search(name) for r in inc_re):
            return False
        if any(r.search(name) for r in exc_re):
            return False
        return True

    sub = df[df["norm"].apply(match_row)].dropna(subset=["Value"]).copy()
    if sub.empty:
        return None, None, pd.DataFrame()

    # pega o último período
    last_dt = sub["Period_dt"].max()
    last_per = sub.loc[sub["Period_dt"]==last_dt, "Period"].iloc[-1] if pd.notna(last_dt) else sub["Period"].iloc[-1]
    sub_last = sub[sub["Period"] == last_per]

    total = float(sub_last["Value"].sum())
    comps = sub_last[["LineItem","Value"]].sort_values("Value", ascending=False)
    return total, str(last_per), comps

# --- preparar DF longo
bsL = _ensure_long(bs_wide if "bs_wide" in globals() else None,
                   bs_long if "bs_long" in globals() else None)

if bsL is None or bsL.empty:
    print("❌ Não há balanço carregado (bs_wide/bs_long). Rode a célula de balanço primeiro.")
else:
    # Regex exatas (aceita com ou sem 'Total' antes), excluindo qualquer 'non current'
    INC_TCL = [r"^(?:total\s+)?current\s+liabilities$"]
    EXC_NON = [r"\bnon\s*-?\s*current\b", r"\bnoncurrent\b"]
    EXC_NOISE = EXC_NON + [r"^total\s+non\s*current\s+liabilities\b", r"^total\s+liabilities\b"]

    # 1) Busca direta
    tcl, per_tcl, name_tcl, cand_TCL = _latest_match(bsL, INC_TCL, EXC_NOISE)

    # 2) Se não achar, somar componentes típicos do passivo circulante
    comps_TCL = pd.DataFrame()
    if tcl is None:
        include_TCL = [
            r"\baccounts? payable\b", r"\btrade payables?\b",
            r"\bacc(?:rued)?\b", r"\bother current liabilities\b",
            r"\btaxes? payable\b",
            r"\bcurrent portion.*long[- ]?term debt\b", r"\bcurrent debt\b",
            r"\bshort[- ]?term borrowings?\b", r"\bshort[- ]?term debt\b",
            r"\bdeferred.*\bcurrent\b"
        ]
        exclude_TCL = EXC_NON + [r"\bassets?\b"]
        tcl_sum, per_sum, comps_TCL = _sum_components(bsL, include_TCL, exclude_TCL)
        if tcl_sum is not None:
            tcl, per_tcl, name_tcl = tcl_sum, per_sum, "Soma de componentes (current liabilities)"

    # ---- saída
    print(f"📑 Passivo Circulante — {LAST.get('ticker','?')}")
    print(f"• Valor:   {_fmt_money(tcl)}")
    print(f"• Período: {per_tcl or '-'}")
    print(f"• Fonte:   {name_tcl or '-'}")

    # Diagnóstico (só se houver conteúdo)
    if cand_TCL is not None and not cand_TCL.empty:
        print("\n🔍 Candidatos TCL (regex exata):")
        display(cand_TCL)
    if comps_TCL is not None and not comps_TCL.empty:
        print("\n🧾 Componentes usados na soma de TCL (último período):")
        display(comps_TCL)

    # Variáveis globais úteis
    TotalCurrentLiabilities_value = tcl
    TotalCurrentLiabilities_period = per_tcl


📑 Passivo Circulante — MLI
• Valor:   $413,134,000.00
• Período: 2025-12-31
• Fonte:   Current Liabilities

🔍 Candidatos TCL (regex exata):


,LineItem_str,Period,Value
251,Current Liabilities,2022-12-31,348295000.0
177,Current Liabilities,2023-12-31,317138000.0
103,Current Liabilities,2024-12-31,397987000.0
32,Current Liabilities,2025-12-31,413134000.0


**Passivo Circulante Operacional e o Passivo Circulante Operacional (Ano anterior)**

In [25]:
# =========================
# CÉLULA — Passivo Circulante Operacional (atual e anterior) via yfinance
# =========================
import re
import math
import pandas as pd

def _ensure_long(bs_wide, bs_long):
    """Garante DF longo (LineItem, Period, Value)."""
    if 'bs_long' in globals() and bs_long is not None and not bs_long.empty:
        return bs_long.copy()
    if 'bs_wide' in globals() and bs_wide is not None and not bs_wide.empty:
        value_cols = [c for c in bs_wide.columns if c != "LineItem"]
        return (
            bs_wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                         var_name="Period", value_name="Value")
                   .dropna(subset=["Value"])
                   .reset_index(drop=True)
        )
    return None

def _series_by_regex(bs_long, include_regexes, exclude_regexes=()):
    """
    Retorna uma série Period->Value para a 1ª linha que bater com include (e não bater com exclude).
    Escolhe o LineItem com MAIOR cobertura de períodos (estável).
    """
    if bs_long is None or bs_long.empty:
        return pd.Series(dtype=float)

    df = bs_long.copy()
    df["LI"] = df["LineItem"].astype(str)
    df["norm"] = df["LI"].str.strip().str.lower()
    df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")

    inc = [re.compile(p, re.I) for p in include_regexes]
    exc = [re.compile(p, re.I) for p in exclude_regexes] if exclude_regexes else []

    def ok(name):
        if not any(r.search(name) for r in inc):
            return False
        if any(r.search(name) for r in exc):
            return False
        return True

    cand = df[df["norm"].apply(ok)].copy()
    if cand.empty:
        return pd.Series(dtype=float)

    # escolhe o LineItem com mais linhas (mais períodos)
    best_name = (cand.groupby("LI").size().sort_values().index[-1])
    sub = cand[cand["LI"] == best_name].sort_values(["Period_dt","Period"], ascending=[True, True])

    s = pd.Series(sub["Value"].astype(float).values, index=sub["Period"].astype(str).values, name=best_name)
    return s

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return "-"

# --- preparar DF longo
bsL = _ensure_long(bs_wide if "bs_wide" in globals() else None,
                   bs_long if "bs_long" in globals() else None)

if bsL is None or bsL.empty:
    print("❌ Não há balanço carregado (bs_wide/bs_long). Rode a célula de balanço primeiro.")
else:
    # Regex (aceita com/sem 'Total'; exclui non-current)
    INC_TCL = [r"^(?:total\s+)?current\s+liabilities$"]
    EXC_NON = [r"\bnon\s*-?\s*current\b", r"\bnoncurrent\b"]
    EXC_NOISE_LIAB = EXC_NON + [r"^total\s+non\s*current\s+liabilities\b", r"^total\s+liabilities\b"]

    # Itens de curto prazo com juros (para subtrair do TCL)
    INC_CURR_PORTION_LTD = [
        r"^current\s+portion.*long[- ]?term\s+debt$",
        r"^current\s+maturit(?:y|ies).*long[- ]?term\s+debt$"
    ]
    INC_ST_BORROW = [
        r"^short[- ]?term\s+borrowings?$",
        r"^short[- ]?term\s+debt$",
        r"^current\s+debt$",
        r"^bank\s+overdrafts?$",
        r"^commercial\s+paper$",
        r"^notes?\s+payable$"  # pode incluir juros em muitas empresas
    ]

    # Séries
    s_TCL  = _series_by_regex(bsL, INC_TCL, EXC_NOISE_LIAB)
    s_CPLTD = _series_by_regex(bsL, INC_CURR_PORTION_LTD, EXC_NON)
    s_STB   = _series_by_regex(bsL, INC_ST_BORROW, EXC_NON)

    # Consolida em DF por período
    df = pd.DataFrame({
        "Total Current Liabilities": s_TCL,
        "Current Portion of LT Debt": s_CPLTD,
        "Short-Term Borrowings/Debt": s_STB,
    })

    # Mantém só períodos com algum TCL
    df = df.dropna(subset=["Total Current Liabilities"], how="all")

    # Ordena por período cronológico
    try:
        order = pd.to_datetime(df.index)
        df = df.iloc[order.argsort()]
    except Exception:
        pass

    # Seleciona últimos 2 períodos (anterior e atual)
    df2 = df.tail(2).copy()

    # Calcula itens com juros (preenche faltas com 0)
    df2["Interest-bearing (current)"] = df2[["Current Portion of LT Debt", "Short-Term Borrowings/Debt"]].fillna(0).sum(axis=1)

    # Passivo Circulante Operacional = TCL - Itens com juros
    df2["PCO (Operacional)"] = df2["Total Current Liabilities"] - df2["Interest-bearing (current)"]

    # Formatação para exibição
    pretty = df2.copy()
    for col in pretty.columns:
        pretty[col] = pretty[col].apply(_fmt_money)

    print(f"🧾 Passivo Circulante Operacional — {LAST.get('ticker','?')}")
    print("Períodos em ordem cronológica; as duas últimas linhas são: ANTERIOR e ATUAL.")
    display(pretty)

    # Resumo (variáveis para usar depois)
    if not df2.empty:
        curr_per = df2.index[-1]
        prev_per = df2.index[-2] if len(df2) >= 2 else None

        PCO_current = float(df2["PCO (Operacional)"].iloc[-1]) if pd.notna(df2["PCO (Operacional)"].iloc[-1]) else None
        PCO_prev    = float(df2["PCO (Operacional)"].iloc[-2]) if prev_per is not None and pd.notna(df2["PCO (Operacional)"].iloc[-2]) else None

        TCL_current = float(df2["Total Current Liabilities"].iloc[-1]) if pd.notna(df2["Total Current Liabilities"].iloc[-1]) else None
        TCL_prev    = float(df2["Total Current Liabilities"].iloc[-2]) if prev_per is not None and pd.notna(df2["Total Current Liabilities"].iloc[-2]) else None

        IB_current  = float(df2["Interest-bearing (current)"].iloc[-1]) if pd.notna(df2["Interest-bearing (current)"].iloc[-1]) else None
        IB_prev     = float(df2["Interest-bearing (current)"].iloc[-2]) if prev_per is not None and pd.notna(df2["Interest-bearing (current)"].iloc[-2]) else None

        print("\n✅ Resumo:")
        print(f"• Período atual ({curr_per}):")
        print(f"   - TCL: {_fmt_money(TCL_current)}")
        print(f"   - Itens com juros (CP): {_fmt_money(IB_current)}")
        print(f"   - PCO (TCL - juros CP): {_fmt_money(PCO_current)}")
        if prev_per is not None:
            print(f"• Período anterior ({prev_per}):")
            print(f"   - TCL: {_fmt_money(TCL_prev)}")
            print(f"   - Itens com juros (CP): {_fmt_money(IB_prev)}")
            print(f"   - PCO (TCL - juros CP): {_fmt_money(PCO_prev)}")

        # Variáveis globais úteis p/ valuation:
        PCO_current_value = PCO_current
        PCO_previous_value = PCO_prev
        TCL_current_value = TCL_current
        TCL_previous_value = TCL_prev
        InterestBearingCurrent_current = IB_current
        InterestBearingCurrent_previous = IB_prev
    else:
        print("⚠️ Não foi possível identificar períodos suficientes para o PCO.")


🧾 Passivo Circulante Operacional — MLI
Períodos em ordem cronológica; as duas últimas linhas são: ANTERIOR e ATUAL.


,Total Current Liabilities,Current Portion of LT Debt,Short-Term Borrowings/Debt,Interest-bearing (current),PCO (Operacional)
2024-12-31,"$397,987,000.00",-,"$1,094,000.00","$1,094,000.00","$396,893,000.00"
2025-12-31,"$413,134,000.00",-,-,$0.00,"$413,134,000.00"



✅ Resumo:
• Período atual (2025-12-31):
   - TCL: $413,134,000.00
   - Itens com juros (CP): $0.00
   - PCO (TCL - juros CP): $413,134,000.00
• Período anterior (2024-12-31):
   - TCL: $397,987,000.00
   - Itens com juros (CP): $1,094,000.00
   - PCO (TCL - juros CP): $396,893,000.00


# **Cash Flow Statements**

In [26]:
# ============================================
# CÉLULA — Cash Flow (anual e trimestral) via yfinance + FCF
# ============================================
import pandas as pd
import numpy as np
import math
import re

# --------------- utilidades ---------------
def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.0f}"
    except:
        return "-"

def _to_wide(df_cf):
    """Converte o yfinance cashflow (index: line items, cols: dates) para wide com col 'LineItem'."""
    if df_cf is None or df_cf.empty:
        return None
    w = df_cf.copy()
    # yfinance vem com colunas DatetimeIndex desc; vamos ordenar asc
    try:
        w = w.loc[:, sorted(w.columns)]
    except Exception:
        pass
    w["LineItem"] = w.index.astype(str)
    w.reset_index(drop=True, inplace=True)
    return w[["LineItem"] + [c for c in w.columns if c != "LineItem"]]

def _to_long(wide):
    if wide is None or wide.empty:
        return None
    value_cols = [c for c in wide.columns if c != "LineItem"]
    long = (wide
            .melt(id_vars=["LineItem"], value_vars=value_cols,
                  var_name="Period", value_name="Value")
            .dropna(subset=["Value"])
            .reset_index(drop=True))
    # normaliza período para string yyyy-mm-dd
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
    except Exception:
        long["Period"] = long["Period"].astype(str)
    # ordem cronológica
    try:
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"], ascending=[True, True])
    except Exception:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes, exclude_regexes=()):
    """Pega a série Period->Value do LineItem com maior cobertura que bate no include e não bate no exclude."""
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)

    inc = [re.compile(p, re.I) for p in include_regexes]
    exc = [re.compile(p, re.I) for p in exclude_regexes] if exclude_regexes else []

    def ok(name):
        if not any(r.search(name) for r in inc):
            return False
        if any(r.search(name) for r in exc):
            return False
        return True

    cand = long_df[long_df["norm"].apply(ok)].copy()
    if cand.empty:
        return pd.Series(dtype=float)

    # qual LineItem tem mais períodos?
    best_name = cand.groupby("LineItem").size().sort_values().index[-1]
    sub = cand[cand["LineItem"] == best_name].copy()
    try:
        sub = sub.sort_values(["Period_dt","Period"], ascending=[True, True])
    except Exception:
        pass
    s = pd.Series(sub["Value"].astype(float).values,
                  index=sub["Period"].astype(str).values,
                  name=best_name)
    return s

# --------------- coleta yfinance ---------------
try:
    _ticker = (LAST.get("ticker") if "LAST" in globals() and isinstance(LAST, dict) and LAST.get("ticker") else None) or symbol
except NameError:
    _ticker = symbol  # se LAST não existir

import yfinance as yf
tk = yf.Ticker(_ticker)

cf_a_raw = tk.cashflow              # anual
cf_q_raw = tk.quarterly_cashflow    # trimestral (guardamos para usar depois, se quiser)

cf_wide = _to_wide(cf_a_raw)
cf_long = _to_long(cf_wide)

if cf_long is None or cf_long.empty:
    print("❌ Cash Flow anual não disponível no yfinance para este ticker.")
else:
    # --------------- aliases ---------------
    # CFO
    INC_CFO = [
        r"^net\s+cash\s+provided\s+by\s+operating\s+activities$",
        r"^net\s+cash\s+from\s+operating\s+activities$",
        r"^operating\s+cash\s+flow$",
        r"^net\s+cash\s+from\s+operating$",
    ]
    # CapEx (geralmente negativo no CF; queremos OUTFLOW positivo para cálculo)
    INC_CAPEX = [
        r"^capital\s+expenditures?$",
        r"^capital\s+expenditure$",
        r"^investment[s]?\s+in\s+property\s+plant\s+and\s+equipment$",
        r"^purchase[s]?\s+of\s+property\s+plant\s+and\s+equipment$",
        r"^additions?\s+to\s+property\s+plant\s+and\s+equipment$",
    ]
    # CFI
    INC_CFI = [
        r"^net\s+cash\s+used\s+for\s+investing\s+activities$",
        r"^net\s+cash\s+from\s+investing\s+activities$",
    ]
    # CFF
    INC_CFF = [
        r"^net\s+cash\s+used\s+for\s+financing\s+activities$",
        r"^net\s+cash\s+from\s+financing\s+activities$",
    ]
    # Dividends Paid
    INC_DIV = [
        r"^payments?\s+of\s+dividends?$",
        r"^dividends?\s+paid$",
        r"^common\s+stock\s+dividends?\s+paid$",
    ]
    # Buybacks
    INC_BUY = [
        r"^repurchase\s+of\s+capital\s+stock$",
        r"^purchase\s+of\s+stock$",
        r"^payments?\s+for\s+repurchase\s+of\s+common\s+stock$",
        r"^treasury\s+stock$",
    ]
    # Net Change in Cash
    INC_NETCASH = [
        r"^net\s+change\s+in\s+cash$",
        r"^net\s+increase\s+\(decrease\)\s+in\s+cash$",
    ]

    EXC_NONE = []

    s_CFO   = _series_by_regex(cf_long, INC_CFO, EXC_NONE)
    s_CAPEX = _series_by_regex(cf_long, INC_CAPEX, EXC_NONE)
    s_CFI   = _series_by_regex(cf_long, INC_CFI, EXC_NONE)
    s_CFF   = _series_by_regex(cf_long, INC_CFF, EXC_NONE)
    s_DIV   = _series_by_regex(cf_long, INC_DIV, EXC_NONE)
    s_BUY   = _series_by_regex(cf_long, INC_BUY, EXC_NONE)
    s_NET   = _series_by_regex(cf_long, INC_NETCASH, EXC_NONE)

    # --------------- monta DF anual e calcula FCF ---------------
    cf_df = pd.DataFrame({
        "CFO (Operating CF)": s_CFO,
        "CapEx (original sign)": s_CAPEX,
        "CFI": s_CFI,
        "CFF": s_CFF,
        "Dividends Paid": s_DIV,
        "Buybacks": s_BUY,
        "Net Change in Cash": s_NET,
    })

    # CapEx_outflow = valor positivo de saída (se s_CAPEX vier negativo, invertimos; se vier positivo, mantemos)
    def capex_outflow(x):
        if pd.isna(x):
            return np.nan
        x = float(x)
        return abs(x)  # tratamos CapEx como saída positiva

    cf_df["CapEx (outflow +)"] = cf_df["CapEx (original sign)"].apply(capex_outflow)

    # FCF = CFO - CapEx_outflow
    cf_df["FCF"] = cf_df["CFO (Operating CF)"] - cf_df["CapEx (outflow +)"]

    # ordenar por período e pegar até 4 mais recentes
    try:
        order = pd.to_datetime(cf_df.index)
        cf_df = cf_df.iloc[order.argsort()]
    except Exception:
        pass
    cf_last4 = cf_df.tail(4).copy()

    # --------------- apresentação ---------------
    pretty = cf_last4.applymap(_fmt_money)
    print(f"💨 Cash Flow (anual) — {_ticker}  | Últimos períodos")
    display(pretty)

    # --------------- variáveis úteis para valuation (último período anual) ---------------
    if not cf_last4.empty:
        last_period = cf_last4.index[-1]
        CFO_last   = float(cf_last4["CFO (Operating CF)"].iloc[-1])                    if pd.notna(cf_last4["CFO (Operating CF)"].iloc[-1]) else None
        Capex_last = float(cf_last4["CapEx (outflow +)"].iloc[-1])                     if pd.notna(cf_last4["CapEx (outflow +)"].iloc[-1]) else None
        FCF_last   = float(cf_last4["FCF"].iloc[-1])                                   if pd.notna(cf_last4["FCF"].iloc[-1]) else None
        DIV_last   = float(cf_last4["Dividends Paid"].iloc[-1])                        if pd.notna(cf_last4["Dividends Paid"].iloc[-1]) else None
        BUY_last   = float(cf_last4["Buybacks"].iloc[-1])                              if pd.notna(cf_last4["Buybacks"].iloc[-1]) else None
        CFI_last   = float(cf_last4["CFI"].iloc[-1])                                   if pd.notna(cf_last4["CFI"].iloc[-1]) else None
        CFF_last   = float(cf_last4["CFF"].iloc[-1])                                   if pd.notna(cf_last4["CFF"].iloc[-1]) else None
        NetCash_last = float(cf_last4["Net Change in Cash"].iloc[-1])                  if pd.notna(cf_last4["Net Change in Cash"].iloc[-1]) else None

        print("\n✅ Resumo (último anual):")
        print(f"• Período: {last_period}")
        print(f"  - CFO:   {_fmt_money(CFO_last)}")
        print(f"  - CapEx: {_fmt_money(Capex_last)}  (outflow positivo)")
        print(f"  - FCF:   {_fmt_money(FCF_last)}")
        print(f"  - CFI:   {_fmt_money(CFI_last)}   | CFF: {_fmt_money(CFF_last)}   | ΔCash: {_fmt_money(NetCash_last)}")
        print(f"  - Dividends: {_fmt_money(DIV_last)} | Buybacks: {_fmt_money(BUY_last)}")

        # torna globais (para valuation)
        CFO_value_annual_last   = CFO_last
        Capex_value_annual_last = Capex_last
        FCF_value_annual_last   = FCF_last
        DividendsPaid_last      = DIV_last
        Buybacks_last           = BUY_last
        CFI_value_annual_last   = CFI_last
        CFF_value_annual_last   = CFF_last
    else:
        print("⚠️ Cash Flow anual sem linhas suficientes para resumo.")

    # (Opcional) Guardar DF's completos para uso posterior
    CF_ANNUAL_WIDE = cf_wide
    CF_ANNUAL_LONG = cf_long
    CF_ANNUAL_LAST4 = cf_last4
    CF_QUARTERLY_RAW = cf_q_raw  # se quiser trabalhar depois com TTM


💨 Cash Flow (anual) — MLI  | Últimos períodos


,CFO (Operating CF),CapEx (original sign),CFI,CFF,Dividends Paid,Buybacks,Net Change in Cash,CapEx (outflow +),FCF
2022-12-31,"$723,943,000","$-37,639,000",-,-,-,"$-38,054,000",-,"$37,639,000","$686,304,000"
2023-12-31,"$672,766,000","$-54,025,000",-,-,-,"$-19,303,000",-,"$54,025,000","$618,741,000"
2024-12-31,"$645,908,000","$-80,203,000",-,-,-,"$-48,681,000",-,"$80,203,000","$565,705,000"
2025-12-31,"$755,444,000","$-68,805,000",-,-,-,"$-243,615,000",-,"$68,805,000","$686,639,000"



✅ Resumo (último anual):
• Período: 2025-12-31
  - CFO:   $755,444,000
  - CapEx: $68,805,000  (outflow positivo)
  - FCF:   $686,639,000
  - CFI:   -   | CFF: -   | ΔCash: -
  - Dividends: - | Buybacks: $-243,615,000


**Despesa Financeira**

In [27]:
# =========================
# CÉLULA — Despesas / Movimentos de Dívida (Issuance-Repayment of Debt)
# =========================
import re, math, pandas as pd, numpy as np

def _fmt_money(x):
    try:
        if x is None or (isinstance(x,float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return "-"

def _ensure_cf_long():
    if "CF_ANNUAL_LONG" in globals() and CF_ANNUAL_LONG is not None and not CF_ANNUAL_LONG.empty:
        return CF_ANNUAL_LONG.copy()
    if "CF_ANNUAL_WIDE" in globals() and CF_ANNUAL_WIDE is not None and not CF_ANNUAL_WIDE.empty:
        value_cols = [c for c in CF_ANNUAL_WIDE.columns if c != "LineItem"]
        return (CF_ANNUAL_WIDE
                .melt(id_vars=["LineItem"], value_vars=value_cols,
                      var_name="Period", value_name="Value")
                .dropna(subset=["Value"])
                .reset_index(drop=True))
    try:
        import yfinance as yf
        tk = yf.Ticker((LAST.get("ticker") if "LAST" in globals() and LAST.get("ticker") else symbol))
        cf = tk.cashflow
        if cf is None or cf.empty:
            return None
        wide = cf.copy()
        wide["LineItem"] = wide.index.astype(str)
        long = wide.melt(id_vars=["LineItem"], var_name="Period", value_name="Value").dropna(subset=["Value"])
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        return long
    except Exception:
        return None

def _series_by_regex(df, include_regexes):
    if df is None or df.empty:
        return pd.Series(dtype=float)
    df = df.copy()
    df["LI"] = df["LineItem"].astype(str)
    df["norm"] = df["LI"].str.strip().str.lower()
    df["Period_dt"] = pd.to_datetime(df["Period"], errors="coerce")

    inc = [re.compile(p,re.I) for p in include_regexes]
    sub = df[df["norm"].apply(lambda n: any(r.search(n) for r in inc))].copy()
    if sub.empty:
        return pd.Series(dtype=float)

    best = sub.groupby("LI").size().sort_values().index[-1]
    sub = sub[sub["LI"]==best].sort_values(["Period_dt","Period"],ascending=[True,True])
    s = pd.Series(sub["Value"].astype(float).values,
                  index=sub["Period"].astype(str).values,
                  name=best)
    return s

# --- busca
cfL = _ensure_cf_long()
if cfL is None or cfL.empty:
    print("❌ Nenhum Cash Flow disponível. Rode a célula de CF anual primeiro.")
else:
    INC_DEBT = [
        r"issuance\s+of\s+debt",
        r"repayment\s+of\s+debt",
        r"issuance\s+\(repayment\)\s+of\s+debt",
        r"borrowings",
        r"notes?\s+payable",
        r"long[- ]?term\s+debt\s+\(repayment\)",
        r"short[- ]?term\s+debt\s+\(repayment\)",
        r"proceeds\s+from\s+borrowings",
    ]
    s_DEBT = _series_by_regex(cfL, INC_DEBT)

    if s_DEBT.empty:
        print("⚠️ Nenhum item de 'Issuance / Repayment of Debt' encontrado no Cash Flow anual.")
    else:
        print(f"💰 Movimentos de Dívida — {(LAST.get('ticker') if 'LAST' in globals() and LAST.get('ticker') else symbol)}")
        display(pd.DataFrame(s_DEBT).rename(columns={s_DEBT.name:"Value"}).applymap(_fmt_money))

        last_period = s_DEBT.index[-1]
        last_value = float(s_DEBT.iloc[-1])
        print(f"\n📆 Último período: {last_period}")
        print(f"💵 Valor (Issuance / Repayment of Debt): {_fmt_money(last_value)}")

        # variável global para valuation / análise
        Debt_Issuance_Repayment_last = last_value


💰 Movimentos de Dívida — MLI


,Value
2022-12-31,"$-204,000.00"
2023-12-31,"$-271,000.00"
2024-12-31,"$-222,000.00"
2025-12-31,"$-185,000.00"



📆 Último período: 2025-12-31
💵 Valor (Issuance / Repayment of Debt): $-185,000.00


**Depreciação e Amortização**


In [28]:
# =========================
# CÉLULA — Depreciação & Amortização (anual) via yfinance
# =========================
import re, math
import pandas as pd
import numpy as np

def _fmt_money(x):
    try:
        if x is None or (isinstance(x,float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.0f}"
    except:
        return "-"

def _ensure_cf_long():
    # Usa o que já temos; se faltar, baixa do yfinance
    if "CF_ANNUAL_LONG" in globals() and CF_ANNUAL_LONG is not None and not CF_ANNUAL_LONG.empty:
        return CF_ANNUAL_LONG.copy()
    if "CF_ANNUAL_WIDE" in globals() and CF_ANNUAL_WIDE is not None and not CF_ANNUAL_WIDE.empty:
        w = CF_ANNUAL_WIDE.copy()
        value_cols = [c for c in w.columns if c != "LineItem"]
        long = (w.melt(id_vars=["LineItem"], value_vars=value_cols,
                       var_name="Period", value_name="Value")
                 .dropna(subset=["Value"])
                 .reset_index(drop=True))
        try:
            long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        except Exception:
            long["Period"] = long["Period"].astype(str)
        try:
            long["Period_dt"] = pd.to_datetime(long["Period"])
            long = long.sort_values(["Period_dt","Period"], ascending=[True, True])
        except Exception:
            pass
        long["norm"] = long["LineItem"].str.strip().str.lower()
        return long

    # fallback: baixar agora
    try:
        import yfinance as yf
        _ticker = (LAST.get("ticker") if "LAST" in globals() and isinstance(LAST, dict) and LAST.get("ticker") else symbol)
        tk = yf.Ticker(_ticker)
        cf = tk.cashflow
        if cf is None or cf.empty:
            return None
        w = cf.copy()
        w["LineItem"] = w.index.astype(str)
        value_cols = [c for c in w.columns if c != "LineItem"]
        long = (w.melt(id_vars=["LineItem"], value_vars=value_cols,
                       var_name="Period", value_name="Value")
                 .dropna(subset=["Value"])
                 .reset_index(drop=True))
        try:
            long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        except Exception:
            long["Period"] = long["Period"].astype(str)
        try:
            long["Period_dt"] = pd.to_datetime(long["Period"])
            long = long.sort_values(["Period_dt","Period"], ascending=[True, True])
        except Exception:
            pass
        long["norm"] = long["LineItem"].str.strip().str.lower()
        return long
    except Exception:
        return None

def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    cand = long_df[long_df["norm"].apply(lambda n: any(r.search(n) for r in inc))].copy()
    if cand.empty:
        return pd.Series(dtype=float)

    # escolhe o line item com mais períodos
    best = cand.groupby("LineItem").size().sort_values().index[-1]
    sub = cand[cand["LineItem"]==best].copy()
    try:
        sub = sub.sort_values(["Period_dt","Period"], ascending=[True,True])
    except Exception:
        pass
    s = pd.Series(sub["Value"].astype(float).values,
                  index=sub["Period"].astype(str).values,
                  name=best)
    return s

# ---- carregar cash flow longo
cfL = _ensure_cf_long()
_ticker = (LAST.get("ticker") if "LAST" in globals() and isinstance(LAST, dict) and LAST.get("ticker") else symbol)

if cfL is None or cfL.empty:
    print("❌ Cash Flow anual não disponível. Rode a célula de CF primeiro.")
else:
    # 1) tenta linha combinada (mais comum no yfinance)
    INC_DA_COMBINED = [
        r"^depreciation\s+and\s+amortization$",
        r"^depreciation,\s*amortization\s+and\s+accretion$",
        r"^depreciation\s+amortization\s+and\s+accretion$",
        r"^amortization\s+and\s+depreciation$",
    ]
    s_DA = _series_by_regex(cfL, INC_DA_COMBINED)

    # 2) se não existir, soma componentes
    if s_DA.empty:
        INC_DEP = [r"^depreciation$"]
        INC_AMO = [r"^amortization$", r"^amortization\s+of\s+intangibles$"]
        INC_DEPLET = [r"^depletion$"]

        s_dep = _series_by_regex(cfL, INC_DEP)
        s_amo = _series_by_regex(cfL, INC_AMO)
        s_dep_letion = _series_by_regex(cfL, INC_DEPLET)

        # alinhamento por período (index)
        df_sum = pd.DataFrame({
            "Depreciation": s_dep,
            "Amortization": s_amo,
            "Depletion": s_dep_letion
        }).fillna(0.0)

        if not df_sum.empty:
            s_DA = df_sum.sum(axis=1)
            s_DA.name = "Depreciation + Amortization (+ Depletion)"

    if s_DA.empty:
        print(f"⚠️ Não encontrei D&A no Cash Flow anual para {_ticker}.")
    else:
        # ordenar cronologicamente
        try:
            idx_ord = pd.to_datetime(s_DA.index).argsort()
            s_DA = s_DA.iloc[idx_ord]
        except Exception:
            pass

        # últimos períodos (até 4)
        s_last = s_DA.tail(4)
        print(f"🛠️ Depreciação & Amortização — {_ticker} (anual)")
        display(pd.DataFrame({"D&A": s_last.apply(_fmt_money)}))

        # último valor (para valuation)
        DA_last_period = s_DA.index[-1]
        DA_last_value  = float(s_DA.iloc[-1])
        print(f"\n📆 Último período: {DA_last_period}")
        print(f"💵 D&A (último anual): {_fmt_money(DA_last_value)}")

        # Variáveis globais úteis
        DepAmort_last_value = DA_last_value
        DepAmort_series = s_DA  # série completa para análises (growth, normalização EBITDA etc.)


🛠️ Depreciação & Amortização — MLI (anual)


,D&A
2022-12-31,"$43,731,000"
2023-12-31,"$39,954,000"
2024-12-31,"$53,133,000"
2025-12-31,"$68,561,000"



📆 Último período: 2025-12-31
💵 D&A (último anual): $68,561,000


**CAPEX**

In [29]:
# ============================================
# CÉLULA — CAPEX (anual e TTM) via yfinance
# ============================================
import pandas as pd, numpy as np, re, math

def _fmt_money(x):
    try:
        if x is None or (isinstance(x,float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.0f}"
    except:
        return "-"

def _to_long_from_wide(wide):
    if wide is None or wide.empty:
        return None
    w = wide.copy()
    w["LineItem"] = w.index.astype(str)
    val_cols = [c for c in w.columns if c != "LineItem"]
    long = (w.melt(id_vars=["LineItem"], value_vars=val_cols,
                   var_name="Period", value_name="Value")
             .dropna(subset=["Value"])
             .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except Exception:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    """Devolve a série Period->Value do line item que tiver MAIS períodos e casar o regex."""
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    cand = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if cand.empty:
        return pd.Series(dtype=float)
    best_name = cand.groupby("LineItem").size().sort_values().index[-1]
    sub = cand[cand["LineItem"]==best_name].copy()
    try:
        sub = sub.sort_values(["Period_dt","Period"])
    except Exception:
        pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best_name)

# --- pegar dados do yfinance
import yfinance as yf
_ticker = (LAST.get("ticker") if "LAST" in globals() and isinstance(LAST, dict) and LAST.get("ticker") else symbol)
tk = yf.Ticker(_ticker)

cf_annual_w = tk.cashflow          # anual
cf_quarter_w = tk.quarterly_cashflow  # trimestral

cf_annual_L = _to_long_from_wide(cf_annual_w)
cf_quarter_L = _to_long_from_wide(cf_quarter_w)

# --- aliases comuns para CAPEX
INC_CAPEX = [
    r"^capital\s+expenditures?$",
    r"^capital\s+expenditure$",
    r"^purchase[s]?\s+of\s+property\s*,?\s*plant\s*&?\s*equipment$",
    r"^investment[s]?\s+in\s+property\s*,?\s*plant\s*&?\s*equipment$",
    r"^additions?\s+to\s+property\s*,?\s*plant\s*&?\s*equipment$",
]

# séries anual e trimestral
s_capex_a = _series_by_regex(cf_annual_L, INC_CAPEX)
s_capex_q = _series_by_regex(cf_quarter_L, INC_CAPEX)

if s_capex_a.empty and s_capex_q.empty:
    print(f"❌ Não encontrei CAPEX no Cash Flow para {_ticker}.")
else:
    # ---- ANUAL: últimos 4 períodos
    capex_a = s_capex_a.copy()
    try:
        capex_a = capex_a.iloc[pd.to_datetime(capex_a.index).argsort()]
    except Exception:
        pass
    capex_a_last4 = capex_a.tail(4)

    # sinal original e outflow positivo
    df_a = pd.DataFrame({
        "CapEx (sinal original)": capex_a_last4,
        "CapEx (outflow +)": capex_a_last4.abs()
    })
    print(f"🏗️ CAPEX — Anual ({_ticker})")
    display(df_a.applymap(_fmt_money))

    # variáveis úteis (anual)
    Capex_annual_last_outflow = float(df_a["CapEx (outflow +)"].iloc[-1]) if not df_a.empty else None
    Capex_annual_prev_outflow = float(df_a["CapEx (outflow +)"].iloc[-2]) if len(df_a)>=2 else None
    Capex_annual_last_period  = df_a.index[-1] if not df_a.empty else None
    Capex_annual_prev_period  = df_a.index[-2] if len(df_a)>=2 else None

    print("\n✅ Resumo Anual:")
    print(f"• Último:   {Capex_annual_last_period}  →  {_fmt_money(Capex_annual_last_outflow)} (outflow +)")
    if Capex_annual_prev_outflow is not None:
        print(f"• Anterior: {Capex_annual_prev_period}  →  {_fmt_money(Capex_annual_prev_outflow)} (outflow +)")

    # ---- TTM (trimestral): soma dos últimos 4 trimestres (em módulo/saída positiva)
    Capex_TTM_outflow = None
    if not s_capex_q.empty:
        try:
            s_capex_q = s_capex_q.iloc[pd.to_datetime(s_capex_q.index).argsort()]
        except Exception:
            pass
        last4_q = s_capex_q.tail(4)
        if not last4_q.empty:
            Capex_TTM_outflow = float(last4_q.abs().sum())
            print(f"\n🕒 CAPEX TTM (últimos 4 trimestres): {_fmt_money(Capex_TTM_outflow)}")
    else:
        print("\nℹ️ Sem série trimestral de CAPEX para calcular TTM.")

    # deixar as variáveis globais para uso no valuation
    Capex_TTM_outflow_value   = Capex_TTM_outflow
    Capex_annual_last_outflow_value = Capex_annual_last_outflow
    Capex_annual_prev_outflow_value = Capex_annual_prev_outflow


🏗️ CAPEX — Anual (MLI)


,CapEx (sinal original),CapEx (outflow +)
2022-12-31,"$-37,639,000","$37,639,000"
2023-12-31,"$-54,025,000","$54,025,000"
2024-12-31,"$-80,203,000","$80,203,000"
2025-12-31,"$-68,805,000","$68,805,000"



✅ Resumo Anual:
• Último:   2025-12-31  →  $68,805,000 (outflow +)
• Anterior: 2024-12-31  →  $80,203,000 (outflow +)

🕒 CAPEX TTM (últimos 4 trimestres): $69,449,000


**Aquisição / Alienação de Subsidiárias**

In [30]:
# ================================
# CÉLULA — Aquisição / Alienação de Subsidiárias (anual) via yfinance
# ================================
import re, math
import pandas as pd
import numpy as np

def _fmt_money(x):
    try:
        if x is None or (isinstance(x,float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.0f}"
    except:
        return "-"

def _to_long_from_wide(w):
    if w is None or w.empty:
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"], ascending=[True,True])
    except Exception:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes, exclude_regexes=()):
    """Retorna a série Period->Value do LineItem com MAIOR cobertura que bate include e não bate exclude."""
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    exc = [re.compile(p, re.I) for p in exclude_regexes] if exclude_regexes else []

    def ok(name):
        if not any(r.search(name) for r in inc):
            return False
        if any(r.search(name) for r in exc):
            return False
        return True

    cand = long_df[ long_df["norm"].apply(ok) ].copy()
    if cand.empty:
        return pd.Series(dtype=float)

    best_name = cand.groupby("LineItem").size().sort_values().index[-1]
    sub = cand[cand["LineItem"]==best_name].copy()
    try:
        sub = sub.sort_values(["Period_dt","Period"], ascending=[True,True])
    except Exception:
        pass
    s = pd.Series(sub["Value"].astype(float).values,
                  index=sub["Period"].astype(str).values,
                  name=best_name)
    return s

# --- carregar cash flow anual do yfinance
import yfinance as yf
_ticker = (LAST.get("ticker") if "LAST" in globals() and isinstance(LAST, dict) and LAST.get("ticker") else symbol)
tk = yf.Ticker(_ticker)
cf_annual_w = tk.cashflow  # DataFrame (index=line items, cols=anos)
cfL = _to_long_from_wide(cf_annual_w)

if cfL is None or cfL.empty:
    print("❌ Cash Flow anual não disponível. Rode a célula de CF ou verifique o ticker.")
else:
    # --------- ALIASES ----------
    # Aquisições (normalmente saídas negativas)
    INC_ACQ = [
        r"^acquisitions?,?\s*(net.*cash.*acquired)?$",
        r"^acquisitions?\s+net\s+of\s+cash\s+acquired$",
        r"^acquisition\s+of\s+business(es)?(,?\s*net\s+of\s+cash\s+acquired)?$",
        r"^purchase[s]?\s+of\s+business(es)?$",
        r"^investments?\s+in\s+subsidiar(y|ies)$",
        r"^purchase\s+of\s+subsidiar(y|ies)$",
        r"^business\s+combinations?$",
    ]
    # Alienações/Divestitures (normalmente entradas positivas)
    INC_DISP = [
        r"^proceeds\s+from\s+sale\s+of\s+business(es)?$",
        r"^sale\s+of\s+subsidiar(y|ies)$",
        r"^dispositions?\s+of\s+business(es)?$",
        r"^divestitures?$",
        r"^disposal\s+of\s+subsidiar(y|ies)$",
        r"^proceeds\s+from\s+disposals?$",
    ]
    EXC_NONE = []

    # Séries
    s_acq  = _series_by_regex(cfL, INC_ACQ, EXC_NONE)   # esperado: negativo (outflow)
    s_disp = _series_by_regex(cfL, INC_DISP, EXC_NONE)  # esperado: positivo (inflow)

    # Monta DataFrame alinhado por período
    df = pd.DataFrame({
        "Aquisições (sinal da origem)": s_acq,   # geralmente negativo
        "Alienações (sinal da origem)": s_disp,  # geralmente positivo
    })

    # Ordena cronologicamente
    try:
        order = pd.to_datetime(df.index)
        df = df.iloc[order.argsort()]
    except Exception:
        pass

    # Últimos 4 períodos
    df4 = df.tail(4).copy()

    # Outflows/inflows normalizados (+)
    df4["Aquisições (outflow +)"] = df4["Aquisições (sinal da origem)"].apply(lambda x: abs(x) if pd.notna(x) else np.nan)
    df4["Alienações (inflow +)"]  = df4["Alienações (sinal da origem)"].apply(lambda x: x if (pd.notna(x) and x>0) else (abs(x) if pd.notna(x) and x<0 else np.nan))

    # M&A Líquido = alienações (como vier) + aquisições (como vier)
    df4["M&A Líquido (CF)"] = df4[["Aquisições (sinal da origem)","Alienações (sinal da origem)"]].fillna(0).sum(axis=1)

    # Apresentação
    pretty = df4.copy()
    for c in pretty.columns:
        pretty[c] = pretty[c].apply(_fmt_money)

    print(f"🤝 Aquisição / Alienação de Subsidiárias — {_ticker} (anual)")
    display(pretty)

    # Variáveis úteis (último período)
    last_period = df4.index[-1] if not df4.empty else None
    Acquisitions_last_raw  = float(df4["Aquisições (sinal da origem)"].iloc[-1]) if (not df4.empty and pd.notna(df4["Aquisições (sinal da origem)"].iloc[-1])) else None
    Disposals_last_raw     = float(df4["Alienações (sinal da origem)"].iloc[-1])  if (not df4.empty and pd.notna(df4["Alienações (sinal da origem)"].iloc[-1])) else None
    MAndA_net_last         = float(df4["M&A Líquido (CF)"].iloc[-1])               if (not df4.empty and pd.notna(df4["M&A Líquido (CF)"].iloc[-1])) else None

    Acquisitions_last_outflow = abs(Acquisitions_last_raw) if Acquisitions_last_raw is not None else None
    Disposals_last_inflow     = (Disposals_last_raw if (Disposals_last_raw is not None and Disposals_last_raw>0)
                                 else (abs(Disposals_last_raw) if (Disposals_last_raw is not None and Disposals_last_raw<0) else None))

    print("\n✅ Resumo (último anual):")
    print(f"• Período: {last_period}")
    print(f"  - Aquisições (sinal origem): {_fmt_money(Acquisitions_last_raw)}  | Outflow (+): {_fmt_money(Acquisitions_last_outflow)}")
    print(f"  - Alienações (sinal origem): {_fmt_money(Disposals_last_raw)}     | Inflow  (+): {_fmt_money(Disposals_last_inflow)}")
    print(f"  - M&A Líquido (CF):          {_fmt_money(MAndA_net_last)}")

    # Exportar variáveis globais p/ valuation
    Acquisitions_last_raw_value   = Acquisitions_last_raw
    Disposals_last_raw_value      = Disposals_last_raw
    MAndA_net_last_value          = MAndA_net_last
    Acquisitions_last_outflow_value = Acquisitions_last_outflow
    Disposals_last_inflow_value     = Disposals_last_inflow


🤝 Aquisição / Alienação de Subsidiárias — MLI (anual)


,Aquisições (sinal da origem),Alienações (sinal da origem),Aquisições (outflow +),Alienações (inflow +),M&A Líquido (CF)
2022-12-31,$0,-,$0,-,$0
2023-12-31,"$-3,999,000",-,"$3,999,000",-,"$-3,999,000"
2024-12-31,"$-611,392,000",-,"$611,392,000",-,"$-611,392,000"
2025-12-31,"$-17,902,000",-,"$17,902,000",-,"$-17,902,000"



✅ Resumo (último anual):
• Período: 2025-12-31
  - Aquisições (sinal origem): $-17,902,000  | Outflow (+): $17,902,000
  - Alienações (sinal origem): -     | Inflow  (+): -
  - M&A Líquido (CF):          $-17,902,000


**Investimentos**

In [31]:
# ================================================
# CÉLULA — Investimentos (linhas do CF via yfinance)
# Prioriza: "Net Investment Purchase And Sale" (líquido)
# Fallback: Purchases / Proceeds de investimentos
# ================================================
import re, math
import pandas as pd
import numpy as np
import yfinance as yf

def _fmt_money(x):
    try:
        if x is None or (isinstance(x,float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.0f}"
    except:
        return "-"

def _to_long_from_wide(w):
    if w is None or w.empty:
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"], ascending=[True,True])
    except Exception:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    """Série Period->Value do LineItem com MAIS períodos que casa os regex."""
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    cand = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if cand.empty:
        return pd.Series(dtype=float)
    best_name = cand.groupby("LineItem").size().sort_values().index[-1]
    sub = cand[cand["LineItem"]==best_name].copy()
    try:
        sub = sub.sort_values(["Period_dt","Period"], ascending=[True,True])
    except Exception:
        pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best_name)

# --- pegar Cash Flow anual do yfinance
_ticker = (LAST.get("ticker") if "LAST" in globals() and isinstance(LAST, dict) and LAST.get("ticker") else symbol)
tk = yf.Ticker(_ticker)
cf_w = tk.cashflow
cfL = _to_long_from_wide(cf_w)

if cfL is None or cfL.empty:
    print("❌ Cash Flow anual não disponível. Verifique o ticker ou rode a célula de CF.")
else:
    # 1) Linha líquida (prioritária)
    INC_NET_INV = [
        r"^net\s+investment\s+purchase\s+and\s+sale$",
        r"^net\s+purchases?\s+and\s+sales?\s+of\s+investments$",
        r"^net\s+investment\s+activity$",
    ]
    s_net = _series_by_regex(cfL, INC_NET_INV)

    # 2) Fallback: compras e proceeds separados
    INC_PURCHASES_INV = [
        r"^purchases?\s+of\s+investments$",
        r"^purchase\s+of\s+investments$",
        r"^additions?\s+to\s+investments$",
        r"^investments?\s+purchased$",
        r"^net\s+purchases?\s+of\s+investments$",
    ]
    INC_PROCEEDS_INV = [
        r"^proceeds?\s+from\s+sale\s+of\s+investments$",
        r"^proceeds?\s+from\s+maturities\s+of\s+investments$",
        r"^proceeds?\s+from\s+sales?\s+and\s+maturities\s+of\s+investments$",
        r"^maturities?\s+of\s+investments$",
        r"^sales?\s+of\s+investments$",
        r"^net\s+proceeds?\s+from\s+investments$",
    ]
    s_purch = _series_by_regex(cfL, INC_PURCHASES_INV)
    s_proc  = _series_by_regex(cfL, INC_PROCEEDS_INV)

    # Montagem e exibição
    if not s_net.empty:
        # Ordena e mostra últimos 4 períodos
        try: s_net = s_net.iloc[pd.to_datetime(s_net.index).argsort()]
        except: pass
        last4 = s_net.tail(4)
        print(f"📊 Investimentos — {_ticker} (linha líquida do CF: Net Investment Purchase And Sale)")
        display(pd.DataFrame({"Net Investment Purchase And Sale": last4.apply(_fmt_money)}))

        # Variáveis úteis (último anual)
        InvestmentsNet_line_last_period = last4.index[-1]
        InvestmentsNet_line_last_value  = float(last4.iloc[-1])
        print(f"\n✅ Último anual: {InvestmentsNet_line_last_period} → {_fmt_money(InvestmentsNet_line_last_value)}")

        # Também calcula versões normalizadas (outflow/inflow positivos)
        InvestmentsNet_outflow_pos = abs(InvestmentsNet_line_last_value) if InvestmentsNet_line_last_value < 0 else 0.0
        InvestmentsNet_inflow_pos  = InvestmentsNet_line_last_value if InvestmentsNet_line_last_value > 0 else 0.0

        # Exporta variáveis
        InvestmentsNet_last_value        = InvestmentsNet_line_last_value  # sinal contábil
        InvestmentsNet_last_outflow_pos  = InvestmentsNet_outflow_pos
        InvestmentsNet_last_inflow_pos   = InvestmentsNet_inflow_pos

    elif (not s_purch.empty) or (not s_proc.empty):
        # Constrói DF alinhado e calcula líquido
        df = pd.DataFrame({
            "Purchases of Investments (orig.)": s_purch,  # tipicamente negativo
            "Proceeds from Investments (orig.)": s_proc,  # tipicamente positivo
        })
        try:
            order = pd.to_datetime(df.index)
            df = df.iloc[order.argsort()]
        except Exception:
            pass
        df4 = df.tail(4).copy()

        # normalizações
        df4["Purchases (outflow +)"] = df4["Purchases of Investments (orig.)"].apply(lambda x: abs(x) if pd.notna(x) else np.nan)
        df4["Proceeds (inflow +)"]   = df4["Proceeds from Investments (orig.)"].apply(
            lambda x: (x if (pd.notna(x) and x>0) else (abs(x) if (pd.notna(x) and x<0) else np.nan))
        )
        df4["Investimentos Líquido (CF)"] = df4[["Proceeds from Investments (orig.)","Purchases of Investments (orig.)"]].fillna(0).sum(axis=1)

        print(f"📈 Investimentos — {_ticker} (construído de Purchases/Proceeds)")
        display(df4.applymap(_fmt_money))

        # Variáveis úteis (último anual)
        last_period = df4.index[-1]
        InvestmentsNet_last_value = float(df4["Investimentos Líquido (CF)"].iloc[-1]) if pd.notna(df4["Investimentos Líquido (CF)"].iloc[-1]) else None
        PurchasesInvest_last_outflow_value = float(df4["Purchases (outflow +)"].iloc[-1]) if pd.notna(df4["Purchases (outflow +)"].iloc[-1]) else None
        ProceedsInvest_last_inflow_value   = float(df4["Proceeds (inflow +)"].iloc[-1]) if pd.notna(df4["Proceeds (inflow +)"].iloc[-1]) else None

        print("\n✅ Último anual (construído):")
        print(f"• Período: {last_period}")
        print(f"  - Investimentos Líquido (CF): {_fmt_money(InvestmentsNet_last_value)}")
        print(f"  - Compras (outflow +): {_fmt_money(PurchasesInvest_last_outflow_value)}")
        print(f"  - Proceeds (inflow +): {_fmt_money(ProceedsInvest_last_inflow_value)}")

    else:
        print(f"⚠️ Não encontrei nem a linha líquida nem as linhas de Purchases/Proceeds para {_ticker}.")


📊 Investimentos — MLI (linha líquida do CF: Net Investment Purchase And Sale)


,Net Investment Purchase And Sale
2022-12-31,"$-217,863,000"
2023-12-31,"$167,086,000"
2024-12-31,"$70,355,000"
2025-12-31,"$16,907,000"



✅ Último anual: 2025-12-31 → $16,907,000


**Outras atividades de investimento**

In [32]:
# ================================
# CÉLULA — Outras atividades de investimento (Net Other Investing Changes) — anual via yfinance
# ================================
import re, math
import pandas as pd
import numpy as np

def _fmt_money(x):
    try:
        if x is None or (isinstance(x,float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.0f}"
    except:
        return "-"

def _to_long_from_wide(w):
    if w is None or w.empty:
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"], ascending=[True,True])
    except Exception:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    """Série Period->Value do line item com MAIS períodos que casa os regex."""
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    cand = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if cand.empty:
        return pd.Series(dtype=float)
    best_name = cand.groupby("LineItem").size().sort_values().index[-1]
    sub = cand[cand["LineItem"]==best_name].copy()
    try:
        sub = sub.sort_values(["Period_dt","Period"], ascending=[True,True])
    except Exception:
        pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best_name)

# --- usar CF que você já tem; se não, baixar
try:
    cf_long = CF_ANNUAL_LONG.copy()
except NameError:
    cf_long = None

try:
    cf_wide = CF_ANNUAL_WIDE.copy()
except NameError:
    cf_wide = None

if (cf_long is None or cf_long.empty):
    if (cf_wide is not None and not cf_wide.empty):
        cf_long = _to_long_from_wide(cf_wide)
    else:
        import yfinance as yf
        _ticker = (LAST.get("ticker") if "LAST" in globals() and isinstance(LAST, dict) and LAST.get("ticker") else symbol)
        tk = yf.Ticker(_ticker)
        cf_wide = tk.cashflow
        cf_long = _to_long_from_wide(cf_wide)

if cf_long is None or cf_long.empty:
    print("❌ Cash Flow anual não disponível. Rode a célula de CF primeiro.")
else:
    # Aliases comuns no yfinance para “outras atividades de investimento”
    INC_OTHER_INV = [
        r"^net\s+other\s+investing\s+changes$",
        r"^other\s+investing\s+activities$",
        r"^other\s+investing$",
        r"^other\s+investing\s+cash\s+flows$"
    ]

    s_other_inv = _series_by_regex(cf_long, INC_OTHER_INV)

    if s_other_inv.empty:
        print("⚠️ Não encontrei 'Net Other Investing Changes' no CF anual para este ticker.")
    else:
        # ordenar e pegar últimos 4 anos
        try:
            s_other_inv = s_other_inv.iloc[pd.to_datetime(s_other_inv.index).argsort()]
        except Exception:
            pass
        last4 = s_other_inv.tail(4)

        print(f"🧩 Outras atividades de investimento — {(LAST.get('ticker') if 'LAST' in globals() and LAST.get('ticker') else symbol)} (anual)")
        display(pd.DataFrame({"Net Other Investing Changes": last4.apply(_fmt_money)}))

        # último valor (contábil — sinal como reportado)
        NetOtherInvesting_last_period = last4.index[-1]
        NetOtherInvesting_last_value  = float(last4.iloc[-1])

        print(f"\n✅ Último anual: {NetOtherInvesting_last_period} → {_fmt_money(NetOtherInvesting_last_value)}")

        # Versões “positivadas” (opcional, para magnitude)
        NetOtherInvesting_outflow_pos = abs(NetOtherInvesting_last_value) if NetOtherInvesting_last_value < 0 else 0.0
        NetOtherInvesting_inflow_pos  = NetOtherInvesting_last_value if NetOtherInvesting_last_value > 0 else 0.0

        # Exporta variáveis globais (para valuation)
        NetOtherInvesting_last_value_contabil = NetOtherInvesting_last_value
        NetOtherInvesting_last_outflow_pos    = NetOtherInvesting_outflow_pos
        NetOtherInvesting_last_inflow_pos     = NetOtherInvesting_inflow_pos


🧩 Outras atividades de investimento — MLI (anual)


,Net Other Investing Changes
2022-12-31,"$11,204,000"
2023-12-31,"$24,925,000"
2024-12-31,"$14,305,000"
2025-12-31,"$44,889,000"



✅ Último anual: 2025-12-31 → $44,889,000


**Fluxo de Caixa das atividades de investimentos**


In [33]:
# ================================================
# CÉLULA — FCAI (Investing Cash Flow) com os 4 itens especificados
# CAPEX (sinal original) + Aquisições (sinal origem) + Net Investment Purchase And Sale + Net Other Investing Changes
# ================================================
import re, math
import pandas as pd
import numpy as np

def _fmt(x):
    try:
        if x is None or (isinstance(x,float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return "-"

def _to_long_from_wide(w):
    if w is None or w.empty: return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"], ascending=[True,True])
    except Exception:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    """Série Period->Value do line item com MAIOR cobertura que casa os regex."""
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    cand = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if cand.empty:
        return pd.Series(dtype=float)
    best_name = cand.groupby("LineItem").size().sort_values().index[-1]
    sub = cand[cand["LineItem"]==best_name].copy()
    try:
        sub = sub.sort_values(["Period_dt","Period"], ascending=[True,True])
    except Exception:
        pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best_name)

# 1) Tenta reutilizar valores globais já existentes; caso não existam, vamos buscar no yfinance.
#    CAPEX precisa ser "sinal original" (não o outflow positivo). Se não existir, extraímos de novo.
CAPEX_original_last = globals().get("Capex_annual_last_original_value", None)  # se você já tiver salvo em alguma célula
Acquisitions_last_raw = globals().get("Acquisitions_last_raw_value", None)
InvestmentsNet_last   = globals().get("InvestmentsNet_last_value", None)
NetOtherInv_last      = globals().get("NetOtherInvesting_last_value_contabil", None)

need_fetch = any(v is None for v in [CAPEX_original_last, Acquisitions_last_raw, InvestmentsNet_last, NetOtherInv_last])

if need_fetch:
    # carregar yfinance cashflow anual
    import yfinance as yf
    _ticker = (LAST.get("ticker") if "LAST" in globals() and isinstance(LAST, dict) and LAST.get("ticker") else symbol)
    tk = yf.Ticker(_ticker)
    cf_w = tk.cashflow
    cfL = _to_long_from_wide(cf_w)

    if cfL is None or cfL.empty:
        raise RuntimeError("Cash Flow anual indisponível no yfinance. Rode a célula de CF antes.")

    # Regex para cada linha
    INC_CAPEX = [
        r"^capital\s+expenditures?$",
        r"^capital\s+expenditure$",
        r"^purchase[s]?\s+of\s+property\s*,?\s*plant\s*&?\s*equipment$",
        r"^investment[s]?\s+in\s+property\s*,?\s*plant\s*&?\s*equipment$",
        r"^additions?\s+to\s+property\s*,?\s*plant\s*&?\s*equipment$",
    ]
    INC_ACQ = [
        r"^acquisitions?,?\s*(net.*cash.*acquired)?$",
        r"^acquisitions?\s+net\s+of\s+cash\s+acquired$",
        r"^acquisition\s+of\s+business(es)?(,?\s*net\s+of\s+cash\s+acquired)?$",
        r"^purchase[s]?\s+of\s+business(es)?$",
        r"^investments?\s+in\s+subsidiar(y|ies)$",
        r"^purchase\s+of\s+subsidiar(y|ies)$",
        r"^business\s+combinations?$",
    ]
    INC_NET_INV = [
        r"^net\s+investment\s+purchase\s+and\s+sale$",
        r"^net\s+purchases?\s+and\s+sales?\s+of\s+investments$",
        r"^net\s+investment\s+activity$",
    ]
    INC_OTHER_INV = [
        r"^net\s+other\s+investing\s+changes$",
        r"^other\s+investing\s+activities$",
        r"^other\s+investing$",
        r"^other\s+investing\s+cash\s+flows$"
    ]
    INC_NET_CFI = [
        r"^net\s+cash\s+(used\s+for|from)\s+investing\s+activities$",
        r"^net\s+cash\s+provided\s+by\s+investing\s+activities$",
    ]

    # séries
    s_capex = _series_by_regex(cfL, INC_CAPEX)
    s_acq   = _series_by_regex(cfL, INC_ACQ)
    s_netinv= _series_by_regex(cfL, INC_NET_INV)
    s_other = _series_by_regex(cfL, INC_OTHER_INV)
    s_cfi   = _series_by_regex(cfL, INC_NET_CFI)  # para conferência

    # últimos valores (sinal contábil)
    if CAPEX_original_last is None and not s_capex.empty:
        CAPEX_original_last = float(s_capex.iloc[-1])
    if Acquisitions_last_raw is None and not s_acq.empty:
        Acquisitions_last_raw = float(s_acq.iloc[-1])
    if InvestmentsNet_last is None and not s_netinv.empty:
        InvestmentsNet_last = float(s_netinv.iloc[-1])
    if NetOtherInv_last is None and not s_other.empty:
        NetOtherInv_last = float(s_other.iloc[-1])

    # último CFI (para checagem)
    CFI_reported_last = float(s_cfi.iloc[-1]) if (s_cfi is not None and not s_cfi.empty) else None
else:
    # Se já havia tudo, pegamos também CFI (se possível) para conferência
    try:
        import yfinance as yf
        _ticker = (LAST.get("ticker") if "LAST" in globals() and LAST.get("ticker") else symbol)
        tk = yf.Ticker(_ticker)
        cf_w = tk.cashflow
        cfL = _to_long_from_wide(cf_w)
        INC_NET_CFI = [
            r"^net\s+cash\s+(used\s+for|from)\s+investing\s+activities$",
            r"^net\s+cash\s+provided\s+by\s+investing\s+activities$",
        ]
        s_cfi = _series_by_regex(cfL, INC_NET_CFI)
        CFI_reported_last = float(s_cfi.iloc[-1]) if (s_cfi is not None and not s_cfi.empty) else None
    except Exception:
        CFI_reported_last = None

# Converte None -> 0 para somatório seguro (e manter sinal contábil)
capex_last   = CAPEX_original_last   if CAPEX_original_last   is not None else 0.0
acq_last     = Acquisitions_last_raw if Acquisitions_last_raw is not None else 0.0
netinv_last  = InvestmentsNet_last   if InvestmentsNet_last   is not None else 0.0
otherinv_last= NetOtherInv_last      if NetOtherInv_last      is not None else 0.0

# FCAI = soma dos quatro itens (sinal contábil de cada um)
FCAI = capex_last + acq_last + netinv_last + otherinv_last

# Impressão com detalhamento
print("🧾 FCAI — Fluxo de Caixa das Atividades de Investimento (último ano)\n")
print(f"CAPEX (sinal original):                 {_fmt(capex_last)}")
print(f"Aquisições (sinal origem):              {_fmt(acq_last)}")
print(f"Net Investment Purchase And Sale:       {_fmt(netinv_last)}")
print(f"Net Other Investing Changes:            {_fmt(otherinv_last)}")
print("---------------------------------------------------------------")
print(f"✅ FCAI calculado (soma):               {_fmt(FCAI)}")

if 'CFI_reported_last' in globals() and CFI_reported_last is not None:
    delta = FCAI - CFI_reported_last
    print(f"\n📎 CFI reportado no CF (yfinance):      {_fmt(CFI_reported_last)}")
    print(f"🔍 Diferença (calc - reportado):        {_fmt(delta)}")
    if abs(delta) > 1e-6:
        print("⚠️ Observação: o CFI reportado inclui outros itens/ajustes não somados aqui.")
    else:
        print("✅ Checagem: soma bate com o CFI reportado.")

# Variável global para usar no valuation
FCAI_last_value = FCAI


🧾 FCAI — Fluxo de Caixa das Atividades de Investimento (último ano)

CAPEX (sinal original):                 $-68,805,000.00
Aquisições (sinal origem):              $-17,902,000.00
Net Investment Purchase And Sale:       $16,907,000.00
Net Other Investing Changes:            $44,889,000.00
---------------------------------------------------------------
✅ FCAI calculado (soma):               $-24,911,000.00


**Índice de Sustentabilidade de Investimentos (ISI)**

In [34]:
# ============================================
# ISI ROBUSTO — usa variáveis globais, yfinance e/ou reconstrução do FCAI
# ============================================
import numpy as np
import pandas as pd
import re
import yfinance as yf

def _fmt(x):
    try:
        return f"${x:,.2f}" if pd.notnull(x) else "-"
    except:
        return "-"

def _to_long_from_wide(w):
    if w is None or w.empty: return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"]   = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"]= pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except: pass
    long["norm"] = long["LineItem"].str.lower().str.strip()
    return long

def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    cand = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if cand.empty:
        return pd.Series(dtype=float)
    best_name = cand.groupby("LineItem").size().sort_values().index[-1]
    sub = cand[cand["LineItem"]==best_name].copy()
    try: sub = sub.sort_values(["Period_dt","Period"])
    except: pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best_name)

def _interpret_isi(isi):
    if not np.isfinite(isi):
        return "⚠️ Dados insuficientes."
    if isi > 0.1:
        return "🚨 Desinvestimento: vendendo ativos/reduzindo investimento."
    if -0.3 <= isi <= 0.0:
        return "🟡 Investimento baixo: crescimento pode estar limitado."
    if -1.3 <= isi < -0.3:
        return "🟢 Reinvestimento saudável: equilíbrio entre CFO e FCAI."
    if isi < -1.3:
        return "🔴 Investimento excessivo: atenção a pressões de caixa/dívida."
    return "⚙️ Situação neutra."

# 1) Tenta usar variáveis globais já existentes
CFO_last   = globals().get("CFO_value_annual_last", None)
FCAI_last  = globals().get("FCAI_last_value", None)

# 2) Se precisar, busca no yfinance
if CFO_last is None or FCAI_last is None:
    _ticker = symbol
    tk = yf.Ticker(_ticker)
    cf_w = tk.cashflow
    cfL  = _to_long_from_wide(cf_w)

    if cfL is not None and not cfL.empty:
        # CFO: net cash from operating activities
        INC_CFO = [
            r"^net\s+cash\s+(provided\s+by|from)\s+operating\s+activities$",
            r"^net\s+cash\s+used\s+for\s+operating\s+activities$",
            r"^operating\s+cash\s+flow$",
        ]
        s_CFO = _series_by_regex(cfL, INC_CFO)
        if CFO_last is None and not s_CFO.empty:
            CFO_last = float(s_CFO.iloc[-1])

        # FCAI reportado: net cash from investing
        INC_CFI = [
            r"^net\s+cash\s+(used\s+for|from)\s+investing\s+activities$",
            r"^net\s+cash\s+provided\s+by\s+investing\s+activities$",
        ]
        s_CFI = _series_by_regex(cfL, INC_CFI)
        if FCAI_last is None and not s_CFI.empty:
            FCAI_last = float(s_CFI.iloc[-1])

# 3) Se ainda faltar o FCAI, tenta reconstruir pela soma dos 4 itens que você usa
if FCAI_last is None:
    capex_last     = globals().get("CAPEX_original_last", None)
    acq_last       = globals().get("Acquisitions_last_raw_value", None)
    netinv_last    = globals().get("InvestmentsNet_last_value", None)
    otherinv_last  = globals().get("NetOtherInvesting_last_value_contabil", None)

    parts = [x for x in [capex_last, acq_last, netinv_last, otherinv_last] if x is not None]
    if parts:
        FCAI_last = float(np.nansum(parts))

# 4) Calcula o ISI se possível
if CFO_last is None or CFO_last == 0 or FCAI_last is None:
    print("📊 MÉTRICA: Índice de Sustentabilidade de Investimentos (ISI)\n")
    print(f"CFO (Operating CF): {_fmt(CFO_last) if CFO_last is not None else '-'}")
    print(f"FCAI (Investing CF): {_fmt(FCAI_last) if FCAI_last is not None else '-'}\n")
    print("🔹 ISI = FCAI / CFO = nan")
    print("\n🧠 Interpretação: ⚠️ Dados insuficientes.")
else:
    ISI = FCAI_last / CFO_last
    print("📊 MÉTRICA: Índice de Sustentabilidade de Investimentos (ISI)\n")
    print(f"CFO (Operating CF): {_fmt(CFO_last)}")
    print(f"FCAI (Investing CF): {_fmt(FCAI_last)}")
    print(f"\n🔹 ISI = FCAI / CFO = {ISI:.2f}")
    print(f"\n🧠 Interpretação: {_interpret_isi(ISI)}")

    # disponibiliza variável global
    ISI_last_value = ISI


📊 MÉTRICA: Índice de Sustentabilidade de Investimentos (ISI)

CFO (Operating CF): $755,444,000.00
FCAI (Investing CF): $-24,911,000.00

🔹 ISI = FCAI / CFO = -0.03

🧠 Interpretação: 🟡 Investimento baixo: crescimento pode estar limitado.


# **DRE**

In [35]:
# =========================================================
# CÉLULA — DRE anual (yfinance) para Valuation
# Entregas principais (último anual):
# Receita, Custos, Resultado Bruto, EBIT, EBITA, IR (Tax)
# + extras: EBITDA, Lucro Líquido, EPS, Ações
# =========================================================
import pandas as pd, numpy as np, re, math
import yfinance as yf

# ---------------- Utils ----------------
def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "-"
        return f"${float(x):,.0f}"
    except:
        return "-"

def _to_wide(df):
    if df is None or df.empty: return None
    w = df.copy()
    try:
        w = w.loc[:, sorted(w.columns)]
    except Exception:
        pass
    w["LineItem"] = w.index.astype(str)
    w.reset_index(drop=True, inplace=True)
    cols = ["LineItem"] + [c for c in w.columns if c != "LineItem"]
    return w[cols]

def _to_long(wide):
    if wide is None or wide.empty: return None
    value_cols = [c for c in wide.columns if c != "LineItem"]
    long = (wide.melt(id_vars=["LineItem"], value_vars=value_cols,
                      var_name="Period", value_name="Value")
            .dropna(subset=["Value"])
            .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"], ascending=[True, True])
    except Exception:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes, exclude_regexes=()):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    exc = [re.compile(p, re.I) for p in exclude_regexes] if exclude_regexes else []
    def ok(name):
        if not any(r.search(name) for r in inc): return False
        if any(r.search(name) for r in exc):     return False
        return True
    cand = long_df[ long_df["norm"].apply(ok) ].copy()
    if cand.empty: return pd.Series(dtype=float)
    best_name = cand.groupby("LineItem").size().sort_values().index[-1]
    sub = cand[cand["LineItem"]==best_name].copy()
    try:
        sub = sub.sort_values(["Period_dt","Period"], ascending=[True, True])
    except Exception:
        pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best_name)

# ---------------- Baixa dados ----------------
_ticker = symbol.upper()
tk = yf.Ticker(_ticker)

is_a_raw = tk.financials               # DRE anual
cf_a_raw = tk.cashflow                 # DFC anual (p/ D&A/Amortização se precisar)

is_wide = _to_wide(is_a_raw)
is_long = _to_long(is_wide)

cf_wide = _to_wide(cf_a_raw)
cf_long = _to_long(cf_wide)

# ---------------- Aliases (Income Statement) ----------------
INC_REVENUE   = [r"^total\s+revenue$", r"^revenues?$", r"^sales$"]
INC_COSTREV   = [r"^cost\s+of\s+revenue$", r"^cost\s+of\s+goods\s+sold$"]
INC_GROSS     = [r"^gross\s+profit$"]
INC_OPERATING = [r"^operating\s+income$", r"^income\s+from\s+operations$", r"^ebit$"]

# ⚠️ IR/Tax — incluir "Tax Provision" (yfinance) e variações
INC_TAX = [
    r"^tax\s+provision$",
    r"^provision\s+for\s+income\s+tax(es)?$",
    r"^income\s+tax(es)?\s+(expense|provision)$",
]

INC_NETINC    = [r"^net\s+income$", r"^net\s+income\s+common$", r"^net\s+income\s+applicable\s+to\s+common\s+shares$"]
INC_EBITDA    = [r"^ebitda$"]

# Amortização isolada (se existir no IS ou no CF)
INC_AMORT_IS  = [r"^amortization$", r"^amortization\s+of\s+intangibles$", r"^amortization\s+expense$"]
INC_AMORT_CF  = [r"^amortization$", r"^amortization\s+of\s+intangibles$", r"^amortization\s+expense$"]
INC_DA_CF     = [r"^depreciation\s+and\s+amortization$", r"^depreciation,\s*amortization\s+and\s+accretion$"]

# ---------------- Extrai séries ----------------
s_rev    = _series_by_regex(is_long, INC_REVENUE)
s_cost   = _series_by_regex(is_long, INC_COSTREV)
s_gross  = _series_by_regex(is_long, INC_GROSS)
s_ebit   = _series_by_regex(is_long, INC_OPERATING)
s_tax    = _series_by_regex(is_long, INC_TAX)         # <— atualizado para "Tax Provision"
s_net    = _series_by_regex(is_long, INC_NETINC)
s_ebitda = _series_by_regex(is_long, INC_EBITDA)

# Amortização: tenta no IS; se não, tenta no CF
s_amort_is = _series_by_regex(is_long, INC_AMORT_IS)
s_amort_cf = _series_by_regex(cf_long,  INC_AMORT_CF)
s_amort    = s_amort_is if not s_amort_is.empty else s_amort_cf

# D&A pelo CF (caso precise estimar EBITDA)
s_da_cf = _series_by_regex(cf_long, INC_DA_CF)

# ---------------- Monta DF anual (últimos 4) ----------------
df_is = pd.DataFrame({
    "Receita": s_rev,
    "Custos": s_cost,
    "Resultado Bruto": s_gross,
    "EBIT": s_ebit,
    "IR (Tax Provision)": s_tax,        # <— nome exibido
    "Lucro Líquido": s_net,
})

# EBITDA: usa reportado, senão tenta EBIT + D&A
if not s_ebitda.empty:
    df_is["EBITDA"] = s_ebitda
elif (not s_da_cf.empty) and (not s_ebit.empty):
    tmp = pd.DataFrame({"EBIT": s_ebit}).join(pd.DataFrame({"D&A": s_da_cf}), how="left")
    df_is["EBITDA"] = tmp["EBIT"] + tmp["D&A"]

# EBITA = EBIT + Amortization (se Amortização isolada existir)
if (not s_ebit.empty) and (not s_amort.empty):
    tmp2 = pd.DataFrame({"EBIT": s_ebit}).join(pd.DataFrame({"Amort": s_amort}), how="left")
    df_is["EBITA"] = tmp2["EBIT"] + tmp2["Amort"]
else:
    df_is["EBITA"] = np.nan  # se não der para isolar Amortização, mantém vazio

# EPS/Ações (se disponível)
INC_EPS_DIL   = [r"^diluted\s+eps$", r"^eps\s+diluted$"]
INC_SH_DIL    = [r"^diluted\s+average\s+shares$", r"^weighted\s+average\s+shares\s+diluted$"]
s_eps_dil     = _series_by_regex(is_long, INC_EPS_DIL)
s_sh_dil      = _series_by_regex(is_long, INC_SH_DIL)
df_is["EPS (Dil.)"]     = s_eps_dil
df_is["Ações (Dil.)"]   = s_sh_dil

# Ordena por período e recorta últimos 4 anos
try:
    order = pd.to_datetime(df_is.index)
    df_is = df_is.iloc[order.argsort()]
except Exception:
    pass
df_last4 = df_is.tail(4).copy()

# ---------------- Apresentação ----------------
pretty = df_last4.copy()
money_cols = ["Receita","Custos","Resultado Bruto","EBIT","EBITA","EBITDA","IR (Tax Provision)","Lucro Líquido"]
for c in money_cols:
    if c in pretty.columns:
        pretty[c] = pretty[c].apply(_fmt_money)

print(f"📄 DRE (anual) — {_ticker} | Últimos períodos")
display(pretty)

# ---------------- Variáveis para Valuation (último anual disponível) ----------------
if not df_last4.empty:
    last_period = df_last4.index[-1]

    def _safe_get(series_name):
        return float(df_last4[series_name].iloc[-1]) if (series_name in df_last4.columns and pd.notna(df_last4[series_name].iloc[-1])) else None

    Receita_last         = _safe_get("Receita")
    Custos_last          = _safe_get("Custos")
    ResultadoBruto_last  = _safe_get("Resultado Bruto")
    EBIT_last            = _safe_get("EBIT")
    EBITA_last           = _safe_get("EBITA")      # pode ser None se não houver Amortização separada
    EBITDA_last          = _safe_get("EBITDA")     # opcional, útil em valuation
    IR_last              = _safe_get("IR (Tax Provision)")   # <— atualizado
    NetIncome_last       = _safe_get("Lucro Líquido")

    EPS_diluted_last     = _safe_get("EPS (Dil.)")
    Shares_diluted_last  = _safe_get("Ações (Dil.)")

    print("\n✅ Variáveis (último anual) para valuation:")
    print(f"• Período: {last_period}")
    print(f"  - Receita:           {_fmt_money(Receita_last)}")
    print(f"  - Custos:            {_fmt_money(Custos_last)}")
    print(f"  - Resultado Bruto:   {_fmt_money(ResultadoBruto_last)}")
    print(f"  - EBIT:              {_fmt_money(EBIT_last)}")
    print(f"  - EBITA:             {_fmt_money(EBITA_last)}   (pode ser '-' se Amortização não estiver isolada)")
    print(f"  - EBITDA:            {_fmt_money(EBITDA_last)}")
    print(f"  - IR (Tax Provision):{_fmt_money(IR_last)}")
    print(f"  - Lucro Líquido:     {_fmt_money(NetIncome_last)}")
    print(f"  - EPS (Dil.):        {EPS_diluted_last if EPS_diluted_last is not None else '-'}")
    print(f"  - Ações (Dil.):      {Shares_diluted_last if Shares_diluted_last is not None else '-'}")
else:
    print("⚠️ DRE anual vazio ou sem períodos suficientes.")


📄 DRE (anual) — MLI | Últimos períodos


,Receita,Custos,Resultado Bruto,EBIT,IR (Tax Provision),Lucro Líquido,EBITDA,EBITA,EPS (Dil.),Ações (Dil.)
2022-12-31,"$3,982,455,000","$2,864,862,000","$1,117,593,000","$869,478,000","$223,322,000","$658,316,000","$920,572,000","$875,052,000",5.82,113110000.0
2023-12-31,"$3,420,345,000","$2,433,511,000","$986,834,000","$737,883,000","$220,762,000","$602,897,000","$886,407,000","$742,888,000",5.30,113662000.0
2024-12-31,"$3,768,766,000","$2,724,328,000","$1,044,438,000","$762,391,000","$205,076,000","$604,879,000","$874,005,000","$776,324,000",5.31,113965000.0
2025-12-31,"$4,178,547,000","$2,966,083,000","$1,212,464,000","$893,101,000","$247,351,000","$765,191,000","$1,081,031,000","$913,871,000",6.86,111492000.0



✅ Variáveis (último anual) para valuation:
• Período: 2025-12-31
  - Receita:           $4,178,547,000
  - Custos:            $2,966,083,000
  - Resultado Bruto:   $1,212,464,000
  - EBIT:              $893,101,000
  - EBITA:             $913,871,000   (pode ser '-' se Amortização não estiver isolada)
  - EBITDA:            $1,081,031,000
  - IR (Tax Provision):$247,351,000
  - Lucro Líquido:     $765,191,000
  - EPS (Dil.):        6.86
  - Ações (Dil.):      111492000.0


# **NOPAT Restrito**

In [36]:
# =========================================================
# CÉLULA — Cálculo do NOPAT Restrito
# Fórmula: NOPAT Restrito = EBIT - IR (Tax Provision)
# =========================================================

try:
    # Garantindo que as variáveis necessárias existam
    if 'EBIT_last' not in globals() or EBIT_last is None:
        raise ValueError("EBIT_last não está definido. Execute a célula da DRE primeiro.")
    if 'IR_last' not in globals() or IR_last is None:
        raise ValueError("IR_last não está definido. Execute a célula da DRE primeiro.")

    # Calcula o NOPAT Restrito
    NOPAT_Restrito = EBIT_last - IR_last

    # Formatação para exibição
    if NOPAT_Restrito == 0 or NOPAT_Restrito is None:
        NOPAT_Restrito_formatted = "$ 0.00"
    else:
        NOPAT_Restrito_formatted = "${:,.2f}".format(NOPAT_Restrito)

    # Exibição do resultado
    print(f"💰 NOPAT Restrito (EBIT - IR): {NOPAT_Restrito_formatted}")

except Exception as e:
    print("⚠️ Erro ao calcular o NOPAT Restrito:", e)


💰 NOPAT Restrito (EBIT - IR): $645,750,000.00


**Despesas Financeiras**

In [37]:
# =========================================================
# CÉLULA — Despesa Financeira (Interest Expense / Investment Losses)
# Fonte: DRE (yfinance)
# =========================================================

try:
    # Usa a estrutura da DRE já baixada
    if 'is_long' not in globals() or is_long.empty:
        raise ValueError("DRE (is_long) não está carregada. Execute a célula principal da DRE antes.")

    # Procurar por Interest Expense ou variações equivalentes
    FIN_EXP = [
        r"^interest\s+expense$",
        r"^investment\s+loss(es)?$",
        r"^non[-\s]?operating\s+income\s*\(loss\)$"
    ]

    s_finexp = None
    for pat in FIN_EXP:
        s_finexp = _series_by_regex(is_long, [pat])
        if not s_finexp.empty:
            break

    if s_finexp is None or s_finexp.empty:
        raise ValueError("Nenhuma linha de despesa financeira encontrada.")

    # Selecionar o último valor anual
    Despesa_Financeira_last = float(s_finexp.iloc[-1])

    # Formatar saída
    if Despesa_Financeira_last == 0:
        Despesa_Financeira_formatted = "$ 0.00"
    else:
        Despesa_Financeira_formatted = f"${Despesa_Financeira_last:,.2f}"

    print(f"💸 Despesa Financeira (último anual): {Despesa_Financeira_formatted}")

except Exception as e:
    print("⚠️ Erro ao extrair Despesa Financeira:", e)


💸 Despesa Financeira (último anual): $108,000.00


**FCO**

In [38]:
# ============================================
# CÉLULA — FCO (robusto p/ bancos e não-bancos)
# - Não-financeiras: FCO ≈ (EBIT − Tax) + D&A
# - Financeiras: FCO = CFO reportado (fallback: Net Income + D&A)
# Saídas globais: FCO_last_value, NOPAT_Restrito, EBIT_last, IR_last
# ============================================
import re, numpy as np, pandas as pd, yfinance as yf
from datetime import datetime

def _fmt(x):
    try:
        return f"${x:,.2f}" if x is not None and np.isfinite(x) else "-"
    except:
        return "-"

def _to_long_from_wide(w):
    if w is None or w.empty: return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    # ordenar por período se possível
    try:
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    sel = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if sel.empty:
        return pd.Series(dtype=float)
    # pega a linha mais "popular"/completa
    best = sel.groupby("LineItem").size().sort_values().index[-1]
    sub = sel[ sel["LineItem"]==best ].copy()
    try: sub = sub.sort_values(["Period_dt","Period"])
    except: pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best)

def _is_financial(info_dict):
    s = (info_dict or {}).get("sector","") or ""
    ind = (info_dict or {}).get("industry","") or ""
    txt = (s + " " + ind).lower()
    keys = ["financial", "bank", "capital markets", "asset management", "insurance", "thrifts", "mortgage"]
    return any(k in txt for k in keys)

# 0) Preparação
_ticker = symbol.upper()
tk = yf.Ticker(_ticker)
info = getattr(tk, "info", {}) or {}

is_fin = _is_financial(info)

# 1) Income Statement (wide -> long)
IS_w = tk.income_stmt if tk.income_stmt is not None and not tk.income_stmt.empty else None
IS_l = _to_long_from_wide(IS_w)

# 2) Cash Flow (wide -> long)
CF_w = tk.cashflow if tk.cashflow is not None and not tk.cashflow.empty else None
CF_l = _to_long_from_wide(CF_w)

# 3) Depreciação & Amortização (D&A)
INC_DA = [
    r"^depreciation\s+and\s+amortization$",
    r"^depreciation,\s*amortization\s+and\s+accretion$",
]
s_DA = _series_by_regex(CF_l, INC_DA)

DA_last = None
if not s_DA.empty:
    DA_last = float(s_DA.iloc[-1])
else:
    # fallback: somar separadas
    s_dep  = _series_by_regex(CF_l, [r"^depreciation$", r"^depreciation\s+expense$", r"^depreciation\s+and\s+depletion$"])
    s_amor = _series_by_regex(CF_l, [r"^amortization$", r"^amortization\s+expense$", r"^amortization\s+of\s+intangibles$"])
    dep_last  = float(s_dep.iloc[-1])  if not s_dep.empty  else 0.0
    amor_last = float(s_amor.iloc[-1]) if not s_amor.empty else 0.0
    DA_last = dep_last + amor_last if (dep_last or amor_last) else None

# 4) CFO reportado (para bancos, é o principal)
INC_CFO = [
    r"^net\s+cash\s+(provided\s+by|from)\s+operating\s+activities$",
    r"^net\s+cash\s+used\s+for\s+operating\s+activities$",
    r"^operating\s+cash\s+flow$",
]
s_CFO = _series_by_regex(CF_l, INC_CFO)
CFO_reported_last = float(s_CFO.iloc[-1]) if not s_CFO.empty else None

# 5) EBIT e IR (apenas p/ não-financeiras)
EBIT_last, IR_last, NOPAT_Restrito = None, None, None

if not is_fin:
    # EBIT (vários labels possíveis)
    INC_EBIT = [
        r"^operating\s+income$",
        r"^ebit$",
        r"^earnings\s+before\s+interest\s+and\s+tax(es)?$",
    ]
    s_EBIT = _series_by_regex(IS_l, INC_EBIT)
    if not s_EBIT.empty:
        EBIT_last = float(s_EBIT.iloc[-1])

    # IR (Tax Provision)
    INC_TAX = [r"^tax\s+provision$", r"^income\s+tax(es)?\s+expense$"]
    s_TAX = _series_by_regex(IS_l, INC_TAX)
    if not s_TAX.empty:
        IR_last = float(s_TAX.iloc[-1])

    # NOPAT restrito
    if EBIT_last is not None and IR_last is not None:
        NOPAT_Restrito = EBIT_last - IR_last
    else:
        # fallback: usar Net Income se faltar algum (menos preciso)
        INC_NI = [r"^net\s+income$"]
        s_NI = _series_by_regex(IS_l, INC_NI)
        if not s_NI.empty:
            NOPAT_Restrito = float(s_NI.iloc[-1])

    # FCO proxy
    FCO = None
    if (NOPAT_Restrito is not None) and (DA_last is not None):
        FCO = NOPAT_Restrito + DA_last
    # se CFO reportado existir, mostramos a comparação
    CFO_for_compare = CFO_reported_last

else:
    # FINANCEIRAS: usar CFO reportado como FCO
    FCO = CFO_reported_last
    # fallback se CFO não existir: Net Income + D&A (proxy bem grosseiro)
    if FCO is None:
        INC_NI = [r"^net\s+income$"]
        s_NI = _series_by_regex(IS_l, INC_NI)
        NI_last = float(s_NI.iloc[-1]) if not s_NI.empty else None
        if NI_last is not None and DA_last is not None:
            FCO = NI_last + DA_last
    CFO_for_compare = None  # já é o próprio

# 6) Impressão
print(f"🧾 FCO — modo {'FINANCEIRO' if is_fin else 'NÃO-FINANCEIRO'} | {symbol}")
if not is_fin:
    print(f"EBIT_last:           {_fmt(EBIT_last)}")
    print(f"IR_last (Tax Prov.): {_fmt(IR_last)}")
    print(f"NOPAT Restrito:      {_fmt(NOPAT_Restrito)}")
else:
    print("Modelo banco: usa CFO reportado como FCO (ou NI + D&A se faltar CFO).")

print(f"D&A (anual):         {_fmt(DA_last)}")
print("-------------------------------------------------")
print(f"✅ FCO (final):       {_fmt(FCO)}")

if (not is_fin) and (CFO_for_compare is not None):
    delta = (NOPAT_Restrito + (DA_last or 0)) - CFO_for_compare
    print(f"\n📎 CFO reportado:     {_fmt(CFO_for_compare)}")
    print(f"🔍 Diferença (proxy - reportado): {_fmt(delta)}")
    print("ℹ️ É normal haver diferença: CFO inclui Δ capital de giro e outros ajustes além de D&A.")

# 7) Exporta pro escopo global
globals().update({
    "FCO_last_value": FCO,
    "NOPAT_Restrito": NOPAT_Restrito,
    "EBIT_last": EBIT_last,
    "IR_last": IR_last
})


🧾 FCO — modo NÃO-FINANCEIRO | MLI
EBIT_last:           $893,101,000.00
IR_last (Tax Prov.): $247,351,000.00
NOPAT Restrito:      $645,750,000.00
D&A (anual):         $68,561,000.00
-------------------------------------------------
✅ FCO (final):       $714,311,000.00

📎 CFO reportado:     $755,444,000.00
🔍 Diferença (proxy - reportado): $-41,133,000.00
ℹ️ É normal haver diferença: CFO inclui Δ capital de giro e outros ajustes além de D&A.


**Δ NC Giro**

In [39]:
# =========================================================
# CÉLULA — NCG via yfinance (anual): TCA_op, TCL_op, ΔNCG
# =========================================================
import re, math
import numpy as np
import pandas as pd
import yfinance as yf

def _fmt(x):
    try:
        return f"${float(x):,.2f}" if x is not None and np.isfinite(x) else "-"
    except Exception:
        return "-"

def _to_long_from_wide(w):
    if w is None or w.empty:
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    # normaliza período e nomes
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except Exception:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    """Retorna série Period->Value do line item com MAIOR cobertura que casa os regex (ignora nomes exatos do Yahoo)."""
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    cand = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if cand.empty:
        return pd.Series(dtype=float)
    best_name = cand.groupby("LineItem").size().sort_values().index[-1]
    sub = cand[cand["LineItem"]==best_name].copy()
    try:
        sub = sub.sort_values(["Period_dt","Period"])
    except Exception:
        pass
    s = pd.Series(sub["Value"].astype(float).values,
                  index=sub["Period"].astype(str).values,
                  name=best_name)
    return s

def _last_and_prev(series):
    """Retorna (último_valor, último_periodo, valor_anterior, período_anterior)."""
    if series is None or series.empty:
        return (None, None, None, None)
    s = series.copy()
    # já está ordenado crescente; último é o mais recente
    last_val, last_per = float(s.iloc[-1]), s.index[-1]
    prev_val, prev_per = (float(s.iloc[-2]), s.index[-2]) if len(s) >= 2 else (None, None)
    return (last_val, last_per, prev_val, prev_per)

# 1) Baixar balanço anual do yfinance
_ticker = symbol.upper()
tk = yf.Ticker(_ticker)
bs_wide = tk.balance_sheet  # anual
if bs_wide is None or bs_wide.empty:
    raise RuntimeError("Balanço anual indisponível no yfinance. Verifique o ticker.")

# 2) Long format + regex para itens
bs_long = _to_long_from_wide(bs_wide)

REG_TCA = [
    r"^total\s+current\s+assets$",
    r"^current\s+assets$",
]
REG_CASH = [
    r"^cash\s+and\s+cash\s+equivalents$",
    r"^cash$",
    r"^cash\s*,?\s*cash\s+equivalents\s+and\s+short\s+term\s+investments$",
    r"^cash\s+and\s+short\s+term\s+investments$",
]
REG_TCL = [
    r"^total\s+current\s+liabilities$",
    r"^current\s+liabilities$",
]
REG_CURR_DEBT = [
    r"^current\s+debt$",
    r"^current\s+portion\s+of\s+long[-\s]?term\s+debt$",
    r"^current\s+portion\s+long[-\s]?term\s+debt$",
    r"^short\s+term\s+debt$",
    r"^short[-\s]?term\s+borrowings$",
]

s_TCA   = _series_by_regex(bs_long, REG_TCA)
s_cash  = _series_by_regex(bs_long, REG_CASH)
s_TCL   = _series_by_regex(bs_long, REG_TCL)
s_cdebt = _series_by_regex(bs_long, REG_CURR_DEBT)

# 3) Último e anterior (quando disponível)
TCA_last,  TCA_per,  TCA_prev,  TCA_per_prev   = _last_and_prev(s_TCA)
Cash_last, Cash_per, Cash_prev, Cash_per_prev  = _last_and_prev(s_cash)
TCL_last,  TCL_per,  TCL_prev,  TCL_per_prev   = _last_and_prev(s_TCL)
Cdebt_last,Cdebt_per,Cdebt_prev,Cdebt_per_prev = _last_and_prev(s_cdebt)

# 4) Ajustes (None -> 0 para cálculos) e composição operacional
c_last_cash   = 0.0 if (Cash_last  is None or math.isnan(Cash_last)) else Cash_last
c_prev_cash   = 0.0 if (Cash_prev  is None or math.isnan(Cash_prev)) else Cash_prev
c_last_cdebt  = 0.0 if (Cdebt_last is None or math.isnan(Cdebt_last)) else Cdebt_last
c_prev_cdebt  = 0.0 if (Cdebt_prev is None or math.isnan(Cdebt_prev)) else Cdebt_prev

# TCA_op = Current Assets - Cash-like
TCA_op_last = (0.0 if TCA_last is None or math.isnan(TCA_last) else TCA_last) - c_last_cash
TCA_op_prev = (0.0 if TCA_prev is None or math.isnan(TCA_prev) else TCA_prev) - c_prev_cash

# TCL_op = Current Liabilities - Current Debt-like
TCL_op_last = (0.0 if TCL_last is None or math.isnan(TCL_last) else TCL_last) - c_last_cdebt
TCL_op_prev = (0.0 if TCL_prev is None or math.isnan(TCL_prev) else TCL_prev) - c_prev_cdebt

# 5) NCG e variação
NCG_last = TCA_op_last - TCL_op_last
NCG_prev = TCA_op_prev - TCL_op_prev
Delta_NCG_last = NCG_last - (NCG_prev if NCG_prev is not None else 0.0)

# 6) Exibir resultados
print(f"📦 NCG — {_ticker}")
print(f"Período atual (anual): {TCA_per or '-'}   | Período anterior: {TCA_per_prev or '-'}\n")

print("Ativo Circulante (Total Current Assets):")
print(f"  atual: {_fmt(TCA_last)}   | anterior: {_fmt(TCA_prev)}")
print("  (-) Caixa & equivalentes (Cash-like):")
print(f"       atual: {_fmt(Cash_last)} | anterior: {_fmt(Cash_prev)}")
print(f"= Ativo Circulante Operacional (TCA_op):  atual {_fmt(TCA_op_last)} | anterior {_fmt(TCA_op_prev)}\n")

print("Passivo Circulante (Total Current Liabilities):")
print(f"  atual: {_fmt(TCL_last)}   | anterior: {_fmt(TCL_prev)}")
print("  (-) Dívida CP / parcela corrente (Current debt-like):")
print(f"       atual: {_fmt(Cdebt_last)} | anterior: {_fmt(Cdebt_prev)}")
print(f"= Passivo Circulante Operacional (TCL_op): atual {_fmt(TCL_op_last)} | anterior {_fmt(TCL_op_prev)}\n")

print(f"NCG Atual:        {_fmt(NCG_last)}")
print(f"NCG Ano Anterior: {_fmt(NCG_prev)}")
print(f"Δ NCG:            {_fmt(Delta_NCG_last)}")

# 7) Interpretação rápida
if np.isfinite(Delta_NCG_last):
    if Delta_NCG_last > 0:
        print("\n🟠 ΔNCG > 0 → aumento do capital de giro (consome caixa; reduz FCFF).")
    elif Delta_NCG_last < 0:
        print("\n🟢 ΔNCG < 0 → liberação de capital de giro (gera caixa; aumenta FCFF).")
    else:
        print("\n⚪ ΔNCG ~ 0 → impacto neutro no fluxo de caixa.")
else:
    print("\n⚠️ ΔNCG indefinido por falta de dados suficientes.")

# 8) Disponibiliza variáveis globais para valuation
TCA_op_last_global  = TCA_op_last
TCL_op_last_global  = TCL_op_last
NCG_last_global     = NCG_last
NCG_prev_global     = NCG_prev
Delta_NCG_last_global = Delta_NCG_last


📦 NCG — MLI
Período atual (anual): 2025-12-31   | Período anterior: 2024-12-31

Ativo Circulante (Total Current Assets):
  atual: $2,445,745,000.00   | anterior: $2,012,229,000.00
  (-) Caixa & equivalentes (Cash-like):
       atual: $1,389,736,000.00 | anterior: $1,059,103,000.00
= Ativo Circulante Operacional (TCA_op):  atual $1,056,009,000.00 | anterior $953,126,000.00

Passivo Circulante (Total Current Liabilities):
  atual: $413,134,000.00   | anterior: $397,987,000.00
  (-) Dívida CP / parcela corrente (Current debt-like):
       atual: $1,094,000.00 | anterior: $796,000.00
= Passivo Circulante Operacional (TCL_op): atual $412,040,000.00 | anterior $397,191,000.00

NCG Atual:        $643,969,000.00
NCG Ano Anterior: $555,935,000.00
Δ NCG:            $88,034,000.00

🟠 ΔNCG > 0 → aumento do capital de giro (consome caixa; reduz FCFF).


**FCFF**

In [40]:
# =========================================================
# FCFF com CAPEX do yfinance (anual)
# Fórmula: FCFF = FCO - CAPEX_outflow - ΔNCG
# (onde CAPEX_outflow é positivo quando há investimento)
# =========================================================
import re, numpy as np, pandas as pd, yfinance as yf

def _fmt(v):
    try:
        return f"${float(v):,.2f}" if v is not None and np.isfinite(v) else "-"
    except:
        return "-"

# Helpers mínimos (usamos os mesmos padrões/regex que já empregamos)
def _to_long_from_wide(w):
    if w is None or w.empty:
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    sel = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if sel.empty:
        return pd.Series(dtype=float)
    best = sel.groupby("LineItem").size().sort_values().index[-1]
    sub = sel[ sel["LineItem"]==best ].copy()
    try: sub = sub.sort_values(["Period_dt","Period"])
    except: pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best)

try:
    # 1) Garantir insumos calculados previamente
    if 'symbol' not in globals():
        raise ValueError("Ticker 'symbol' não definido.")
    if 'FCO_last_value' not in globals():
        raise ValueError("FCO_last_value não encontrado. Execute a célula do FCO (NOPAT + D&A).")
    if 'Delta_NCG_last_global' not in globals():
        raise ValueError("Delta_NCG_last_global não encontrado. Execute a célula do NCG (yfinance).")

    FCO_val  = 0.0 if FCO_last_value is None or np.isnan(FCO_last_value) else float(FCO_last_value)
    dNCG_val = 0.0 if Delta_NCG_last_global is None or np.isnan(Delta_NCG_last_global) else float(Delta_NCG_last_global)

    # 2) CAPEX via yfinance (Cash Flow anual)
    _ticker = symbol.upper()
    tk = yf.Ticker(_ticker)
    cf_wide = tk.cashflow
    cf_long = _to_long_from_wide(cf_wide)

    # Aliases para CAPEX (yahoo costuma usar "Capital Expenditure")
    INC_CAPEX = [
        r"^capital\s+expenditure(s)?$",
        r"^purchase(s)?\s+of\s+property.*equipment$",   # alguns emitentes
        r"^additions\s+to\s+property.*equipment$",
    ]
    s_capex = _series_by_regex(cf_long, INC_CAPEX)
    if s_capex.empty:
        raise ValueError("CAPEX não encontrado no Cash Flow do yfinance.")

    capex_cf_last = float(s_capex.iloc[-1])   # tipicamente NEGATIVO quando investe
    capex_period  = s_capex.index[-1]

    # Converte para desembolso positivo (outflow)
    # Ex.: se capex_cf_last = -500 → capex_outflow = 500
    capex_outflow = -capex_cf_last if capex_cf_last < 0 else 0.0

    # 3) FCFF = FCO - CAPEX_outflow - ΔNCG
    FCFF = FCO_val - capex_outflow - dNCG_val

    # 4) Exibir
    print(f"💰 FCFF — {_ticker}")
    print(f"Período (anual) referência CAPEX: {capex_period}\n")
    print(f"FCO (NOPAT + D&A):        {_fmt(FCO_val)}")
    print(f"CAPEX (contábil, yfin):   {_fmt(capex_cf_last)}  (negativo = investimento)")
    print(f"CAPEX (outflow positivo): {_fmt(capex_outflow)}")
    print(f"Δ Capital de Giro:        {_fmt(dNCG_val)}")
    print("----------------------------------------------------")
    print(f"✅ FCFF (último ano):      {_fmt(FCFF)}")

    # Interpretação rápida
    if np.isfinite(FCFF):
        if FCFF > 0:
            print("\n🟢 FCFF positivo → geração de caixa após reinvestimentos.")
        elif FCFF < 0:
            print("\n🔴 FCFF negativo → consumo de caixa (investimento elevado ou geração operacional baixa).")
        else:
            print("\n⚪ FCFF ~ 0 → equilíbrio entre operação e reinvestimentos.")
    else:
        print("\n⚠️ FCFF não pôde ser calculado (dados insuficientes).")

    # Variáveis globais para valuation
    CAPEX_cf_last_value   = capex_cf_last       # valor contábil (normalmente negativo)
    CAPEX_outflow_last    = capex_outflow       # desembolso positivo
    FCFF_last_value       = FCFF
    FCFF_period_reference = capex_period

except Exception as e:
    print("⚠️ Erro ao calcular FCFF com CAPEX do yfinance:", e)


💰 FCFF — MLI
Período (anual) referência CAPEX: 2025-12-31

FCO (NOPAT + D&A):        $714,311,000.00
CAPEX (contábil, yfin):   $-68,805,000.00  (negativo = investimento)
CAPEX (outflow positivo): $68,805,000.00
Δ Capital de Giro:        $88,034,000.00
----------------------------------------------------
✅ FCFF (último ano):      $557,472,000.00

🟢 FCFF positivo → geração de caixa após reinvestimentos.


**Novas Dívidas**

In [41]:
# =========================================================
# CÉLULA — Cálculo de NOVAS DÍVIDAS (usando yfinance)
# Fórmula: Novas_Dívidas = Aquisições (sinal origem) + Net Investment Purchase And Sale
# =========================================================
import numpy as np
import pandas as pd
import re, yfinance as yf

def _fmt(v):
    try:
        return f"${float(v):,.2f}" if v is not None and np.isfinite(v) else "-"
    except:
        return "-"

def _to_long_from_wide(w):
    if w is None or w.empty:
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    sel = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if sel.empty:
        return pd.Series(dtype=float)
    best = sel.groupby("LineItem").size().sort_values().index[-1]
    sub = sel[ sel["LineItem"]==best ].copy()
    try: sub = sub.sort_values(["Period_dt","Period"])
    except: pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best)

try:
    if 'symbol' not in globals():
        raise ValueError("Ticker 'symbol' não definido.")

    _ticker = symbol.upper()
    tk = yf.Ticker(_ticker)
    cf_wide = tk.cashflow
    if cf_wide is None or cf_wide.empty:
        raise ValueError("Fluxo de caixa (cash flow) não disponível no yfinance.")

    cf_long = _to_long_from_wide(cf_wide)

    # 1️⃣ Aquisição / Alienação de Subsidiárias
    REG_ACQ = [r"acquisition.*disposition.*subsidiaries", r"acquisition.*subsidiary", r"disposition.*subsidiary"]
    s_acq = _series_by_regex(cf_long, REG_ACQ)
    acq_last = float(s_acq.iloc[-1]) if not s_acq.empty else 0.0
    acq_period = s_acq.index[-1] if not s_acq.empty else "-"

    # 2️⃣ Net Investment Purchase And Sale
    REG_INV = [r"net\s+investment\s+purchase.*sale", r"investments", r"net.*investing.*purchase"]
    s_inv = _series_by_regex(cf_long, REG_INV)
    inv_last = float(s_inv.iloc[-1]) if not s_inv.empty else 0.0
    inv_period = s_inv.index[-1] if not s_inv.empty else "-"

    # 3️⃣ Calcular Novas Dívidas
    Novas_dividas = acq_last + inv_last

    # 4️⃣ Exibir resultados
    print(f"🏦 NOVAS DÍVIDAS — {_ticker}\n")
    print(f"Aquisições/Alienações (Acquisition/Disposition of Subsidiaries): {_fmt(acq_last)}   [{acq_period}]")
    print(f"Investimentos (Net Investment Purchase And Sale):               {_fmt(inv_last)}   [{inv_period}]")
    print("------------------------------------------------------------")
    print(f"✅ Novas Dívidas (último ano):                                  {_fmt(Novas_dividas)}")

    # Interpretação
    if np.isfinite(Novas_dividas):
        if Novas_dividas > 0:
            print("\n🟢 Valor positivo → aumento de recursos investidos / expansão financeira.")
        elif Novas_dividas < 0:
            print("\n🔴 Valor negativo → redução líquida de investimentos ou desinvestimentos.")
        else:
            print("\n⚪ Nenhuma alteração significativa registrada.")
    else:
        print("\n⚠️ Dados insuficientes para interpretar Novas Dívidas.")

    # Variável global para valuation
    Novas_dividas_last_value = Novas_dividas

except Exception as e:
    print("⚠️ Erro ao calcular Novas Dívidas:", e)


🏦 NOVAS DÍVIDAS — MLI

Aquisições/Alienações (Acquisition/Disposition of Subsidiaries): $0.00   [-]
Investimentos (Net Investment Purchase And Sale):               $16,907,000.00   [2025-12-31]
------------------------------------------------------------
✅ Novas Dívidas (último ano):                                  $16,907,000.00

🟢 Valor positivo → aumento de recursos investidos / expansão financeira.


**FCDA**

In [42]:
# =========================================================
# CÉLULA — FCDA (Fluxo de Caixa Disponível aos Acionistas/Devedores, adaptado)
# Fórmula base (conforme sua metodologia):
#     FCDA = FCFF + Novas_Dívidas + IR (Tax Provision)
# ---------------------------------------------------------
# Pré-requisitos calculados em células anteriores:
#   • FCFF_last_value                (FCFF do último ano)
#   • Novas_dividas_last_value       (Aquisições + Net Investment Purchase And Sale)
#   • IR_last                        (Tax Provision do DRE anual)
# =========================================================
import numpy as np

def _fmt(v):
    try:
        return f"${float(v):,.2f}" if v is not None and np.isfinite(v) else "-"
    except:
        return "-"

try:
    # --------- Validações ---------
    needed = ["FCFF_last_value", "Novas_dividas_last_value", "IR_last"]
    for var in needed:
        if var not in globals():
            raise ValueError(f"Variável '{var}' não encontrada. Garanta que as células anteriores foram executadas.")

    # --------- Sanitização ---------
    FCFF_val   = 0.0 if (FCFF_last_value is None or not np.isfinite(float(FCFF_last_value))) else float(FCFF_last_value)
    ND_val     = 0.0 if (Novas_dividas_last_value is None or not np.isfinite(float(Novas_dividas_last_value))) else float(Novas_dividas_last_value)
    IR_val     = 0.0 if (IR_last is None or not np.isfinite(float(IR_last))) else float(IR_last)  # Tax Provision (normalmente positivo = despesa)

    # --------- Cálculo ---------
    FCDA = FCFF_val + ND_val + IR_val

    # --------- Saída ---------
    print("🏁 FCDA — Cálculo consolidado\n")
    print(f"FCFF:                 {_fmt(FCFF_val)}")
    print(f"Novas Dívidas:        {_fmt(ND_val)}   (Aquisições + Net Investment Purchase/Sale)")
    print(f"IR (Tax Provision):   {_fmt(IR_val)}")
    print("----------------------------------------------------")
    print(f"✅ FCDA (último ano): {_fmt(FCDA)}")

    # Interpretação simples
    if np.isfinite(FCDA):
        if FCDA > 0:
            print("\n🟢 FCDA positivo → caixa disponível após FCFF, movimentos de investimentos/financiamentos e imposto provisionado.")
        elif FCDA < 0:
            print("\n🔴 FCDA negativo → consumo líquido de caixa nessa ótica consolidada.")
        else:
            print("\n⚪ FCDA ~ 0 → equilíbrio aproximado.")
    else:
        print("\n⚠️ FCDA não pôde ser interpretado (dados insuficientes).")

    # --------- Exporta variável global ---------
    FCDA_last_value = FCDA

except Exception as e:
    print("⚠️ Erro ao calcular o FCDA:", e)


🏁 FCDA — Cálculo consolidado

FCFF:                 $557,472,000.00
Novas Dívidas:        $16,907,000.00   (Aquisições + Net Investment Purchase/Sale)
IR (Tax Provision):   $247,351,000.00
----------------------------------------------------
✅ FCDA (último ano): $821,730,000.00

🟢 FCDA positivo → caixa disponível após FCFF, movimentos de investimentos/financiamentos e imposto provisionado.


# **CAPM - Custo do Capital Próprio**

**Custo de Capital Próprio**

In [43]:
# =========================================================
# CÉLULA — CAPM (Ke) usando o coletor (fund_df -> metrics["Beta"])
# Ke = Rf + Beta * (Rm - Rf)
# =========================================================
import numpy as np
import pandas as pd

def _grab_beta():
    # 1) do dicionário metrics (coletor)
    if 'metrics' in globals() and isinstance(metrics, dict) and metrics.get("Beta") is not None:
        return metrics["Beta"], "metrics['Beta']"
    # 2) do DataFrame met_df, caso exista
    if 'met_df' in globals() and isinstance(met_df, pd.DataFrame):
        row = met_df.loc[met_df['Metric'].str.lower()=='beta', 'Value']
        if not row.empty and row.iloc[0] is not None:
            return row.iloc[0], "met_df['Beta']"
    # 3) outros nomes comuns
    for nm in ["beta_value","beta_str","Beta","beta"]:
        if nm in globals() and globals()[nm] is not None:
            return globals()[nm], nm
    return None, None

def _pick_first(names):
    for nm in names:
        if nm in globals() and globals()[nm] is not None:
            return globals()[nm], nm
    return None, None

def _to_float_safe(x):
    try:
        if isinstance(x, str):
            s = x.strip().replace('%','').replace(',','.')
            return float(s)
        return float(x)
    except:
        return None

def _pct_to_decimal(x):
    # 7.5 -> 0.075 ; 0.075 -> 0.075
    if x is None: return None
    x = float(x)
    return x if (-1 <= x <= 1) else x/100.0

def _fmt_pct(x):
    try:
        return f"{x*100:,.2f}%"
    except:
        return "-"

try:
    # --------- Beta ---------
    beta_raw, beta_src = _grab_beta()
    beta = _to_float_safe(beta_raw)

    # --------- Rf ---------
    Rf_raw, Rf_src = _pick_first(["Rf_val", "Rf_US", "Rf", "interest_rate"])
    Rf_dec = _pct_to_decimal(_to_float_safe(Rf_raw))

    # --------- Rm ---------
    Rm_raw, Rm_src = _pick_first(["Rm_val", "mean_yearly_return", "sp500_expected_return", "market_return"])
    Rm_dec = _pct_to_decimal(_to_float_safe(Rm_raw))

    missing = []
    if beta is None:   missing.append("Beta (metrics['Beta'])")
    if Rf_dec is None: missing.append("Rf (Rf_val/Rf_US/Rf/interest_rate)")
    if Rm_dec is None: missing.append("Rm (Rm_val/mean_yearly_return)")
    if missing:
        raise ValueError("Faltam variáveis: " + ", ".join(missing) + ".")

    # --------- CAPM ---------
    ERP = Rm_dec - Rf_dec
    Ke_dec = Rf_dec + beta * ERP

    print("📈 CAPM — Custo do Capital Próprio (Ke)\n")
    print(f"Fonte Beta: {beta_src} | Beta = {beta:.3f}")
    print(f"Fonte Rf:   {Rf_src}   | Rf   = {_fmt_pct(Rf_dec)}")
    print(f"Fonte Rm:   {Rm_src}   | Rm   = {_fmt_pct(Rm_dec)}")
    print(f"ERP (Rm - Rf) = {_fmt_pct(ERP)}")
    print("------------------------------------------------")
    print(f"✅ Ke (CAPM) = {_fmt_pct(Ke_dec)}")

    # Exporta para valuation
    Ke_capm = Ke_dec

except Exception as e:
    print("⚠️ Erro ao calcular o CAPM (Ke):", e)


📈 CAPM — Custo do Capital Próprio (Ke)

Fonte Beta: metrics['Beta'] | Beta = 1.090
Fonte Rf:   Rf_val   | Rf   = 4.51%
Fonte Rm:   mean_yearly_return   | Rm   = 13.47%
ERP (Rm - Rf) = 8.96%
------------------------------------------------
✅ Ke (CAPM) = 14.27%


**Ki (Kd) - Custo da Dívida**

In [44]:
# =========================================================
# CÉLULA — Ki (Kd) • Custo da Dívida (anual, yfinance)
# Regras:
#   • Se Despesa de Juros ou Dívida Total indisponível/≈0 → Ki = Rf + inflação
#   • Caso contrário → Ki = |InterestExpense| / TotalDebt
# Observações:
#   • Em yfinance, Interest Expense costuma ser NEGATIVO (saída). Usamos abs().
#   • TotalDebt: tenta "Total Debt"; fallback: (Long-Term Debt + Current Debt/Current Portion)
#   • Usa variáveis já extraídas quando existirem; senão, extrai agora.
# =========================================================
import re, numpy as np, pandas as pd, yfinance as yf

# ---------- helpers mínimos (idempotentes) ----------
def _fmt_pct(x):
    try:
        return f"{x*100:,.2f}%"
    except:
        return "-"

def _fmt_money(x):
    try:
        return f"${float(x):,.2f}"
    except:
        return "-"

def _to_long_from_wide(w):
    if w is None or w.empty:
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    sel = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if sel.empty:
        return pd.Series(dtype=float)
    best = sel.groupby("LineItem").size().sort_values().index[-1]
    sub = sel[ sel["LineItem"]==best ].copy()
    try: sub = sub.sort_values(["Period_dt","Period"])
    except: pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best)

def _to_float_pct_or_dec(x):
    """Aceita '4.5', '4,5', '4.5%', 0.045 → retorna decimal (0.045)."""
    if x is None: return None
    try:
        if isinstance(x, str):
            s = x.strip().replace('%','').replace(',','.')
            val = float(s)
        else:
            val = float(x)
        return val if -1 <= val <= 1 else val/100.0
    except:
        return None

# ---------- coleta/garantia do ticker ----------
if 'symbol' not in globals():
    raise ValueError("Ticker 'symbol' não definido.")
_ticker = symbol.upper()
tk = yf.Ticker(_ticker)

# ---------- 1) Despesa Financeira (último anual) ----------
# Tenta usar variável já calculada; se não, extrai agora
if 'Despesa_Financeira_last' in globals() and Despesa_Financeira_last is not None:
    interest_expense_last = float(Despesa_Financeira_last)
    ie_period = None
else:
    is_wide = tk.financials
    is_long = _to_long_from_wide(is_wide)
    REG_FINEXP = [
        r"^interest\s+expense$",
        r"^non[-\s]?operating\s+income\s*\(loss\)$",  # fallback amplo (cuidado)
        r"^investment\s+loss(es)?$"
    ]
    s_ie = None
    for pat in REG_FINEXP:
        s_ie = _series_by_regex(is_long, [pat])
        if not s_ie.empty:
            break
    if s_ie is None or s_ie.empty:
        interest_expense_last, ie_period = None, None
    else:
        interest_expense_last = float(s_ie.iloc[-1])   # costuma ser NEGATIVO
        ie_period = s_ie.index[-1]

# ---------- 2) Dívida Total (Total Debt) ----------
bs_wide = tk.balance_sheet
bs_long = _to_long_from_wide(bs_wide)

# tenta "Total Debt"
REG_TDEBT = [r"^total\s+debt$"]
s_tdebt = _series_by_regex(bs_long, REG_TDEBT)

if not s_tdebt.empty:
    total_debt_last = float(s_tdebt.iloc[-1])
    td_period = s_tdebt.index[-1]
else:
    # fallback: Long-Term Debt + Current Portion/Short-Term
    REG_LTD   = [r"^long[-\s]?term\s+debt$"]
    REG_CURR  = [r"^current\s+debt$", r"^current\s+portion\s+of\s+long[-\s]?term\s+debt$", r"^short[-\s]?term\s+(debt|borrowings)$"]
    s_ltd  = _series_by_regex(bs_long, REG_LTD)
    s_cdebt = _series_by_regex(bs_long, REG_CURR)
    ltd_last   = float(s_ltd.iloc[-1])   if not s_ltd.empty   else 0.0
    cdebt_last = float(s_cdebt.iloc[-1]) if not s_cdebt.empty else 0.0
    total_debt_last = ltd_last + cdebt_last
    td_period = s_ltd.index[-1] if not s_ltd.empty else (s_cdebt.index[-1] if not s_cdebt.empty else None)

# ---------- 3) Inflação EUA (decimal) ----------
# usa o que já existir no ambiente
infl_raw = None
for nm in ["inflation", "inflation_str", "Inflation_US", "inflation_us", "US_inflation"]:
    if nm in globals() and globals()[nm] is not None:
        infl_raw = globals()[nm]
        break
inflation_dec = _to_float_pct_or_dec(infl_raw)

# ---------- 4) Rf (decimal) para fallback ----------
Rf_raw = None
for nm in ["Rf_val", "Rf_US", "Rf", "interest_rate"]:
    if nm in globals() and globals()[nm] is not None:
        Rf_raw = globals()[nm]
        break
Rf_dec = _to_float_pct_or_dec(Rf_raw)

# ---------- 5) Cálculo do Ki ----------
Ki_dec = None
explain = ""

valid_ie   = (interest_expense_last is not None and np.isfinite(interest_expense_last))
valid_debt = (total_debt_last is not None and np.isfinite(total_debt_last) and abs(total_debt_last) > 1e-6)

if valid_ie and valid_debt:
    # yfinance traz despesa negativa → usa valor absoluto
    Ki_dec = abs(interest_expense_last) / total_debt_last
    explain = "Ki = |Interest Expense| / Total Debt"
else:
    # fallback: Ki = Rf + inflação (sua regra)
    if Rf_dec is not None and inflation_dec is not None:
        Ki_dec = Rf_dec + inflation_dec
        explain = "Ki (fallback) = Rf + inflação (dados de juros/dívida indisponíveis)"
    else:
        explain = "Dados insuficientes para Ki (faltam despesa de juros/dívida e também Rf/inflação)."

# ---------- 6) Saída ----------
print(f"🏦 Ki (Kd) — Custo da Dívida • {_ticker}\n")

print("• Despesa Financeira (últ. anual):", "-" if interest_expense_last is None else _fmt_money(interest_expense_last),
      f"[{ie_period}]" if ie_period else "")
print("• Dívida Total (últ. anual):      ", "-" if total_debt_last is None else _fmt_money(total_debt_last),
      f"[{td_period}]" if td_period else "")
print("• Rf (p/ fallback):               ", "-" if Rf_dec is None else _fmt_pct(Rf_dec))
print("• Inflação EUA (p/ fallback):     ", "-" if inflation_dec is None else _fmt_pct(inflation_dec))
print("------------------------------------------------------------")
if Ki_dec is not None:
    print(f"✅ {explain}")
    print(f"➡️  Ki (Kd) = {_fmt_pct(Ki_dec)}")
else:
    print("⚠️ Não foi possível calcular Ki (Kd).")

# exporta
Ki_value = Ki_dec


🏦 Ki (Kd) — Custo da Dívida • MLI

• Despesa Financeira (últ. anual): $108,000.00 
• Dívida Total (últ. anual):       $27,490,000.00 [2025-12-31]
• Rf (p/ fallback):                4.51%
• Inflação EUA (p/ fallback):      -
------------------------------------------------------------
✅ Ki = |Interest Expense| / Total Debt
➡️  Ki (Kd) = 0.39%


**Estrutura de Capital - Po e PL**

In [45]:
# =========================================================
# CÉLULA — Po (Dívida), PL (Equity) e Pesos de Capital
# Fontes (yfinance, balanço anual):
#   • Po  = Total Debt  (fallback: Long-Term Debt + Current/Short-Term Debt)
#   • PL  = Common Stock Equity  (fallbacks: Total Stockholders' Equity, Shareholder's Equity, etc.)
# Pesos:
#   wE = PL/(Po+PL)   |   wD = Po/(Po+PL)
# =========================================================
import re, numpy as np, pandas as pd, yfinance as yf

def _fmt_money(x):
    try:
        return f"${float(x):,.2f}"
    except:
        return "-"

def _fmt_ratio(x):
    try:
        return f"{float(x)*100:,.2f}%"
    except:
        return "-"

def _to_long_from_wide(w):
    if w is None or w.empty:
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    sel = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if sel.empty:
        return pd.Series(dtype=float)
    # pega o line item com maior cobertura histórica
    best = sel.groupby("LineItem").size().sort_values().index[-1]
    sub = sel[ sel["LineItem"]==best ].copy()
    try: sub = sub.sort_values(["Period_dt","Period"])
    except: pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best)

def _last_val(s):
    if s is None or s.empty:
        return (None, None)
    return float(s.iloc[-1]), s.index[-1]

try:
    if 'symbol' not in globals():
        raise ValueError("Ticker 'symbol' não definido.")
    _ticker = symbol.upper()
    tk = yf.Ticker(_ticker)
    bs_wide = tk.balance_sheet  # anual
    if bs_wide is None or bs_wide.empty:
        raise RuntimeError("Balanço anual indisponível no yfinance.")

    bs_long = _to_long_from_wide(bs_wide)

    # ----- Po: Total Debt (fallbacks) -----
    if 'total_debt_last' in globals() and total_debt_last is not None:
        Po_val, Po_period = float(total_debt_last), None
    else:
        REG_TDEBT = [r"^total\s+debt$"]
        s_tdebt = _series_by_regex(bs_long, REG_TDEBT)
        if not s_tdebt.empty:
            Po_val, Po_period = _last_val(s_tdebt)
        else:
            # Fallback: Long-Term Debt + Current/Short-Term Debt
            REG_LTD   = [r"^long[-\s]?term\s+debt$"]
            REG_CURR  = [r"^current\s+debt$", r"^current\s+portion\s+of\s+long[-\s]?term\s+debt$", r"^short[-\s]?term\s+(debt|borrowings)$"]
            s_ltd  = _series_by_regex(bs_long, REG_LTD)
            s_cdebt = _series_by_regex(bs_long, REG_CURR)
            ltd_last, _   = _last_val(s_ltd)
            cdebt_last, _ = _last_val(s_cdebt)
            Po_val = (ltd_last or 0.0) + (cdebt_last or 0.0)
            # melhor período disponível
            Po_period = (s_ltd.index[-1] if not s_ltd.empty else (s_cdebt.index[-1] if not s_cdebt.empty else None))

    # ----- PL: Common Stock Equity (fallbacks) -----
    if 'CommonEquity_last' in globals() and CommonEquity_last is not None:
        PL_val, PL_period = float(CommonEquity_last), None
    else:
        REG_EQ_PRIMARY = [r"^common\s+stock\s+equity$"]
        s_eq = _series_by_regex(bs_long, REG_EQ_PRIMARY)
        if s_eq.empty:
            REG_EQ_FALL = [
                r"^total\s+stockholders'? equity$",
                r"^total\s+shareholders'? equity$",
                r"^stockholders'? equity$",
                r"^shareholders'? equity$",
                r"^total\s+equity$"
            ]
            s_eq = _series_by_regex(bs_long, REG_EQ_FALL)
        PL_val, PL_period = _last_val(s_eq)

    # ----- Sanitize -----
    Po_val = 0.0 if (Po_val is None or not np.isfinite(Po_val)) else float(Po_val)
    PL_val = 0.0 if (PL_val is None or not np.isfinite(PL_val)) else float(PL_val)

    total_cap = Po_val + PL_val

    wE = (PL_val/total_cap) if total_cap > 0 else 0.0
    wD = (Po_val/total_cap) if total_cap > 0 else 0.0

    # ----- Saída -----
    print(f"🏦 Estrutura de Capital — {_ticker}")
    if Po_period or PL_period:
        print(f"Período ref.: Dívida [{Po_period or '-'}], Equity [{PL_period or '-'}]\n")

    print(f"PL (Equity):    {_fmt_money(PL_val)}")
    print(f"Po (Dívida):    {_fmt_money(Po_val)}")
    print(f"Total (Po+PL):  {_fmt_money(total_cap)}")
    print("------------------------------------------------")
    print(f"Peso do Equity (PL/(Po+PL)):  {_fmt_ratio(wE)}")
    print(f"Peso da Dívida (Po/(Po+PL)):  {_fmt_ratio(wD)}")

    # ----- Exporta globais -----
    Po_value        = Po_val
    PL_value        = PL_val
    We_equity       = wE
    Wd_debt         = wD
    Total_cap_value = total_cap

except Exception as e:
    print("⚠️ Erro ao calcular Po/PL e pesos:", e)


🏦 Estrutura de Capital — MLI
Período ref.: Dívida [-], Equity [2025-12-31]

PL (Equity):    $3,209,966,000.00
Po (Dívida):    $27,490,000.00
Total (Po+PL):  $3,237,456,000.00
------------------------------------------------
Peso do Equity (PL/(Po+PL)):  99.15%
Peso da Dívida (Po/(Po+PL)):  0.85%


**WACC**

In [46]:
# =========================================================
# WACC — versão robusta com cálculo automático do TaxRate via DRE (yfinance)
# =========================================================

import numpy as np
import pandas as pd
import re

def _fmt_pct(x):
    try:
        return f"{x*100:,.2f}%"
    except:
        return "-"

# ---------- Cálculo automático da TaxRate ----------
TaxRate, tax_source = None, None
try:
    tk = yf.Ticker(symbol)
    is_df = tk.financials
    if is_df is not None and not is_df.empty:
        df = is_df.copy()
        df.index = df.index.str.lower().str.strip()

        # procura padrões flexíveis
        tax_candidates = [i for i in df.index if re.search(r"tax", i)]
        pretax_candidates = [i for i in df.index if re.search(r"pretax|income\s+before\s+tax|earnings\s+before\s+tax", i)]

        if tax_candidates and pretax_candidates:
            tax_line = tax_candidates[0]
            pretax_line = pretax_candidates[0]

            # pega o último valor não nulo
            tax_val = df.loc[tax_line].dropna().iloc[-1]
            pretax_val = df.loc[pretax_line].dropna().iloc[-1]

            if pretax_val != 0:
                TaxRate = abs(tax_val / pretax_val)
                tax_source = f"calculado automaticamente ({tax_line} / {pretax_line})"
            else:
                TaxRate = 0.21
                tax_source = "fallback padrão 21% (EBT = 0)"
        else:
            TaxRate = 0.21
            tax_source = "fallback padrão 21% (linhas não encontradas)"
    else:
        TaxRate = 0.21
        tax_source = "fallback padrão 21% (DRE vazio)"
except Exception as e:
    TaxRate = 0.21
    tax_source = f"erro no parser ({e}); fallback 21%"

# ---------- Captura dos valores ----------
Ke_val = next((float(globals()[nm]) for nm in ["Ke_capm", "Ke", "Ke_value"] if nm in globals()), None)
Ki_val = next((float(globals()[nm]) for nm in ["Ki_value", "Ki_dec", "Ki", "Ki_float"] if nm in globals()), None)
We_val = next((float(globals()[nm]) for nm in ["We_equity", "PL_Po_PL"] if nm in globals()), None)
Wd_val = next((float(globals()[nm]) for nm in ["Wd_debt", "Po_Po_PL"] if nm in globals()), None)

missing = [nm for nm, v in {"Ke":Ke_val, "Ki":Ki_val, "We":We_val, "Wd":Wd_val}.items() if v is None]
if missing:
    raise ValueError(f"⚠️ Faltam variáveis para o cálculo do WACC: {', '.join(missing)}")

# ---------- Cálculo ----------
WACC_val = (We_val * Ke_val) + (Wd_val * Ki_val * (1 - TaxRate))

# ---------- Saída ----------
print("🏛️ Weighted Average Cost of Capital (WACC) — versão robusta")
print("------------------------------------------------------")
print(f"Ke (Equity).............: {_fmt_pct(Ke_val)}")
print(f"Ki (Debt)...............: {_fmt_pct(Ki_val)}")
print(f"We (PL/(Po+PL)).........: {_fmt_pct(We_val)}")
print(f"Wd (Po/(Po+PL)).........: {_fmt_pct(Wd_val)}")
print(f"Taxa Efetiva (Tc).......: {_fmt_pct(TaxRate)} ({tax_source})")
print("------------------------------------------------------")
print(f"✅ WACC = {_fmt_pct(WACC_val)}")

# ---------- Exporta ----------
WACC_value = WACC_val


🏛️ Weighted Average Cost of Capital (WACC) — versão robusta
------------------------------------------------------
Ke (Equity).............: 14.27%
Ki (Debt)...............: 0.39%
We (PL/(Po+PL)).........: 99.15%
Wd (Po/(Po+PL)).........: 0.85%
Taxa Efetiva (Tc).......: 0.11% (calculado automaticamente (tax effect of unusual items / pretax income))
------------------------------------------------------
✅ WACC = 14.16%


**Indicadores Conceituais (Co, EVA, ROE>Ke, ROI>WACC)**

In [47]:
# =========================================================
# 📊 ANÁLISE CONCEITUAL — Avaliação Econômica e Eficiência do Capital
# Integrada ao pipeline yfinance + fund_df
# =========================================================
import numpy as np
import pandas as pd
import yfinance as yf
import re

def _fmt_money(x):
    try: return f"${float(x):,.2f}"
    except: return "-"

def _fmt_pct_dec(x):
    try: return f"{float(x)*100:,.2f}%"
    except: return "-"

def _to_long_from_wide(w):
    if w is None or w.empty:
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    sel = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if sel.empty:
        return pd.Series(dtype=float)
    best = sel.groupby("LineItem").size().sort_values().index[-1]
    sub = sel[ sel["LineItem"]==best ].copy()
    try: sub = sub.sort_values(["Period_dt","Period"])
    except: pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best)

def _get_PL_value():
    if 'PL_value' in globals() and PL_value is not None:
        return float(PL_value)
    tk = yf.Ticker(symbol)
    bs_long = _to_long_from_wide(tk.balance_sheet)
    for r in [r"^common\s+stock\s+equity$", r"^total\s+(stockholders|shareholders)'\s? equity$", r"^total\s+equity$"]:
        s = _series_by_regex(bs_long, [r])
        if not s.empty:
            return float(s.iloc[-1])
    return None

def _get_LL_value():
    if 'NetIncome_last' in globals() and NetIncome_last is not None:
        return float(NetIncome_last)
    tk = yf.Ticker(symbol)
    is_long = _to_long_from_wide(tk.financials)
    for r in [r"^net\s+income$", r"^net\s+income\s+applicable.*common", r"^consolidated\s+net\s+income$"]:
        s = _series_by_regex(is_long, [r])
        if not s.empty:
            return float(s.iloc[-1])
    return None

def _get_OperatingIncome():
    tk = yf.Ticker(symbol)
    is_long = _to_long_from_wide(tk.financials)
    for r in [r"^operating\s+income$", r"^ebit$"]:
        s = _series_by_regex(is_long, [r])
        if not s.empty:
            return float(s.iloc[-1])
    return None

def _get_InvestedCapital():
    tk = yf.Ticker(symbol)
    bs_long = _to_long_from_wide(tk.balance_sheet)
    s = _series_by_regex(bs_long, [r"^invested\s+capital$"])
    if not s.empty:
        return float(s.iloc[-1])
    return None

# ---------- Coleta de insumos ----------
PL = _get_PL_value()
LL = _get_LL_value()

Ke = next((float(globals()[nm]) for nm in ["Ke_capm", "Ke_value", "Ke"] if nm in globals() and globals()[nm] is not None), None)
WACC = next((float(globals()[nm]) for nm in ["WACC_value", "wacc", "WACC"] if nm in globals() and globals()[nm] is not None), None)

ROE_frac = metrics.get("ROE") if 'metrics' in globals() and isinstance(metrics, dict) else None
ROI_frac = metrics.get("ROI") if 'metrics' in globals() and isinstance(metrics, dict) else None

if ROE_frac is None and LL and PL:
    ROE_frac = LL / PL

if ROI_frac is None:
    opinc = _get_OperatingIncome()
    invcap = _get_InvestedCapital()
    if opinc and invcap:
        ROI_frac = opinc / invcap

# ---------- 1) Custo de Oportunidade ----------
print("📘 1️⃣ CUSTO DE OPORTUNIDADE (Co = PL × Ke)")
if PL is None or Ke is None:
    print("⚠️ Dados insuficientes para Co (precisa de PL e Ke).")
    Co = None
else:
    Co = PL * Ke
    print(f"PL:  {_fmt_money(PL)}")
    print(f"Ke:  {_fmt_pct_dec(Ke)}")
    print(f"Co:  {_fmt_money(Co)}")

# ---------- 2) EVA e Ganho em Excesso ----------
print("\n📘 2️⃣ LUCRO ECONÔMICO (EVA = LL − Co) e Ganho em Excesso")
if LL is None or Co is None:
    print("⚠️ Dados insuficientes para EVA (precisa de LL e Co).")
else:
    EVA = LL - Co
    GE  = LL - EVA
    print(f"LL:              {_fmt_money(LL)}")
    print(f"EVA (LL − Co):   {_fmt_money(EVA)}")
    print(f"Ganho em Excesso (LL − EVA): {_fmt_money(GE)}")
    if EVA > 0:
        print("✅ A empresa ESTÁ criando valor econômico.")
    elif EVA < 0:
        print("⚠️ A empresa ESTÁ destruindo valor econômico.")
    else:
        print("🟰 Neutro: LL cobre exatamente o custo do capital próprio.")

# ---------- 3) ROE > Ke ----------
print("\n📘 3️⃣ ROE vs Ke")
if ROE_frac is None or Ke is None:
    print("⚠️ Dados insuficientes para ROE vs Ke.")
else:
    diff = ROE_frac - Ke
    print(f"ROE:  {_fmt_pct_dec(ROE_frac)}")
    print(f"Ke:   {_fmt_pct_dec(Ke)}")
    print(f"Δ(ROE − Ke): {_fmt_pct_dec(diff)}")
    if diff > 0:
        print("✅ ROE > Ke → retorno ao acionista acima do custo de capital próprio.")
    elif diff < 0:
        print("⚠️ ROE < Ke → retorno abaixo do exigido.")
    else:
        print("🟰 ROE = Ke.")

# ---------- 4) ROI > WACC ----------
print("\n📘 4️⃣ ROI vs WACC")
if ROI_frac is None or WACC is None:
    print("⚠️ Dados insuficientes para ROI vs WACC.")
else:
    diff = ROI_frac - WACC
    print(f"ROI:  {_fmt_pct_dec(ROI_frac)}")
    print(f"WACC: {_fmt_pct_dec(WACC)}")
    print(f"Δ(ROI − WACC): {_fmt_pct_dec(diff)}")
    if diff > 0:
        print("✅ ROI > WACC → criação de valor sobre o capital total investido.")
    elif diff < 0:
        print("⚠️ ROI < WACC → destruição de valor.")
    else:
        print("🟰 ROI = WACC.")

# ---------- Resumo conceitual ----------
print("""
————————————————————————————————————————————————————————
🧭 GUIA RÁPIDO
• Co (PL×Ke): mínimo que o equity deveria render dado o risco.
• EVA (LL − Co): valor econômico criado após remunerar o equity.
• ROE vs Ke: eficiência do capital próprio (acima/abaixo do exigido).
• ROI vs WACC: criação de valor para TODOS os financiadores (equity + dívida).
————————————————————————————————————————————————————————
""")


📘 1️⃣ CUSTO DE OPORTUNIDADE (Co = PL × Ke)
PL:  $3,209,966,000.00
Ke:  14.27%
Co:  $458,205,707.52

📘 2️⃣ LUCRO ECONÔMICO (EVA = LL − Co) e Ganho em Excesso
LL:              $765,191,000.00
EVA (LL − Co):   $306,985,292.48
Ganho em Excesso (LL − EVA): $458,205,707.52
✅ A empresa ESTÁ criando valor econômico.

📘 3️⃣ ROE vs Ke
ROE:  28.22%
Ke:   14.27%
Δ(ROE − Ke): 13.95%
✅ ROE > Ke → retorno ao acionista acima do custo de capital próprio.

📘 4️⃣ ROI vs WACC
ROI:  25.28%
WACC: 14.16%
Δ(ROI − WACC): 11.12%
✅ ROI > WACC → criação de valor sobre o capital total investido.

————————————————————————————————————————————————————————
🧭 GUIA RÁPIDO
• Co (PL×Ke): mínimo que o equity deveria render dado o risco.
• EVA (LL − Co): valor econômico criado após remunerar o equity.
• ROE vs Ke: eficiência do capital próprio (acima/abaixo do exigido).
• ROI vs WACC: criação de valor para TODOS os financiadores (equity + dívida).
————————————————————————————————————————————————————————



# **Valuation**

In [48]:
# =========================================================
# CÉLULA — Preparação para Valuation (pipeline yfinance)
# Sai com: Shares, gLL_M, bNOPAT, gNOPAT, CP, FCFF
# =========================================================
import pandas as pd
import numpy as np
import yfinance as yf
import re
from datetime import datetime, timedelta

# ------------- Helpers de formatação -------------
def _fmt_pct(x):
    try: return f"{float(x)*100:,.2f}%"
    except: return "-"

def _fmt_money(x):
    try: return f"${float(x):,.2f}"
    except: return "-"

def _to_long_from_wide(w):
    if w is None or w.empty:
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period"] = pd.to_datetime(long["Period"]).dt.date.astype(str)
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    sel = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if sel.empty:
        return pd.Series(dtype=float)
    best = sel.groupby("LineItem").size().sort_values().index[-1]
    sub = sel[ sel["LineItem"]==best ].copy()
    try: sub = sub.sort_values(["Period_dt","Period"])
    except: pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best)

# ------------- 1) Shares Outstanding -------------
shares_out = None
try:
    tk = yf.Ticker(symbol)

    # 1a) série histórica de ações (preferível)
    try:
        sh = tk.get_shares_full(start="2000-01-01", end=datetime.today().strftime("%Y-%m-%d"))
        if sh is not None and not sh.empty:
            shares_out = int(sh["BasicShares"].dropna().iloc[-1])
    except Exception:
        pass

    # 1b) info rápido
    if shares_out is None:
        info = getattr(tk, "info", {}) or {}
        if "sharesOutstanding" in info and info["sharesOutstanding"]:
            shares_out = int(info["sharesOutstanding"])

    # 1c) fallback por balanço (menos preciso): Common Stock Equity / Book Value per Share
    if shares_out is None:
        bs_long = _to_long_from_wide(tk.balance_sheet)
        is_long = _to_long_from_wide(tk.financials)
        # Equity
        REG_EQ = [r"^common\s+stock\s+equity$", r"^total\s+(stockholders|shareholders)'\s? equity$", r"^total\s+equity$"]
        s_eq = pd.Series(dtype=float)
        for pat in REG_EQ:
            s_eq = _series_by_regex(bs_long, [pat])
            if not s_eq.empty: break
        # Book Value per Share (do fund_df/metrics, se houver)
        BVPS = None
        if 'metrics' in globals() and isinstance(metrics, dict):
            BVPS = metrics.get("BookValuePS", None)
        if (s_eq is not None and not s_eq.empty) and BVPS not in (None, 0):
            shares_out = int(round(float(s_eq.iloc[-1]) / float(BVPS)))

except Exception as e:
    shares_out = None

# ------------- 2) gLL ponderado ao mercado -------------
# gLL = (1 - payout) * ROE
payout = None
roe_frac = None
if 'metrics' in globals() and isinstance(metrics, dict):
    payout = metrics.get("Payout", None)           # fração (0..1)
    roe_frac = metrics.get("ROE", None)            # fração (0..1)

gLL = None
if payout is not None and roe_frac is not None:
    gLL = (1 - payout) * roe_frac

# retorno médio do S&P 500 (caso não exista mean_yearly_return já calculado)
mean_ret_dec = None
try:
    if 'mean_yearly_return' in globals() and mean_yearly_return is not None:
        # variável antiga vinha em %
        mean_ret_dec = float(mean_yearly_return) / 100.0
    else:
        sp = yf.download("^GSPC", period="10y", interval="1d", progress=False)
        px = sp["Adj Close"] if "Adj Close" in sp else sp["Close"]
        daily = px.pct_change()
        mean_ret_dec = float(daily.mean() * 252)  # aprox anual
except Exception:
    mean_ret_dec = None

gLL_M = None
if gLL is not None and mean_ret_dec is not None:
    gLL_M = 0.5 * (gLL + mean_ret_dec)

# ------------- 3) bNOPAT = (CAPEX − D&A) / EBIT -------------
# • CAPEX (cashflow: capitalExpenditures) é negativo → usamos valor absoluto para “investimento”
# • D&A (Depreciação e Amortização) (cashflow)
# • EBIT (DRE)
cf_wide = tk.cashflow
cf_long = _to_long_from_wide(cf_wide)
is_wide = tk.financials
is_long = _to_long_from_wide(is_wide)

# CAPEX
capex_last = None
try:
    if cf_wide is not None and not cf_wide.empty and "Capital Expenditures" in cf_wide.index:
        capex_last = float(cf_wide.loc["Capital Expenditures"].dropna().iloc[-1])
    else:
        s_cx = _series_by_regex(cf_long, [r"capital\s+expenditure"])
        if not s_cx.empty:
            capex_last = float(s_cx.iloc[-1])
except Exception:
    pass

# D&A
da_last = None
try:
    if cf_wide is not None and not cf_wide.empty:
        for cand in ["Depreciation & Amortization", "Depreciation", "Amortization"]:
            if cand in cf_wide.index and not cf_wide.loc[cand].dropna().empty:
                da_last = float(cf_wide.loc[cand].dropna().iloc[-1]); break
    if da_last is None:
        s_da = _series_by_regex(cf_long, [r"depreciation", r"amortization"])
        if not s_da.empty:
            da_last = float(s_da.iloc[-1])
except Exception:
    pass

# EBIT
ebit_last = None
try:
    if is_wide is not None and not is_wide.empty:
        for cand in ["EBIT", "Operating Income"]:
            if cand in is_wide.index and not is_wide.loc[cand].dropna().empty:
                ebit_last = float(is_wide.loc[cand].dropna().iloc[-1]); break
    if ebit_last is None:
        s_ebit = _series_by_regex(is_long, [r"^ebit$", r"operating\s+income"])
        if not s_ebit.empty:
            ebit_last = float(s_ebit.iloc[-1])
except Exception:
    pass

bNOPAT = None
CAPEX_abs = None
if capex_last is not None and da_last is not None and ebit_last not in (None, 0):
    CAPEX_abs = abs(capex_last)  # capex (uso de caixa) como positivo
    bNOPAT = (CAPEX_abs - da_last) / ebit_last

# ------------- 4) gNOPAT = bNOPAT × ROI -------------
ROI_frac = None
if 'metrics' in globals() and isinstance(metrics, dict):
    ROI_frac = metrics.get("ROI", None)
# fallback de ROI (se necessário) — Operating Income / Invested Capital
if ROI_frac is None:
    invcap = None
    try:
        bs_long = _to_long_from_wide(tk.balance_sheet)
        s_ic = _series_by_regex(bs_long, [r"^invested\s+capital$"])
        if not s_ic.empty:
            invcap = float(s_ic.iloc[-1])
    except Exception:
        pass
    if invcap not in (None, 0) and ebit_last is not None:
        ROI_frac = ebit_last / invcap

gNOPAT = None
if bNOPAT is not None and ROI_frac is not None:
    gNOPAT = bNOPAT * ROI_frac

# ------------- 5) Crescimento na perpetuidade (CP) -------------
# Mantendo sua premissa fixa de 2% (padrão conservador de longo prazo).
CP = 0.02

# ------------- 6) FCFF (usar já calculado; senão tentar aproximar) -------------
FCFF_val = None
if 'FCFF_value' in globals() and FCFF_value is not None:
    FCFF_val = float(FCFF_value)
else:
    # tentativa de reconstrução: FCFF ≈ NOPAT + D&A − ΔNWC − CAPEX
    # NOPAT restrito (usamos EBIT - TaxProvision) já foi calculado em células anteriores?
    NOPAT_Restrito = None
    if 'NOPAT_Restrito' in globals():
        NOPAT_Restrito = globals()['NOPAT_Restrito']
    # Se não, aproximar: NOPAT ≈ EBIT × (1 − TaxRate)
    TaxRate = None
    for nm in ["TaxRate", "tax_rate", "Tax_Provision_Rate"]:
        if nm in globals() and globals()[nm] is not None:
            TaxRate = float(globals()[nm]); break
    if TaxRate is None:
        TaxRate = 0.21

    if NOPAT_Restrito is None and ebit_last is not None:
        NOPAT_Restrito = ebit_last * (1 - TaxRate)

    # ΔNWC (variação do capital de giro operacional) se já calculado:
    Var_NC_Giro = None
    if 'Var_NC_Giro' in globals() and Var_NC_Giro is not None:
        Var_NC_Giro = float(Var_NC_Giro)

    if (NOPAT_Restrito is not None) and (da_last is not None) and (CAPEX_abs is not None):
        if Var_NC_Giro is None:
            # se não houver ΔNWC, aproximar sem ΔNWC
            FCFF_val = NOPAT_Restrito + da_last - CAPEX_abs
        else:
            FCFF_val = NOPAT_Restrito + da_last - Var_NC_Giro - CAPEX_abs

# ------------- Saída organizada -------------
rows = []

rows.append(["Nº de Ações (Shares Outstanding)",
             "-" if shares_out is None else f"{shares_out:,}",
             "Quantidade de ações em circulação (base para valor por ação)"])

rows.append(["gLL ponderado ao mercado (gLL_M)",
             "-" if gLL_M is None else _fmt_pct(gLL_M),
             "Média de (gLL) e do retorno médio do S&P 500 (10y)"])

rows.append(["bNOPAT = (CAPEX − D&A) / EBIT",
             "-" if bNOPAT is None else f"{bNOPAT:,.4f}",
             "Taxa de reinvestimento operacional baseada em investimentos líquidos"])

rows.append(["gNOPAT = bNOPAT × ROI",
             "-" if gNOPAT is None else _fmt_pct(gNOPAT),
             "Crescimento implícito do NOPAT via reinvestimento x retorno"])

rows.append(["Crescimento na Perpetuidade (CP)",
             _fmt_pct(CP),
             "Hipótese de longo prazo (conservadora, 2%)"])

rows.append(["FCFF (último ano)",
             "-" if FCFF_val is None else _fmt_money(FCFF_val),
             "Fluxo de caixa livre para a firma (base do DCF)"])

prep_df = pd.DataFrame(rows, columns=["Métrica", "Valor", "Notas"])

print(f"🧱 Preparação para Valuation — {symbol}")
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 80, 'display.width', 140):
    display(prep_df)

# Exporta para uso posterior
Shares_Outstanding = shares_out
gLL_M_value        = gLL_M
bNOPAT_value       = bNOPAT
gNOPAT_value       = gNOPAT
CP_value           = CP
FCFF_prepared      = FCFF_val


🧱 Preparação para Valuation — MLI


,Métrica,Valor,Notas
0,Nº de Ações (Shares Outstanding),"110,570,613",Quantidade de ações em circulação (base para valor por ação)
1,gLL ponderado ao mercado (gLL_M),18.79%,Média de (gLL) e do retorno médio do S&P 500 (10y)
2,bNOPAT = (CAPEX − D&A) / EBIT,0.0350,Taxa de reinvestimento operacional baseada em investimentos líquidos
3,gNOPAT = bNOPAT × ROI,0.88%,Crescimento implícito do NOPAT via reinvestimento x retorno
4,Crescimento na Perpetuidade (CP),2.00%,"Hipótese de longo prazo (conservadora, 2%)"
5,FCFF (último ano),"$662,056,390.00",Fluxo de caixa livre para a firma (base do DCF)


**BRIDGE — Normaliza/Reconstrói insumos para o DCF (VP)**

In [49]:
# =========================================================
# CÉLULA BRIDGE — Normaliza/Reconstrói insumos para o DCF (VP)
# Prepara: FCFF_prepared, WACC_value, Shares_Outstanding, gLL_M_value, gNOPAT_value, CP_value
# =========================================================
import math
import numpy as np
import pandas as pd
import yfinance as yf

def _safe_float(x, default=None):
    try:
        return float(x)
    except:
        return default

def _from_globals(*names, default=None, convert_to_float=True):
    """Pega a primeira variável existente em globals() dentre os nomes passados."""
    for n in names:
        if n in globals():
            v = globals()[n]
            if convert_to_float:
                vv = _safe_float(v, None)
                if vv is not None:
                    return vv
            else:
                if v is not None:
                    return v
    return default

# 1) SHARES
Shares_Outstanding = _from_globals("Shares_Outstanding", default=None)
if Shares_Outstanding in (None, 0):
    try:
        tk = yf.Ticker(symbol)
        info = getattr(tk, "info", {}) or {}
        Shares_Outstanding = _safe_float(info.get("sharesOutstanding"), None)
    except:
        Shares_Outstanding = None

# 2) FCFF (último ano)
# Tentativas: FCFF_prepared, FCFF (número), senão recompõe: FCO - CAPEX - ΔNCG
FCFF_prepared = _from_globals("FCFF_prepared", "FCFF_value", "FCFF", default=None)

if FCFF_prepared is None:
    # tentar recompor com FCO (NOPAT + D&A), capex_yf e Var_NC_Giro (Δ NCG)
    FCO_val = _from_globals("FCO_value", "FCO", default=None)       # já deve estar em $
    capex_yf = _from_globals("capex_yf", "CAPEX_value", default=None)
    dNCG    = _from_globals("Var_NC_Giro", "Var_NC_Giro_value", default=0.0)

    if FCO_val is not None and capex_yf is not None:
        # fórmula robusta: FCFF = FCO - CAPEX - ΔNCG  (ΔNCG negativo ajuda o FCFF)
        FCFF_prepared = FCO_val - capex_yf - dNCG

# 3) WACC_value (fração)
WACC_value = _from_globals("WACC_value", default=None)
if WACC_value is None:
    # tentar reconstruir a partir de Ke, Ki, pesos (PL_Po_PL, Po_Po_PL) e imposto Tc
    Ke_val = _from_globals("Ke", default=None)
    Ki_val = _from_globals("Ki_value", "Ki", default=None)
    # converter % -> fração se for > 1
    if Ke_val is not None and Ke_val > 1: Ke_val = Ke_val/100.0
    if Ki_val is not None and Ki_val > 1: Ki_val = Ki_val/100.0

    We = _from_globals("PL_Po_PL", default=None)  # peso do equity
    Wd = _from_globals("Po_Po_PL", default=None)  # peso da dívida

    # se pesos não existirem, tentar derivar de PL e Dívida
    if (We is None or Wd is None):
        PL_val = _from_globals("PL", "CommonEquity", default=None)
        total_debt = _from_globals("total_debt", "TotalDebt_value", default=None)
        if PL_val is not None and total_debt is not None and (PL_val + total_debt) != 0:
            We = PL_val / (PL_val + total_debt)
            Wd = total_debt / (PL_val + total_debt)

    # imposto efetivo (Tc) → se não soubermos, usa 21% padrão
    Tc = _from_globals("Tc_value", "TaxRate_value", default=0.21)

    # por padrão, usar (Ke * We) + (Ki * (1 − Tc) * Wd)
    if (Ke_val is not None) and (Ki_val is not None) and (We is not None) and (Wd is not None):
        WACC_value = Ke_val * We + Ki_val * (1.0 - Tc) * Wd

# 4) gLL_M_value (crescimento 1ª fase)
gLL_M_value = _from_globals("gLL_M_value", default=None)
if gLL_M_value is None:
    # fallback 5%
    gLL_M_value = 0.05

# 5) gNOPAT_value (se não houver, usar gLL_M_value como aproximação)
gNOPAT_value = _from_globals("gNOPAT_value", default=None)
if gNOPAT_value is None:
    gNOPAT_value = gLL_M_value

# 6) CP_value (perpétuo)
CP_value = _from_globals("CP_value", default=None)
if CP_value is None:
    CP_value = 0.02  # 2%

# 7) Por segurança: garante WACC > g∞
if WACC_value is not None and WACC_value <= CP_value:
    CP_value = max(0.0, WACC_value - 0.01)

# 8) Print de diagnóstico rápido
def _fmt_money(x):
    try:
        return f"${x:,.2f}"
    except:
        return str(x)

def _fmt_pct(x):
    try:
        return f"{x*100:,.2f}%"
    except:
        return str(x)

print("🧩 Bridge — Insumos normalizados para DCF:")
print(" • Shares_Outstanding:", f"{int(Shares_Outstanding):,}" if Shares_Outstanding not in (None,0) else None)
print(" • FCFF_prepared    :", _fmt_money(FCFF_prepared) if FCFF_prepared is not None else None)
print(" • WACC_value       :", _fmt_pct(WACC_value) if WACC_value is not None else None)
print(" • gLL_M_value      :", _fmt_pct(gLL_M_value))
print(" • gNOPAT_value     :", _fmt_pct(gNOPAT_value))
print(" • CP_value (g∞)    :", _fmt_pct(CP_value))

# Deixa as variáveis no escopo global (garantia)
globals().update({
    "Shares_Outstanding": Shares_Outstanding,
    "FCFF_prepared": FCFF_prepared,
    "WACC_value": WACC_value,
    "gLL_M_value": gLL_M_value,
    "gNOPAT_value": gNOPAT_value,
    "CP_value": CP_value
})


🧩 Bridge — Insumos normalizados para DCF:
 • Shares_Outstanding: 110,570,613
 • FCFF_prepared    : $662,056,390.00
 • WACC_value       : 14.16%
 • gLL_M_value      : 18.79%
 • gNOPAT_value     : 0.88%
 • CP_value (g∞)    : 2.00%


**Valuation Bancos**

In [50]:
# ============================================================
# CÉLULA — Valuation para BANCOS: Residual Income (RI) e DDM/Net Payout
# Posição recomendada: DEPOIS da Bridge/Normalizador e ANTES do comparativo final
# Saídas globais (usadas no comparativo):
#   FairPrice            -> Preferência automática para bancos (RI por padrão)
#   FairPrice_Model      -> "Residual Income (Banco)" ou "DDM/Net Payout (Banco)"
#   FairPrice_RI         -> Preço pelo Residual Income (se calculado)
#   FairPrice_DDM        -> Preço pelo DDM/Net Payout (se calculado)
# Para NÃO-financeiras, esta célula não muda seu DCF.
# ============================================================
import math
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta

def _safe_float(x, default=None):
    try:
        if x is None: return default
        return float(x)
    except:
        return default

def _fmt_money(x):
    try:
        return f"${x:,.2f}" if x is not None and np.isfinite(x) else "-"
    except:
        return "-"

def _fmt_pct(x):
    try:
        return f"{x*100:,.2f}%" if x is not None and np.isfinite(x) else "-"
    except:
        return "-"

def _last_nonnull(s):
    try:
        ss = s.dropna()
        return float(ss.iloc[-1]) if not ss.empty else None
    except:
        return None

def _avg_last_two(s):
    try:
        ss = s.dropna()
        if len(ss) == 0: return None
        if len(ss) == 1: return float(ss.iloc[-1])
        return float(ss.iloc[-2:]).mean()
    except:
        return None

def _is_financial(info):
    sec = (info or {}).get("sector","") or ""
    ind = (info or {}).get("industry","") or ""
    txt = (sec + " " + ind).lower()
    keys = ["financial", "bank", "capital markets", "asset management", "insurance", "thrifts", "mortgage"]
    return any(k in txt for k in keys)

def _to_long_from_wide(w):
    if w is None or (hasattr(w, "empty") and w.empty):
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

import re
def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    sel = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if sel.empty:
        return pd.Series(dtype=float)
    best = sel.groupby("LineItem").size().sort_values().index[-1]
    sub = sel[ sel["LineItem"]==best ].copy()
    try: sub = sub.sort_values(["Period_dt","Period"])
    except: pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best)

# --- Coletas base
tk = yf.Ticker(symbol.upper())
info = getattr(tk, "info", {}) or {}
is_fin = _is_financial(info)

# Se NÃO for financeiro, não alteramos o seu pipeline DCF
if not is_fin:
    print("Modo: NÃO-FINANCEIRO — mantém DCF/FCFF normalmente. (Nenhuma mudança aplicada nesta célula.)")
else:
    print("Modo: FINANCEIRO — ativando Residual Income (RI) e DDM/Net Payout.")

# Preço atual (melhor esforço)
current_price = None
try:
    h = tk.history(period="1d")
    if h is not None and not h.empty:
        current_price = float(h["Close"].iloc[-1])
except: pass
if current_price is None:
    current_price = _safe_float(info.get("currentPrice"))

# Shares
Shares_Outstanding = globals().get("Shares_Outstanding", _safe_float(info.get("sharesOutstanding")))
globals()["Shares_Outstanding"] = Shares_Outstanding

# ===== 1) Residual Income (RI) por ação =====
FairPrice_RI, FairPrice_DDM = None, None

# BVPS = Common Equity / Shares
bs = tk.balance_sheet if tk.balance_sheet is not None and not tk.balance_sheet.empty else None
CommonEquity, CommonEquity_avg = None, None
if bs is not None:
    for cand in ["Common Stock Equity", "Total Stockholder Equity", "Total Shareholder Equity", "Total Equity"]:
        if cand in bs.index:
            CommonEquity = _last_nonnull(bs.loc[cand])
            if CommonEquity is not None:
                CommonEquity_avg = _avg_last_two(bs.loc[cand])
                break

BVPS = None
if CommonEquity not in (None, 0) and Shares_Outstanding not in (None, 0):
    BVPS = CommonEquity / Shares_Outstanding

# Net Income e ROE (usa média de 2 anos no denominador para suavizar)
IS = tk.income_stmt if tk.income_stmt is not None and not tk.income_stmt.empty else None
NetIncome = None
if IS is not None and "Net Income" in IS.index:
    NetIncome = _last_nonnull(IS.loc["Net Income"])
if NetIncome is None:
    NetIncome = _safe_float(info.get("netIncomeToCommon"))

ROE = None
if NetIncome not in (None, 0) and CommonEquity_avg not in (None, 0):
    ROE = NetIncome / CommonEquity_avg

# Ke via CAPM (se já existir no global, usa; senão calcula)
Ke = globals().get("Ke", None)
if Ke is None:
    Beta = None
    # tentar vir do met_df (Finviz parse)
    try:
        if "met_df" in globals():
            row = met_df.loc[met_df["Metric"]=="Beta","Value"]
            if not row.empty:
                Beta = _safe_float(row.iloc[0])
    except:
        pass
    if Beta is None:
        Beta = _safe_float(info.get("beta"))
    # Rf via ^TNX
    Rf_val = globals().get("Rf_val", None)
    if Rf_val is None:
        try:
            tnx = yf.download("^TNX", period="5d", interval="1d", progress=False)
            if tnx is not None and not tnx.empty:
                Rf_val = float(tnx["Close"].iloc[-1]) / 100.0
        except:
            Rf_val = None
    # Prêmio de mercado ~ CAGR 10y S&P
    MarketReturn = globals().get("mean_yearly_return", None)
    if MarketReturn is None:
        try:
            end = datetime.today()
            start = end - timedelta(days=365*10+30)
            spx = yf.download("^GSPC", start=start.strftime("%Y-%m-%d"), end=end.strftime("%Y-%m-%d"), progress=False)
            if spx is not None and not spx.empty:
                p0 = float(spx["Adj Close"].iloc[0]); p1 = float(spx["Adj Close"].iloc[-1])
                years = max((spx.index[-1] - spx.index[0]).days / 365.25, 1)
                MarketReturn = (p1/p0)**(1/years) - 1
        except:
            MarketReturn = None
    if isinstance(MarketReturn,(int,float)) and MarketReturn > 1:
        MarketReturn = MarketReturn/100.0

    if (Beta is not None) and (Rf_val is not None) and (MarketReturn is not None):
        Ke = Rf_val + Beta * (MarketReturn - Rf_val)

# Fallbacks conservadores
if Ke is None: Ke = 0.11           # 11%
g_term = globals().get("CP_value", 0.02)  # 2% perpetuidade default
if Ke <= g_term:
    g_term = max(0.0, Ke - 0.01)   # garante Ke > g∞

# Compute RI price if FINANCIAL
if is_fin and (BVPS not in (None,0)) and (ROE is not None) and (Ke not in (None,0)) and (Ke > g_term):
    FairPrice_RI = BVPS + ((ROE - Ke)/(Ke - g_term)) * BVPS

# ===== 2) DDM / Net Payout por ação =====
# Payout por ação = Dividendos por ação + (Recompras líquidas por ação)
# Dividendos por ação (dividend TTM do Finviz parse, se existir)
div_ps = None
try:
    if "met_df" in globals():
        r = met_df.loc[met_df["Metric"]=="DividendTTM","Value"]
        if not r.empty and r.iloc[0] not in (None, "-", "nan"):
            div_ps = _safe_float(r.iloc[0])
except:
    pass
# Fallback: dividendYield * price (não é ideal), só se nada melhor
if div_ps is None:
    try:
        y = None
        if "met_df" in globals():
            r = met_df.loc[met_df["Metric"]=="DividendYield","Value"]
            if not r.empty and r.iloc[0] not in (None, "-", "nan"):
                y = _safe_float(r.iloc[0])  # fração
        if y is not None and current_price is not None:
            div_ps = y * current_price
    except:
        pass

# Recompras líquidas (cashflow): Repurchase Of Stock (outflow negativo), Sale Of Stock (inflow)
CF = tk.cashflow if tk.cashflow is not None and not tk.cashflow.empty else None
CF_l = _to_long_from_wide(CF)

def _get_last(cf_long, regex_list):
    s = _series_by_regex(cf_long, regex_list)
    return float(s.iloc[-1]) if not s.empty else None

repurchase = _get_last(CF_l, [r"^repurchase\s+of\s+stock$", r"^purchase\s+of\s+stock$", r"^treasury\s+stock\s+purchased$"])
sale_stock = _get_last(CF_l, [r"^sale\s+of\s+stock$", r"^issuance\s+of\s+stock$", r"^common\s+stock\s+issued$"])
dividends_paid_cash = None
for patt in [r"^common\s+stock\s+dividends\s+paid$", r"^cash\s+dividends\s+paid$"]:
    v = _get_last(CF_l, [patt])
    if v is not None:
        dividends_paid_cash = v
        break

# Normalização de sinais:
# - yfinance costuma trazer "Repurchase Of Stock" como valor NEGATIVO (saída de caixa)
# - "Sale of Stock" como POSITIVO (entrada de caixa)
# - "Dividends Paid" como NEGATIVO
net_buyback_cash = None
if (repurchase is not None) or (sale_stock is not None):
    rep = repurchase if repurchase is not None else 0.0
    sal = sale_stock if sale_stock is not None else 0.0
    # caixa gasto líquido com recompras (positivo = de fato recomprou)
    net_buyback_cash = max(0.0, -(rep) - max(0.0, sal))  # se houve emissão relevante, reduz recompras líquidas

# Payout em $ (caixa) = |dividends_paid_cash| + net_buyback_cash
payout_cash_total = None
if (dividends_paid_cash is not None) or (net_buyback_cash is not None):
    div_cash = abs(dividends_paid_cash) if dividends_paid_cash is not None else 0.0
    bb_cash  = net_buyback_cash if net_buyback_cash is not None else 0.0
    payout_cash_total = div_cash + bb_cash

# Payout por ação (preferência 1: dividend TTM do papel + recompras/ação; fallback: tudo por ação via caixa)
payout_ps = None
if (div_ps is not None) and (payout_cash_total is not None) and Shares_Outstanding not in (None,0):
    payout_ps = div_ps + (payout_cash_total / Shares_Outstanding)
elif (div_ps is not None):
    payout_ps = div_ps
elif (payout_cash_total is not None) and Shares_Outstanding not in (None,0):
    payout_ps = payout_cash_total / Shares_Outstanding

# Preço DDM (Gordon): P0 = payout_next / (Ke - g∞) ~ payout_ps * (1+g∞)/(Ke - g∞)
if is_fin and (payout_ps not in (None, 0)) and (Ke not in (None, 0)) and (Ke > g_term):
    FairPrice_DDM = payout_ps * (1.0 + g_term) / (Ke - g_term)

# ===== Escolha automática para bancos =====
if is_fin:
    # preferir RI; se não tiver, usar DDM; se tiver ambos, mostrar ambos e usar RI como "FairPrice"
    chosen = None
    if FairPrice_RI is not None:
        chosen = ("Residual Income (Banco)", FairPrice_RI)
    elif FairPrice_DDM is not None:
        chosen = ("DDM/Net Payout (Banco)", FairPrice_DDM)

    if chosen is not None:
        globals()["FairPrice_Model"] = chosen[0]
        globals()["FairPrice"]       = float(chosen[1])
        # Desarma DCF (evita misturar no comparativo se você quiser)
        globals()["FCFF_prepared"] = None
        globals()["WACC_value"]    = None

# ===== Diagnóstico =====
print("\n🧭 Diagnóstico (modo bancos)" if is_fin else "\n🧭 Diagnóstico (não-financeiro)")
print(" Setor:", info.get("sector","-"), "| Indústria:", info.get("industry","-"))
print(" Preço atual:", _fmt_money(current_price))
if is_fin:
    print("\n→ Residual Income")
    print(" BVPS:", _fmt_money(BVPS), "| ROE:", _fmt_pct(ROE), "| Ke:", _fmt_pct(Ke), "| g∞:", _fmt_pct(g_term))
    print(" FairPrice_RI:", _fmt_money(FairPrice_RI))
    print("\n→ DDM / Net Payout")
    print(" Dividendos/ação (TTM aprox.):", _fmt_money(div_ps))
    if payout_cash_total is not None:
        print(" Payout caixa total (Div + (Buyback − Emissão)):", _fmt_money(payout_cash_total))
    print(" Payout/ação (estimado):", _fmt_money(payout_ps))
    print(" FairPrice_DDM:", _fmt_money(FairPrice_DDM))
    if "FairPrice" in globals():
        print("\n✅ FairPrice escolhido:", globals().get("FairPrice_Model"), "→", _fmt_money(globals().get("FairPrice")))
    else:
        print("\n⚠️ Não foi possível computar um preço justo específico para bancos (faltam variáveis).")
else:
    print(" Nenhum override aplicado — seu DCF permanece como principal.")


Modo: NÃO-FINANCEIRO — mantém DCF/FCFF normalmente. (Nenhuma mudança aplicada nesta célula.)

🧭 Diagnóstico (não-financeiro)
 Setor: Industrials | Indústria: Metal Fabrication
 Preço atual: $139.51
 Nenhum override aplicado — seu DCF permanece como principal.


In [51]:
# ============================================================
# CÉLULA — Valuation para BANCOS: Residual Income (RI) e DDM/Net Payout
# Posição recomendada: DEPOIS da Bridge/Normalizador e ANTES do comparativo final
# Saídas globais (usadas no comparativo):
#   FairPrice            -> Preferência automática para bancos (RI por padrão)
#   FairPrice_Model      -> "Residual Income (Banco)" ou "DDM/Net Payout (Banco)"
#   FairPrice_RI         -> Preço pelo Residual Income (se calculado)
#   FairPrice_DDM        -> Preço pelo DDM/Net Payout (se calculado)
# Para NÃO-financeiras, esta célula não muda seu DCF.
# ============================================================
import math
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta

def _safe_float(x, default=None):
    try:
        if x is None: return default
        return float(x)
    except:
        return default

def _fmt_money(x):
    try:
        return f"${x:,.2f}" if x is not None and np.isfinite(x) else "-"
    except:
        return "-"

def _fmt_pct(x):
    try:
        return f"{x*100:,.2f}%" if x is not None and np.isfinite(x) else "-"
    except:
        return "-"

def _last_nonnull(s):
    try:
        ss = s.dropna()
        return float(ss.iloc[-1]) if not ss.empty else None
    except:
        return None

def _avg_last_two(s):
    try:
        ss = s.dropna()
        if len(ss) == 0: return None
        if len(ss) == 1: return float(ss.iloc[-1])
        return float(ss.iloc[-2:]).mean()
    except:
        return None

def _is_financial(info):
    sec = (info or {}).get("sector","") or ""
    ind = (info or {}).get("industry","") or ""
    txt = (sec + " " + ind).lower()
    keys = ["financial", "bank", "capital markets", "asset management", "insurance", "thrifts", "mortgage"]
    return any(k in txt for k in keys)

def _to_long_from_wide(w):
    if w is None or (hasattr(w, "empty") and w.empty):
        return None
    df = w.copy()
    df["LineItem"] = df.index.astype(str)
    val_cols = [c for c in df.columns if c != "LineItem"]
    long = (df.melt(id_vars=["LineItem"], value_vars=val_cols,
                    var_name="Period", value_name="Value")
              .dropna(subset=["Value"])
              .reset_index(drop=True))
    try:
        long["Period_dt"] = pd.to_datetime(long["Period"])
        long = long.sort_values(["Period_dt","Period"])
    except:
        pass
    long["norm"] = long["LineItem"].str.strip().str.lower()
    return long

import re
def _series_by_regex(long_df, include_regexes):
    if long_df is None or long_df.empty:
        return pd.Series(dtype=float)
    inc = [re.compile(p, re.I) for p in include_regexes]
    sel = long_df[ long_df["norm"].apply(lambda n: any(r.search(n) for r in inc)) ].copy()
    if sel.empty:
        return pd.Series(dtype=float)
    best = sel.groupby("LineItem").size().sort_values().index[-1]
    sub = sel[ sel["LineItem"]==best ].copy()
    try: sub = sub.sort_values(["Period_dt","Period"])
    except: pass
    return pd.Series(sub["Value"].astype(float).values,
                     index=sub["Period"].astype(str).values,
                     name=best)

# --- Coletas base
tk = yf.Ticker(symbol.upper())
info = getattr(tk, "info", {}) or {}
is_fin = _is_financial(info)

# Se NÃO for financeiro, não alteramos o seu pipeline DCF
if not is_fin:
    print("Modo: NÃO-FINANCEIRO — mantém DCF/FCFF normalmente. (Nenhuma mudança aplicada nesta célula.)")
else:
    print("Modo: FINANCEIRO — ativando Residual Income (RI) e DDM/Net Payout.")

# Preço atual (melhor esforço)
current_price = None
try:
    h = tk.history(period="1d")
    if h is not None and not h.empty:
        current_price = float(h["Close"].iloc[-1])
except: pass
if current_price is None:
    current_price = _safe_float(info.get("currentPrice"))

# Shares
Shares_Outstanding = globals().get("Shares_Outstanding", _safe_float(info.get("sharesOutstanding")))
globals()["Shares_Outstanding"] = Shares_Outstanding

# ===== 1) Residual Income (RI) por ação =====
FairPrice_RI, FairPrice_DDM = None, None

# BVPS = Common Equity / Shares
bs = tk.balance_sheet if tk.balance_sheet is not None and not tk.balance_sheet.empty else None
CommonEquity, CommonEquity_avg = None, None
if bs is not None:
    for cand in ["Common Stock Equity", "Total Stockholder Equity", "Total Shareholder Equity", "Total Equity"]:
        if cand in bs.index:
            CommonEquity = _last_nonnull(bs.loc[cand])
            if CommonEquity is not None:
                CommonEquity_avg = _avg_last_two(bs.loc[cand])
                break

BVPS = None
if CommonEquity not in (None, 0) and Shares_Outstanding not in (None, 0):
    BVPS = CommonEquity / Shares_Outstanding

# Net Income e ROE (usa média de 2 anos no denominador para suavizar)
IS = tk.income_stmt if tk.income_stmt is not None and not tk.income_stmt.empty else None
NetIncome = None
if IS is not None and "Net Income" in IS.index:
    NetIncome = _last_nonnull(IS.loc["Net Income"])
if NetIncome is None:
    NetIncome = _safe_float(info.get("netIncomeToCommon"))

ROE = None
if NetIncome not in (None, 0) and CommonEquity_avg not in (None, 0):
    ROE = NetIncome / CommonEquity_avg

# Ke via CAPM (se já existir no global, usa; senão calcula)
Ke = globals().get("Ke", None)
if Ke is None:
    Beta = None
    # tentar vir do met_df (Finviz parse)
    try:
        if "met_df" in globals():
            row = met_df.loc[met_df["Metric"]=="Beta","Value"]
            if not row.empty:
                Beta = _safe_float(row.iloc[0])
    except:
        pass
    if Beta is None:
        Beta = _safe_float(info.get("beta"))
    # Rf via ^TNX
    Rf_val = globals().get("Rf_val", None)
    if Rf_val is None:
        try:
            tnx = yf.download("^TNX", period="5d", interval="1d", progress=False)
            if tnx is not None and not tnx.empty:
                Rf_val = float(tnx["Close"].iloc[-1]) / 100.0
        except:
            Rf_val = None
    # Prêmio de mercado ~ CAGR 10y S&P
    MarketReturn = globals().get("mean_yearly_return", None)
    if MarketReturn is None:
        try:
            end = datetime.today()
            start = end - timedelta(days=365*10+30)
            spx = yf.download("^GSPC", start=start.strftime("%Y-%m-%d"), end=end.strftime("%Y-%m-%d"), progress=False)
            if spx is not None and not spx.empty:
                p0 = float(spx["Adj Close"].iloc[0]); p1 = float(spx["Adj Close"].iloc[-1])
                years = max((spx.index[-1] - spx.index[0]).days / 365.25, 1)
                MarketReturn = (p1/p0)**(1/years) - 1
        except:
            MarketReturn = None
    if isinstance(MarketReturn,(int,float)) and MarketReturn > 1:
        MarketReturn = MarketReturn/100.0

    if (Beta is not None) and (Rf_val is not None) and (MarketReturn is not None):
        Ke = Rf_val + Beta * (MarketReturn - Rf_val)

# Fallbacks conservadores
if Ke is None: Ke = 0.11           # 11%
g_term = globals().get("CP_value", 0.02)  # 2% perpetuidade default
if Ke <= g_term:
    g_term = max(0.0, Ke - 0.01)   # garante Ke > g∞

# Compute RI price if FINANCIAL
if is_fin and (BVPS not in (None,0)) and (ROE is not None) and (Ke not in (None,0)) and (Ke > g_term):
    FairPrice_RI = BVPS + ((ROE - Ke)/(Ke - g_term)) * BVPS

# ===== 2) DDM / Net Payout por ação =====
# Payout por ação = Dividendos por ação + (Recompras líquidas por ação)
# Dividendos por ação (dividend TTM do Finviz parse, se existir)
div_ps = None
try:
    if "met_df" in globals():
        r = met_df.loc[met_df["Metric"]=="DividendTTM","Value"]
        if not r.empty and r.iloc[0] not in (None, "-", "nan"):
            div_ps = _safe_float(r.iloc[0])
except:
    pass
# Fallback: dividendYield * price (não é ideal), só se nada melhor
if div_ps is None:
    try:
        y = None
        if "met_df" in globals():
            r = met_df.loc[met_df["Metric"]=="DividendYield","Value"]
            if not r.empty and r.iloc[0] not in (None, "-", "nan"):
                y = _safe_float(r.iloc[0])  # fração
        if y is not None and current_price is not None:
            div_ps = y * current_price
    except:
        pass

# Recompras líquidas (cashflow): Repurchase Of Stock (outflow negativo), Sale Of Stock (inflow)
CF = tk.cashflow if tk.cashflow is not None and not tk.cashflow.empty else None
CF_l = _to_long_from_wide(CF)

def _get_last(cf_long, regex_list):
    s = _series_by_regex(cf_long, regex_list)
    return float(s.iloc[-1]) if not s.empty else None

repurchase = _get_last(CF_l, [r"^repurchase\s+of\s+stock$", r"^purchase\s+of\s+stock$", r"^treasury\s+stock\s+purchased$"])
sale_stock = _get_last(CF_l, [r"^sale\s+of\s+stock$", r"^issuance\s+of\s+stock$", r"^common\s+stock\s+issued$"])
dividends_paid_cash = None
for patt in [r"^common\s+stock\s+dividends\s+paid$", r"^cash\s+dividends\s+paid$"]:
    v = _get_last(CF_l, [patt])
    if v is not None:
        dividends_paid_cash = v
        break

# Normalização de sinais:
# - yfinance costuma trazer "Repurchase Of Stock" como valor NEGATIVO (saída de caixa)
# - "Sale of Stock" como POSITIVO (entrada de caixa)
# - "Dividends Paid" como NEGATIVO
net_buyback_cash = None
if (repurchase is not None) or (sale_stock is not None):
    rep = repurchase if repurchase is not None else 0.0
    sal = sale_stock if sale_stock is not None else 0.0
    # caixa gasto líquido com recompras (positivo = de fato recomprou)
    net_buyback_cash = max(0.0, -(rep) - max(0.0, sal))  # se houve emissão relevante, reduz recompras líquidas

# Payout em $ (caixa) = |dividends_paid_cash| + net_buyback_cash
payout_cash_total = None
if (dividends_paid_cash is not None) or (net_buyback_cash is not None):
    div_cash = abs(dividends_paid_cash) if dividends_paid_cash is not None else 0.0
    bb_cash  = net_buyback_cash if net_buyback_cash is not None else 0.0
    payout_cash_total = div_cash + bb_cash

# Payout por ação (preferência 1: dividend TTM do papel + recompras/ação; fallback: tudo por ação via caixa)
payout_ps = None
if (div_ps is not None) and (payout_cash_total is not None) and Shares_Outstanding not in (None,0):
    payout_ps = div_ps + (payout_cash_total / Shares_Outstanding)
elif (div_ps is not None):
    payout_ps = div_ps
elif (payout_cash_total is not None) and Shares_Outstanding not in (None,0):
    payout_ps = payout_cash_total / Shares_Outstanding

# Preço DDM (Gordon): P0 = payout_next / (Ke - g∞) ~ payout_ps * (1+g∞)/(Ke - g∞)
if is_fin and (payout_ps not in (None, 0)) and (Ke not in (None, 0)) and (Ke > g_term):
    FairPrice_DDM = payout_ps * (1.0 + g_term) / (Ke - g_term)

# ===== Escolha automática para bancos =====
if is_fin:
    # preferir RI; se não tiver, usar DDM; se tiver ambos, mostrar ambos e usar RI como "FairPrice"
    chosen = None
    if FairPrice_RI is not None:
        chosen = ("Residual Income (Banco)", FairPrice_RI)
    elif FairPrice_DDM is not None:
        chosen = ("DDM/Net Payout (Banco)", FairPrice_DDM)

    if chosen is not None:
        globals()["FairPrice_Model"] = chosen[0]
        globals()["FairPrice"]       = float(chosen[1])
        # Desarma DCF (evita misturar no comparativo se você quiser)
        globals()["FCFF_prepared"] = None
        globals()["WACC_value"]    = None

# ===== Diagnóstico =====
print("\n🧭 Diagnóstico (modo bancos)" if is_fin else "\n🧭 Diagnóstico (não-financeiro)")
print(" Setor:", info.get("sector","-"), "| Indústria:", info.get("industry","-"))
print(" Preço atual:", _fmt_money(current_price))
if is_fin:
    print("\n→ Residual Income")
    print(" BVPS:", _fmt_money(BVPS), "| ROE:", _fmt_pct(ROE), "| Ke:", _fmt_pct(Ke), "| g∞:", _fmt_pct(g_term))
    print(" FairPrice_RI:", _fmt_money(FairPrice_RI))
    print("\n→ DDM / Net Payout")
    print(" Dividendos/ação (TTM aprox.):", _fmt_money(div_ps))
    if payout_cash_total is not None:
        print(" Payout caixa total (Div + (Buyback − Emissão)):", _fmt_money(payout_cash_total))
    print(" Payout/ação (estimado):", _fmt_money(payout_ps))
    print(" FairPrice_DDM:", _fmt_money(FairPrice_DDM))
    if "FairPrice" in globals():
        print("\n✅ FairPrice escolhido:", globals().get("FairPrice_Model"), "→", _fmt_money(globals().get("FairPrice")))
    else:
        print("\n⚠️ Não foi possível computar um preço justo específico para bancos (faltam variáveis).")
else:
    print(" Nenhum override aplicado — seu DCF permanece como principal.")


Modo: NÃO-FINANCEIRO — mantém DCF/FCFF normalmente. (Nenhuma mudança aplicada nesta célula.)

🧭 Diagnóstico (não-financeiro)
 Setor: Industrials | Indústria: Metal Fabrication
 Preço atual: $139.51
 Nenhum override aplicado — seu DCF permanece como principal.


**Valuation Tech**

In [52]:
# =========================================================
# CÉLULA — Valuation Growth-Tech (detecção + painel + Reverse-DCF)
# Colar após a célula da DRE e antes do DCF
# Requer: yfinance (já usado no seu pipeline)
# Usa globais se disponíveis: symbol, current_price, Shares_Outstanding, CommonEquity,
#                             Ke (ou WACC_value), Rf_val, etc.
# Saídas: GROWTH_TECH_MODE (bool), painel de métricas, fair price (growth-tech) opcional
# =========================================================
import math, numpy as np, pandas as pd, yfinance as yf
from datetime import datetime

# ---------- helpers básicos ----------
def _safe_float(x, default=None):
    try:
        if x is None: return default
        return float(x)
    except:
        return default

def _fmt_money(x):
    try:
        return "-" if x is None or (isinstance(x, float) and (math.isnan(x))) else f"${x:,.2f}"
    except:
        return "-"

def _fmt_pct(x):
    try:
        return "-" if x is None or (isinstance(x, float) and (math.isnan(x))) else f"{x*100:,.2f}%"
    except:
        return "-"

def _last_nonnull_row(df, row_name):
    if df is None or df.empty or row_name not in df.index:
        return None
    s = df.loc[row_name].dropna()
    return float(s.iloc[-1]) if not s.empty else None

def _yoy_growth(series_like):
    """Crescimento YoY com base em série trimestral. Considera último trimestre vs mesmo trimestre do ano anterior."""
    try:
        s = pd.Series(series_like).dropna()
        if len(s) < 5:
            return None
        # último valor (t), valor ~4 trimestres antes (t-4)
        return float(s.iloc[-1]/s.iloc[-5] - 1.0)
    except:
        return None

# ---------- pegar ticker e info ----------
_ticker = symbol.upper() if 'symbol' in globals() else input("Digite o ticker (ex: IONQ): ").strip().upper()
tk = yf.Ticker(_ticker)
info = getattr(tk, "info", {}) or {}

# Preço atual (usa global se existir)
px = None
if "current_price" in globals() and current_price is not None:
    px = _safe_float(current_price)
if px is None:
    try:
        h = tk.history(period="5d")
        if h is not None and not h.empty:
            px = float(h["Close"].iloc[-1])
    except:
        pass
if px is None:
    px = _safe_float(info.get("currentPrice"))

# Shares e PL (Common Equity)
shares = globals().get("Shares_Outstanding", None)
if shares in (None, 0):
    shares = _safe_float(info.get("sharesOutstanding"))

# Balanço / DRE / CF (amplo)
bs_w = tk.balance_sheet if tk.balance_sheet is not None and not tk.balance_sheet.empty else None
is_w = tk.income_stmt  if tk.income_stmt  is not None and not tk.income_stmt.empty  else None
cf_w = tk.cashflow     if tk.cashflow     is not None and not tk.cashflow.empty     else None

# Converte para "longo" simplificado (opcional)
def _row(df, name):
    return _last_nonnull_row(df, name)

# Common Equity (PL)
CommonEquity = globals().get("CommonEquity", None)
if CommonEquity is None and bs_w is not None:
    for cand in ["Common Stock Equity","Total Stockholder Equity","Total Shareholder Equity","Total Equity"]:
        CommonEquity = _row(bs_w, cand)
        if CommonEquity is not None: break

# Receita (último anual) e margem operacional
Revenue_last = _row(is_w, "Total Revenue") or _row(is_w, "Operating Revenue")
OperatingIncome_last = _row(is_w, "Operating Income")
OperMargin = None
if (Revenue_last not in (None,0)) and (OperatingIncome_last is not None):
    OperMargin = OperatingIncome_last / Revenue_last

# Receita trimestral (para YoY de crescimento)
try:
    q = tk.quarterly_income_stmt
    rev_q = q.loc["Total Revenue"].dropna() if (q is not None and not q.empty and "Total Revenue" in q.index) else None
except:
    rev_q = None
Rev_YoY = _yoy_growth(rev_q) if rev_q is not None else None

# D&A, Capex, CFO e FCF
DA_last = _row(cf_w, "Depreciation And Amortization") or _row(cf_w, "Depreciation")
Capex_last = _row(cf_w, "Capital Expenditure")
CFO_last   = _row(cf_w, "Operating Cash Flow")
FCF_last   = None
if CFO_last is not None and Capex_last is not None:
    FCF_last = CFO_last + Capex_last  # (capex costuma ser negativo na base)

# Caixa e dívida (para EV)
Cash = _row(bs_w, "Cash And Cash Equivalents") or _row(bs_w, "Cash Cash Equivalents And Short Term Investments")
TotalDebt = _row(bs_w, "Total Debt")

MarketCap = None
if shares not in (None,0) and px not in (None,0):
    MarketCap = shares * px
if MarketCap is None:
    MarketCap = _safe_float(info.get("marketCap"))

EnterpriseValue = None
if MarketCap is not None:
    EnterpriseValue = MarketCap + (TotalDebt or 0.0) - (Cash or 0.0)

# EV/Sales (ttm)
EV_Sales = None
if EnterpriseValue not in (None,0) and Revenue_last not in (None,0):
    EV_Sales = EnterpriseValue / Revenue_last

# Rule of 40 = growth YoY + margem operacional
Rule40 = None
if Rev_YoY is not None and OperMargin is not None:
    Rule40 = Rev_YoY + OperMargin

# Cash Runway (anos)
# burn = -FCF se FCF < 0; runway = Cash / |FCF|
CashRunway_years = None
if (Cash not in (None,0)) and (FCF_last is not None) and (FCF_last < 0):
    CashRunway_years = Cash / abs(FCF_last)

# ---------------- detecção Growth-Tech ----------------
# Critérios: setor/indústria tech OU (EPS<=0 ou FCF<0) → modo growth-tech
sector = (info.get("sector") or "").lower()
industry = (info.get("industry") or "").lower()
is_tech_sector = any(k in (sector + " " + industry) for k in ["technology","semiconductor","software","ai","artificial","cloud","comput"])
pre_lucro = None

# EPS global (se existir no pipeline)
eps_global = None
for key in ["trailingEps","forwardEps","epsTrailingTwelveMonths"]:
    if eps_global is None:
        eps_global = _safe_float(info.get(key))

if eps_global is not None and eps_global <= 0:
    pre_lucro = True
elif FCF_last is not None and FCF_last < 0:
    pre_lucro = True
else:
    pre_lucro = False

GROWTH_TECH_MODE = bool(is_tech_sector and pre_lucro)

# ---------------- Reverse-DCF implícito (focado em receita) ----------------
# Objetivo: descobrir o CAGR de receita implícito para justificar o preço (via EV)
# FCF_t = Sales_t * FCF_margin_target
# TV = Sales_10 * FCF_margin_target * (1+g_term) / (WACC - g_term)
# PV = sum( FCF_t / (1+WACC)^t ) + TV / (1+WACC)^10
# Resolvemos g para PV ≈ EV (quando EV disponível); se EV não existir, usa MarketCap.
FCF_margin_target = 0.18   # 18% default p/ empresas que amadurecem (ajuste se quiser)
g_term            = 0.03   # 3% perpetuidade
WACC              = None

# usa Ke do CAPM se existir; senão WACC_value; senão 11%
if "Ke" in globals() and Ke is not None:
    WACC = _safe_float(Ke, 0.11)
elif "WACC_value" in globals() and WACC_value is not None:
    WACC = _safe_float(WACC_value, 0.11)
else:
    WACC = 0.11

# garante WACC > g_term
if WACC <= g_term:
    g_term = max(0.0, WACC - 0.01)

EV_for_model = EnterpriseValue if EnterpriseValue is not None else MarketCap
ReverseDCF_CAGR_impl = None
FairPrice_GrowthTech = None

def _pv_from_growth(Sales0, g, fcfm, wacc, ginf):
    # projeta 10 anos
    pv = 0.0
    Sales_t = Sales0
    for t in range(1, 11):
        Sales_t = Sales_t * (1.0 + g)
        FCF_t = Sales_t * fcfm
        pv += FCF_t / ((1.0 + wacc) ** t)
    # terminal via perpetuidade
    TV = Sales_t * fcfm * (1.0 + ginf) / (wacc - ginf)
    pv += TV / ((1.0 + wacc) ** 10)
    return pv

if GROWTH_TECH_MODE and EV_for_model not in (None,0) and Revenue_last not in (None,0):
    # busca g que zere f(g) = PV(g) - EV
    def f(g):
        return _pv_from_growth(Revenue_last, g, FCF_margin_target, WACC, g_term) - EV_for_model

    # bisseção entre -10% e 100% a.a.
    lo, hi = -0.10, 1.00
    flo, fhi = f(lo), f(hi)
    if flo * fhi > 0:
        # se não cruza, tenta ampliar hi
        for hi_try in [1.5, 2.0]:
            fhi = f(hi_try)
            if flo * fhi <= 0:
                hi = hi_try
                break
    # só resolve se houver mudança de sinal
    if flo * fhi <= 0:
        for _ in range(60):
            mid = 0.5*(lo+hi)
            fm = f(mid)
            if abs(fm) < 1e-6:
                break
            if flo * fm <= 0:
                hi, fhi = mid, fm
            else:
                lo, flo = mid, fm
        ReverseDCF_CAGR_impl = 0.5*(lo+hi)

    # também produzimos um “FairPrice” alternativo, caso queira comparar:
    if shares not in (None,0) and EV_for_model is not None:
        # usa g = min(Reverse found, 40%) como base para “bull conservador”
        g_used = ReverseDCF_CAGR_impl if ReverseDCF_CAGR_impl is not None else 0.30
        g_used = min(max(g_used, 0.0), 0.40)
        pv_ev = _pv_from_growth(Revenue_last, g_used, FCF_margin_target, WACC, g_term)
        # valor do equity ≈ EV - dívida + caixa
        equity_val = pv_ev - (TotalDebt or 0.0) + (Cash or 0.0)
        FairPrice_GrowthTech = equity_val / shares if equity_val is not None else None

# ---------------- Painel/saídas ----------------
rows = []
rows.append(["Ticker", _ticker])
rows.append(["Setor / Indústria", f"{info.get('sector','-')} / {info.get('industry','-')}"])
rows.append(["Preço atual", _fmt_money(px)])
rows.append(["Market Cap", _fmt_money(MarketCap)])
rows.append(["Enterprise Value", _fmt_money(EnterpriseValue)])
rows.append(["Receita (últ. anual)", _fmt_money(Revenue_last)])
rows.append(["Margem Operacional", _fmt_pct(OperMargin)])
rows.append(["Crescimento Receita YoY (q/q-4)", _fmt_pct(Rev_YoY)])
rows.append(["EV / Sales (ttm)", f"{EV_Sales:,.2f}x" if EV_Sales is not None else "-"])
rows.append(["Rule of 40 (YoY + OpMargin)", _fmt_pct(Rule40)])
rows.append(["Free Cash Flow (últ. anual)", _fmt_money(FCF_last)])
rows.append(["Caixa", _fmt_money(Cash)])
rows.append(["Dívida Total", _fmt_money(TotalDebt)])
rows.append(["Cash Runway (anos)", f"{CashRunway_years:,.2f}" if CashRunway_years is not None else "-"])
rows.append(["WACC usado", _fmt_pct(WACC)])
rows.append(["Margem FCF alvo (steady)", _fmt_pct(FCF_margin_target)])
rows.append(["g∞ (perpetuidade)", _fmt_pct(g_term)])
rows.append(["Reverse-DCF CAGR implícito", _fmt_pct(ReverseDCF_CAGR_impl) if ReverseDCF_CAGR_impl is not None else "-"])
rows.append(["Preço justo (Growth-Tech, opcional)", _fmt_money(FairPrice_GrowthTech)])

print("🧪 Detecção de Modo Growth-Tech:", "SIM" if GROWTH_TECH_MODE else "NÃO")
display(pd.DataFrame(rows, columns=["Item","Valor"]))

# ---------- Nota/Leitura automática ----------
insights = []
if GROWTH_TECH_MODE:
    if Rule40 is not None:
        if Rule40 >= 0.40:
            insights.append("✅ Rule of 40 ≥ 40% — crescimento de qualidade.")
        elif Rule40 < 0.20:
            insights.append("⚠️ Rule of 40 < 20% — crescimento pouco eficiente.")
    if CashRunway_years is not None:
        if CashRunway_years >= 3.0:
            insights.append("✅ Runway ≥ 3 anos — conforto de caixa.")
        elif CashRunway_years < 1.5:
            insights.append("⚠️ Runway < 1,5 ano — risco de diluição/financiamento.")
    if EV_Sales is not None and EV_Sales > 30:
        insights.append("⚠️ EV/Sales > 30× — preço embute expectativas extremas.")
    if ReverseDCF_CAGR_impl is not None:
        if ReverseDCF_CAGR_impl > 0.60:
            insights.append("⚠️ Mercado embute CAGR > 60% por 10 anos — barra muito alta.")
        elif ReverseDCF_CAGR_impl < 0.25:
            insights.append("✅ Crescimento implícito moderado — potencial assimetria se tese entregar.")
else:
    insights.append("ℹ️ Empresa não identificada como Growth-Tech (pré-lucro/FCF<0 em tech). Use DCF/EVA/RI conforme seu pipeline.")

print("🧠 Leituras automáticas:")
if insights:
    for s in insights:
        print("• " + s)
else:
    print("• Sem sinais fortes nesta etapa.")

# ---------- Exporta flag global ----------
GROWTH_TECH_MODE = bool(GROWTH_TECH_MODE)
FairPrice_GrowthTech_value = FairPrice_GrowthTech  # opcional, para usar adiante no comparativo

# DICA: Na sua célula do DCF, você pode fazer:
# if GROWTH_TECH_MODE: SKIP (ou apenas comparar DCF legacy) e priorizar painel Growth-Tech + Reverse-DCF


🧪 Detecção de Modo Growth-Tech: NÃO


,Item,Valor
0,Ticker,MLI
1,Setor / Indústria,Industrials / Metal F...
2,Preço atual,$139.51
3,Market Cap,"$15,425,705,612.25"
4,Enterprise Value,"$14,988,538,612.25"
5,Receita (últ. anual),"$3,982,455,000.00"
6,Margem Operacional,21.83%
7,Crescimento Receita Y...,-16.16%
8,EV / Sales (ttm),3.76x
9,Rule of 40 (YoY + OpM...,5.67%


🧠 Leituras automáticas:
• ℹ️ Empresa não identificada como Growth-Tech (pré-lucro/FCF<0 em tech). Use DCF/EVA/RI conforme seu pipeline.


**VP**

In [53]:
# =========================================================
# CÉLULA — DCF (10 anos + perpetuidade) com integração Growth-Tech
# Requer (se disponíveis): FCFF_prepared, gLL_M_value, CP_value (g∞),
#                          WACC_value (ou Ke), Shares_Outstanding, TotalDebt, Cash,
#                          FairPrice_GrowthTech_value, GROWTH_TECH_MODE
# Produz: VP_total, EnterpriseValue_DCF, EquityValue_DCF, FairPrice
# =========================================================
import math, numpy as np, pandas as pd

def _safe_float(x, default=None):
    try:
        return float(x)
    except:
        return default

def _fmt_money(x):
    try:
        return "-" if x is None or (isinstance(x,float) and (math.isnan(x))) else f"${x:,.2f}"
    except:
        return "-"

def _fmt_pct(x):
    try:
        return "-" if x is None or (isinstance(x,float) and (math.isnan(x))) else f"{x*100:,.2f}%"
    except:
        return "-"

# ---------------- 1) Checagem de modo Growth-Tech ----------------
GROWTH_TECH_MODE = bool(globals().get("GROWTH_TECH_MODE", False))
if GROWTH_TECH_MODE:
    # Se já calculamos o FairPrice_GrowthTech na célula anterior, só propagamos
    FairPrice_GrowthTech_value = globals().get("FairPrice_GrowthTech_value", None)
    print("⏭️ Modo Growth-Tech ATIVO — DCF clássico não é o método principal.")
    if FairPrice_GrowthTech_value is not None:
        FairPrice = float(FairPrice_GrowthTech_value)
        print("✅ Usando 'Preço justo (Growth-Tech)' calculado na célula anterior.")
        print("FairPrice (Growth-Tech):", _fmt_money(FairPrice))
    else:
        print("⚠️ FairPrice_GrowthTech_value ausente. Você pode usar apenas o painel Growth-Tech/Reverse-DCF.")
        FairPrice = None
    # Zera saídas de DCF para não conflitar com comparativos
    VP_total = None
    EnterpriseValue_DCF = None
    EquityValue_DCF = None
    # Se desejar ainda assim rodar um DCF por conferência, troque o return por um bloco opcional.
else:
    # ---------------- 2) Inputs base do DCF ----------------
    FCFF0 = globals().get("FCFF_prepared", None)
    if FCFF0 is None:
        # fallback: tenta FCFF direto (caso você tenha outro nome)
        FCFF0 = globals().get("FCFF", None)

    g_years = _safe_float(globals().get("gLL_M_value", None), default=0.05)  # crescimento de 1–5 anos (ou 10y se você usa assim)
    g_term  = _safe_float(globals().get("CP_value", None), default=0.02)     # crescimento em perpetuidade
    WACC    = _safe_float(globals().get("WACC_value", None), default=None)

    # Se WACC não existir, tenta Ke (CAPM); se >1, assume que veio em % e normaliza
    if WACC is None:
        Ke_try = globals().get("Ke", None)
        WACC = _safe_float(Ke_try, default=0.11)
        if WACC is not None and WACC > 1.0:
            WACC = WACC / 100.0
    # Se ainda None, fallback conservador
    if WACC is None:
        WACC = 0.11

    # Clamps e consistência
    g_years = max(min(g_years, 0.30), -0.10)  # limite prático
    g_term  = max(min(g_term, 0.04), -0.01)   # limite prático
    if WACC <= g_term:
        g_term = max(0.0, WACC - 0.01)

    # Necessários para equity bridge
    shares = _safe_float(globals().get("Shares_Outstanding", None), default=None)
    cash   = _safe_float(globals().get("Cash", None), default=0.0)
    debt   = _safe_float(globals().get("TotalDebt", None), default=0.0)

    # ---------------- 3) Validações mínimas ----------------
    if FCFF0 is None:
        raise ValueError("⚠️ FCFF_prepared ausente. Rode a célula que calcula o FCFF antes do DCF.")
    if FCFF0 == 0:
        print("ℹ️ FCFF0 = 0. O DCF ficará ~0. Verifique FCO/Capex/ΔNCG.")
    if shares in (None, 0):
        print("⚠️ Shares_Outstanding ausente. O FairPrice por ação será omitido.")

    # ---------------- 4) Projeções 10 anos + perpetuidade ----------------
    horizon = 10
    fcff = []
    f = FCFF0
    for t in range(1, horizon+1):
        f = f * (1.0 + g_years)
        fcff.append(f)

    # PV dos 10 anos
    disc = [(1.0 / ((1.0 + WACC) ** t)) for t in range(1, horizon+1)]
    pv_10y = sum([fcff[t-1] * disc[t-1] for t in range(1, horizon+1)])

    # Terminal value e PV do terminal
    TV = fcff[-1] * (1.0 + g_term) / (WACC - g_term)
    PV_TV = TV / ((1.0 + WACC) ** horizon)

    VP_total = pv_10y + PV_TV  # Valor presente dos FCFFs (Enterprise Value DCF)
    EnterpriseValue_DCF = VP_total

    # Passa para equity: EV − dívida + caixa
    EquityValue_DCF = EnterpriseValue_DCF - debt + cash

    FairPrice = None
    if shares not in (None, 0):
        FairPrice = EquityValue_DCF / shares

    # ---------------- 5) Saída amigável ----------------
    print("🏛️ DCF (10y + g∞) — resultado")
    out = [
        ["FCFF0 (base)",              _fmt_money(FCFF0)],
        ["g (anos projetados)",       _fmt_pct(g_years)],
        ["g∞ (perpetuidade)",         _fmt_pct(g_term)],
        ["WACC",                      _fmt_pct(WACC)],
        ["PV (10 anos)",              _fmt_money(pv_10y)],
        ["PV (Terminal)",             _fmt_money(PV_TV)],
        ["Enterprise Value (DCF)",    _fmt_money(EnterpriseValue_DCF)],
        ["(-) Dívida + (+) Caixa",    f"-{_fmt_money(debt)}  |  +{_fmt_money(cash)}"],
        ["Equity Value (DCF)",        _fmt_money(EquityValue_DCF)],
        ["FairPrice (por ação)",      _fmt_money(FairPrice) if FairPrice is not None else "— (defina Shares_Outstanding)"],
    ]
    display(pd.DataFrame(out, columns=["Item","Valor"]))

# --------------- 6) Exporta FairPrice p/ comparativos ---------------
# (Se Growth-Tech: FairPrice é Growth-Tech; se não: é o DCF)
globals()["FairPrice"] = FairPrice


🏛️ DCF (10y + g∞) — resultado


,Item,Valor
0,FCFF0 (base),"$662,056,390.00"
1,g (anos projetados),18.79%
2,g∞ (perpetuidade),2.00%
3,WACC,14.16%
4,PV (10 anos),"$8,293,241,330.38"
5,PV (Terminal),"$8,268,167,730.04"
6,Enterprise Value (DCF),"$16,561,409,060.42"
7,(-) Dívida + (+) Caixa,"-$23,851,000.00 | +..."
8,Equity Value (DCF),"$16,998,576,060.42"
9,FairPrice (por ação),$153.74


**Preço Justo da Ação**

In [54]:
# =========================================================
# PREÇO JUSTO — Consolidação (DCF principal + Checagens de Balanço)
# =========================================================
import yfinance as yf
import pandas as pd
import numpy as np

def _fmt_money(x):
    return f"${x:,.2f}" if x is not None and not np.isnan(x) else "-"

def _safe_last(series_or_row):
    try:
        s = series_or_row.dropna()
        return float(s.iloc[-1]) if not s.empty else None
    except:
        return None

tk = yf.Ticker(symbol)

# ---------- 1) Dados de balanço ----------
bs = tk.balance_sheet if tk.balance_sheet is not None and not tk.balance_sheet.empty else None

TotalAssets = None
TotalLiabilities = None
CommonEquity = None
CurrentAssets = None

if bs is not None:
    # Total Assets
    for cand in ["Total Assets"]:
        if cand in bs.index:
            TotalAssets = _safe_last(bs.loc[cand])
            break
    # Total Liabilities (preferir “Total Liabilities Net Minority Interest” se existir; caso contrário “Total Liabilities”)
    for cand in ["Total Liabilities Net Minority Interest", "Total Liabilities"]:
        if cand in bs.index:
            TotalLiabilities = _safe_last(bs.loc[cand])
            if TotalLiabilities is not None:
                break
    # Current Assets
    for cand in ["Current Assets", "Total Current Assets"]:
        if cand in bs.index:
            CurrentAssets = _safe_last(bs.loc[cand])
            if CurrentAssets is not None:
                break
    # Equity contábil
    for cand in ["Common Stock Equity", "Total Stockholder Equity", "Total Shareholder Equity", "Total Equity"]:
        if cand in bs.index:
            CommonEquity = _safe_last(bs.loc[cand])
            if CommonEquity is not None:
                break

# ---------- 2) Shares Outstanding ----------
shares = None
try:
    sh = tk.get_shares_full(start="2000-01-01")
    if sh is not None and not sh.empty and "BasicShares" in sh:
        shares = float(sh["BasicShares"].dropna().iloc[-1])
except:
    pass
if shares is None:
    info = getattr(tk, "info", {}) or {}
    shares = float(info.get("sharesOutstanding")) if info.get("sharesOutstanding") else None

# ---------- 3) Preço atual ----------
current_price = None
try:
    price_df = tk.history(period="1d")
    if price_df is not None and not price_df.empty:
        current_price = float(price_df["Close"].iloc[-1])
    elif "currentPrice" in info:
        current_price = float(info["currentPrice"])
except:
    pass

# ---------- 4) Preço justo DCF (principal) ----------
# Já calculado na célula de DCF como FairPrice. Se não houver, fica None.
PJA_DCF = None
if "FairPrice" in globals() and FairPrice is not None:
    PJA_DCF = float(FairPrice)

# ---------- 5) Checagens baseadas em balanço ----------
BVPS = (CommonEquity / shares) if (CommonEquity is not None and shares not in (None, 0)) else None
NCAV_ps = None
if (CurrentAssets is not None) and (TotalLiabilities is not None) and (shares not in (None, 0)):
    NCAV_ps = (CurrentAssets - TotalLiabilities) / shares

# ---------- 6) Tabela de saída ----------
rows = []
rows.append(["Preço atual", _fmt_money(current_price), "Cotação de mercado"])
rows.append(["Preço justo (DCF)", _fmt_money(PJA_DCF), "Nosso valor principal (FCFF + WACC + g∞)"])
rows.append(["BVPS (Equity/Share)", _fmt_money(BVPS), "Preço contábil por ação (piso de balanço)"])
rows.append(["NCAV por ação", _fmt_money(NCAV_ps), "Ativo Circulante − Passivo Total (Graham-like) por ação"])

df = pd.DataFrame(rows, columns=["Métrica", "Valor", "Notas"])
print(f"🏷️ Preço Justo — {symbol}")
display(df)

# ---------- 7) Margens vs preço atual ----------
def margin(px_model, px):
    if px_model in (None, np.nan) or px in (None, np.nan) or px_model == 0:
        return (None, None)
    diff = px_model - px
    pct  = diff / px
    return diff, pct

margins = []
for name, pxm in [("DCF", PJA_DCF), ("BVPS", BVPS), ("NCAV", NCAV_ps)]:
    diff, pct = margin(pxm, current_price)
    margins.append([f"Margem vs Preço — {name}", _fmt_money(diff), f"{pct*100:,.2f}%" if pct is not None else "-"])

df_m = pd.DataFrame(margins, columns=["Comparação", "Margem ($)", "Margem (%)"])
display(df_m)


🏷️ Preço Justo — MLI


,Métrica,Valor,Notas
0,Preço atual,$139.51,Cotação de mercado
1,Preço justo (DCF),$153.74,Nosso valor principal...
2,BVPS (Equity/Share),$16.20,Preço contábil por aç...
3,NCAV por ação,$10.00,Ativo Circulante − Pa...


,Comparação,Margem ($),Margem (%)
0,Margem vs Preço — DCF,$14.23,10.20%
1,Margem vs Preço — BVPS,$-123.31,-88.39%
2,Margem vs Preço — NCAV,$-129.51,-92.83%


**Score Composto de Valor**

In [55]:
# =========================================================
# CÉLULA — SCORE COMPOSTO v3.5 (Integrado + Inteligente)
# - Usa os resultados de valuation (DCF/RI/EVA/Growth-Tech)
# - Atribui pesos dinâmicos conforme o tipo da empresa
# - Gera nota 0..1 e parecer qualitativo (Forte / Neutro / Fraco)
# =========================================================
import numpy as np, pandas as pd, math

# ---------------- Helpers ----------------
def _safe(x, default=np.nan):
    try:
        if x is None or (isinstance(x, float) and not np.isfinite(x)):
            return default
        return float(x)
    except:
        return default

def _fmt_money(x):
    try:
        if x is None or np.isnan(x): return "-"
        return f"${x:,.2f}"
    except: return "-"

def _fmt_pct(x):
    try:
        if x is None or np.isnan(x): return "-"
        return f"{x*100:,.2f}%"
    except: return "-"

def _norm(x, low, high):
    if x is None or np.isnan(x): return 0.0
    return np.clip((x - low) / (high - low), 0, 1)

# ---------------- Inputs globais ----------------
symbol        = globals().get("symbol", "?")
current_price = _safe(globals().get("current_price", np.nan))

FairPrice_DCF = _safe(globals().get("FairPrice", np.nan))
FairPrice_GT  = _safe(globals().get("FairPrice_GrowthTech", np.nan))
PJA_EVA       = _safe(globals().get("PJA_Excesso_EVA", np.nan))
valor_intrinseco = _safe(globals().get("valor_intrinseco", np.nan))
BVPS          = _safe(globals().get("VPA", np.nan))
NCAV          = _safe(globals().get("NCAV_per_share", np.nan))
Sector        = str(globals().get("info", {}).get("sector", "")).lower()

# ---------------- Lógica adaptativa ----------------
# Define modo (Growth-Tech / Financeiro / Tradicional)
if "tech" in Sector or "semiconductor" in Sector or "software" in Sector or "ai" in Sector:
    mode = "Growth-Tech"
elif "financial" in Sector or "bank" in Sector or "insurance" in Sector:
    mode = "Financeiro"
else:
    mode = "Tradicional"

# ---------------- Métricas comparativas ----------------
# Upside (principal base de pontuação)
def _upside(fair, px):
    if fair is None or px in (None,0) or np.isnan(fair): return np.nan
    return (fair/px) - 1.0

up_DCF = _upside(FairPrice_DCF, current_price)
up_GT  = _upside(FairPrice_GT, current_price)
up_EVA = _upside(PJA_EVA, current_price)
up_GRA = _upside(valor_intrinseco, current_price)

# Cheapness: BVPS e NCAV
cheap_pb  = (BVPS/current_price - 1) if BVPS not in (None,0,np.nan) else np.nan
cheap_ncav = (NCAV/current_price - 1) if NCAV not in (None,0,np.nan) else np.nan

# ---------------- Sistema de pesos ----------------
if mode == "Growth-Tech":
    weights = dict(upGT=0.50, upDCF=0.15, upEVA=0.10, upGRA=0.05, pb=0.10, ncav=0.10)
elif mode == "Financeiro":
    weights = dict(upDCF=0.50, upEVA=0.15, upGRA=0.15, pb=0.15, ncav=0.05, upGT=0.00)
else:  # Tradicional
    weights = dict(upDCF=0.45, upEVA=0.20, upGRA=0.15, pb=0.10, ncav=0.10, upGT=0.00)

# ---------------- Normalização dos indicadores ----------------
n_upDCF = _norm(up_DCF, -0.5, 1.0)
n_upGT  = _norm(up_GT,  -0.5, 1.0)
n_upEVA = _norm(up_EVA, -0.5, 1.0)
n_upGRA = _norm(up_GRA, -0.5, 1.0)
n_pb    = _norm(cheap_pb, -0.8, 0.5)
n_ncav  = _norm(cheap_ncav, -0.8, 0.5)

# Score final ponderado
score = (
    n_upDCF * weights.get("upDCF",0) +
    n_upGT  * weights.get("upGT",0) +
    n_upEVA * weights.get("upEVA",0) +
    n_upGRA * weights.get("upGRA",0) +
    n_pb    * weights.get("pb",0) +
    n_ncav  * weights.get("ncav",0)
) / sum(weights.values())

# Qualificação textual
if score >= 0.70:
    label = "🟢 Forte (bom potencial)"
elif score >= 0.40:
    label = "🟡 Neutro (equilibrado)"
else:
    label = "🔴 Fraco (cuidado)"

# ---------------- Saída visual ----------------
print(f"🧮 Score Composto v3.5 — {symbol} ({mode})")
table = pd.DataFrame([
    ["Preço atual", _fmt_money(current_price), "-", "-"],
    ["Preço justo (DCF)", _fmt_money(FairPrice_DCF), _fmt_pct(up_DCF), f"{n_upDCF:.3f}"],
    ["Preço justo (Growth-Tech)", _fmt_money(FairPrice_GT), _fmt_pct(up_GT), f"{n_upGT:.3f}"],
    ["Preço justo (EVA)", _fmt_money(PJA_EVA), _fmt_pct(up_EVA), f"{n_upEVA:.3f}"],
    ["Preço justo (Graham)", _fmt_money(valor_intrinseco), _fmt_pct(up_GRA), f"{n_upGRA:.3f}"],
    ["Cheap P/B", _fmt_pct(cheap_pb), "-", f"{n_pb:.3f}"],
    ["Deep NCAV", _fmt_pct(cheap_ncav), "-", f"{n_ncav:.3f}"],
    ["", "", "", ""],
    ["SCORE FINAL (0..1)", f"{score:.3f}", "", label],
], columns=["Métrica","Valor/Resultado","Upside","Normalizado"])
display(table)

# Expor variáveis globais
globals().update(dict(Score_Final=score, Score_Label=label, Score_Mode=mode))



🧮 Score Composto v3.5 — MLI (Tradicional)


,Métrica,Valor/Resultado,Upside,Normalizado
0,Preço atual,$139.51,-,-
1,Preço justo (DCF),$153.74,10.20%,0.401
2,Preço justo (Growth-T...,-,-,0.000
3,Preço justo (EVA),-,-,0.000
4,Preço justo (Graham),-,-,0.000
5,Cheap P/B,-,-,0.000
6,Deep NCAV,-,-,0.000
7,,,,
8,SCORE FINAL (0..1),0.181,,🔴 Fraco (cuidado)


# **Comparativo de Avaliações**

In [56]:
# =========================================================
# CÉLULA — COMPARATIVO FINAL v3.3
# Integra: DCF/EVA/Graham/RI (bancos) + Growth-Tech (novo)
# Usa variáveis globais se existirem e não quebra se algo faltar.
# Saídas:
#   - df_comp_fmt (tabela comparativa formatada)
#   - df_rank_fmt (ranking de upside)
#   - best_model, best_value, best_upside
# =========================================================
import math, numpy as np, pandas as pd

# ------------- Helpers -------------
def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and (np.isnan(x) or not np.isfinite(x))):
            return "-"
        return f"${float(x):,.2f}"
    except:
        return "-"

def _fmt_pct(x):
    try:
        if x is None or (isinstance(x, float) and (np.isnan(x) or not np.isfinite(x))):
            return "-"
        return f"{float(x)*100:,.2f}%"
    except:
        return "-"

def _safe_float(x, default=None):
    try:
        return float(x)
    except:
        return default

def _margem(model_price, px):
    try:
        if model_price is None or px in (None, 0):
            return (None, None)
        diff = float(model_price) - float(px)
        pct  = diff / float(px)
        return diff, pct
    except:
        return (None, None)

# ------------- Inputs vindos do pipeline (se existirem) -------------
symbol          = globals().get("symbol", "?")
current_price   = globals().get("current_price", None)

# Preços/Modelos tradicionais (se existirem nas suas células):
FairPrice_DCF   = globals().get("FairPrice", None)                # DCF/FCFF ou RI (bancos) dependendo do seu fluxo
PJA_EVA         = globals().get("PJA_Excesso_EVA", None)          # EVA clássico (não-financeiros)
PJA_Legacy      = globals().get("PJA_Excesso_Legacy", None)       # sua versão “legacy” de lucro em excesso
valor_intrinseco= globals().get("valor_intrinseco", None)         # Graham sqrt(EPS*BVPS*22.5)

# NOVO: Growth-Tech (célula DCF integrada cria estas variáveis quando entra no modo Growth-Tech)
FairPrice_GT    = globals().get("FairPrice_GrowthTech", None)     # preço justo pelo painel Growth-Tech
GT_notes        = globals().get("GrowthTech_note", None)          # texto/nota do painel Growth-Tech

# ------------- Monta lista de modelos disponíveis -------------
rows = []
rows.append(["Preço Atual", current_price, None, None, "-"])

# DCF/RI (seu fluxo define automaticamente qual foi calculado)
if FairPrice_DCF is not None:
    diff, pct = _margem(FairPrice_DCF, current_price)
    rows.append(["DCF / RI (automático)", FairPrice_DCF, diff, pct,
                 "Fluxo de Caixa (não-fin.) ou Residual Income (bancos)"])

# EVA clássico
if PJA_EVA is not None:
    diff, pct = _margem(PJA_EVA, current_price)
    rows.append(["Lucro em Excesso (EVA)", PJA_EVA, diff, pct,
                 "PL + (ROE−Ke)·PL/(Ke−g∞)"])

# Legacy (se você ainda calcula)
if PJA_Legacy is not None:
    diff, pct = _margem(PJA_Legacy, current_price)
    rows.append(["Lucro em Excesso (Legacy)", PJA_Legacy, diff, pct,
                 "Goodwill=LE/Ke; LFE=g·LE (x10)"])

# Graham
if valor_intrinseco is not None:
    diff, pct = _margem(valor_intrinseco, current_price)
    rows.append(["Graham (Valor Intrínseco)", valor_intrinseco, diff, pct,
                 "√(EPS × BVPS × 22.5)"])

# Growth-Tech (NOVIDADE)
if FairPrice_GT is not None:
    diff, pct = _margem(FairPrice_GT, current_price)
    rows.append(["Growth-Tech (Painel)", FairPrice_GT, diff, pct,
                 "EV/Sales, Regra dos 40, Reverse DCF, Runway de Caixa"])

# ------------- Tabela comparativa -------------
df_comp = pd.DataFrame(rows, columns=["Modelo","Preço Justo","Diferença ($)","Margem (%)","Descrição"])

# Formatação segura
df_comp_fmt = df_comp.copy()
df_comp_fmt["Preço Justo"]   = df_comp_fmt["Preço Justo"].apply(_fmt_money)
df_comp_fmt["Diferença ($)"] = df_comp_fmt["Diferença ($)"].apply(_fmt_money)
df_comp_fmt["Margem (%)"]    = df_comp_fmt["Margem (%)"].apply(_fmt_pct)

print(f"📊 Comparativo — {symbol}")
display(df_comp_fmt)

# ------------- Ranking de Upside -------------
rank_rows = []
for _, r in df_comp.iterrows():
    if r["Modelo"] == "Preço Atual":
        continue
    pj = _safe_float(r["Preço Justo"])
    px = _safe_float(current_price)
    if pj is None or px in (None, 0):
        continue
    up = (pj/px) - 1.0
    rank_rows.append([r["Modelo"], pj, px, up])

if rank_rows:
    df_rank = pd.DataFrame(rank_rows, columns=["Modelo","Preço Justo (Modelo)","Preço Atual","Upside"])
    df_rank = df_rank.sort_values("Upside", ascending=False).reset_index(drop=True)

    df_rank_fmt = df_rank.copy()
    df_rank_fmt["Preço Justo (Modelo)"] = df_rank_fmt["Preço Justo (Modelo)"].apply(_fmt_money)
    df_rank_fmt["Preço Atual"]          = df_rank_fmt["Preço Atual"].apply(_fmt_money)
    df_rank_fmt["Upside"]               = df_rank_fmt["Upside"].apply(_fmt_pct)

    print(f"📈 Upside Rank — {symbol}")
    display(df_rank_fmt)

    # Destaque do topo
    top = df_rank.iloc[0]
    best_model  = top["Modelo"]
    best_value  = float(top["Preço Justo (Modelo)"])
    best_upside = float(top["Upside"])
    print(f"🏁 Método mais atrativo: {best_model} | Upside: {_fmt_pct(best_upside)}")

    # Nota automática (curta)
    if best_model.startswith("Growth-Tech"):
        print("🧠 Nota: empresa de crescimento/tech com métricas de tração (EV/Sales/Regra dos 40).")
        if GT_notes:
            print(f"   • {GT_notes}")
    elif "DCF / RI" in best_model:
        print("🧠 Nota: método principal pelo seu pipeline (DCF para não-financeiros; RI para bancos).")
else:
    print("⚠️ Sem dados suficientes para ranking de upside.")

# Exponibiliza globais úteis para células seguintes:
globals().update(dict(
    df_comp_fmt=df_comp_fmt,
    df_rank_fmt=df_rank_fmt if 'df_rank_fmt' in locals() else None
))




📊 Comparativo — MLI


,Modelo,Preço Justo,Diferença ($),Margem (%),Descrição
0,Preço Atual,$139.51,-,-,-
1,DCF / RI (automático),$153.74,$14.23,10.20%,Fluxo de Caixa (não-f...
2,Graham (Valor Intríns...,-,-,-,√(EPS × BVPS × 22.5)


📈 Upside Rank — MLI


,Modelo,Preço Justo (Modelo),Preço Atual,Upside
0,DCF / RI (automático),$153.74,$139.51,10.20%
1,Graham (Valor Intríns...,-,$139.51,-


🏁 Método mais atrativo: DCF / RI (automático) | Upside: 10.20%
🧠 Nota: método principal pelo seu pipeline (DCF para não-financeiros; RI para bancos).


# **S&P 500 vs Ticker - Performace**





In [57]:
# =========================================================
# “S&P vs. Ticker” — Performance Pack v2.1 (10 anos)
# - Usa 'symbol' já definido nas células anteriores (ex.: "MLI")
# - Métricas: CAGR, Vol (anual), Sharpe (ex-Rf), Max Drawdown, Beta, Correlação
# - Simula $10k e plota 4 gráficos interativos (Plotly)
# =========================================================

import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
import plotly.graph_objects as go

# ---------------- Helpers robustos ----------------
TRADING_DAYS = 252

def _coerce_float(x, default=None):
    """Converte para float. Se vier Series/array, pega o último valor finito."""
    try:
        arr = None
        if isinstance(x, (list, tuple)):
            arr = np.asarray(x)
        elif hasattr(x, "values"):  # Series/DataFrame/Index/ndarray
            arr = np.asarray(x)
        if arr is not None:
            arr = arr.reshape(-1)
            arr = arr[np.isfinite(arr)]
            if arr.size:
                return float(arr[-1])
            return default
        return float(x)
    except Exception:
        return default

def _to_series1d(x):
    """Garante 1D Series a partir de Series/DataFrame/array."""
    if x is None:
        return pd.Series(dtype=float)
    if isinstance(x, pd.Series):
        return x.dropna()
    if isinstance(x, pd.DataFrame):
        if x.shape[1] == 1:
            return x.iloc[:, 0].dropna()
        # se vier DataFrame com várias colunas, pega a 1ª
        return x.iloc[:, 0].dropna()
    # numpy array ou list
    try:
        arr = np.asarray(x).reshape(-1)
        return pd.Series(arr).dropna()
    except Exception:
        return pd.Series(dtype=float)

def daily_returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().dropna(how="any")

def cumulative_returns(returns: pd.DataFrame) -> pd.DataFrame:
    cr = (1 + returns).cumprod()
    cr.iloc[0] = 1.0
    return cr

def cagr(prices: pd.Series) -> float:
    prices = _to_series1d(prices)
    if prices.size < 2:
        return np.nan
    p0, p1 = prices.iloc[0], prices.iloc[-1]
    days = (prices.index[-1] - prices.index[0]).days if hasattr(prices.index, "dtype") else None
    if days is None or days <= 0:
        # fallback por contagem de pontos (aproximação)
        years = max(prices.size / TRADING_DAYS, 1e-9)
    else:
        years = max(days / 365.25, 1e-9)
    return (p1 / p0) ** (1 / years) - 1

def annual_vol(daily_ret) -> float:
    r = _to_series1d(daily_ret)
    return r.std(ddof=1) * np.sqrt(TRADING_DAYS) if r.size else np.nan

def sharpe_excess(daily_ret, rf_annual):
    """
    Sharpe anualizado (excesso sobre Rf).
    - daily_ret: série de retornos diários
    - rf_annual: taxa livre de risco anual (fração, e.g., 0.04) ou Series/array
    """
    r = _to_series1d(daily_ret)
    if r.size == 0:
        return np.nan
    rf = _coerce_float(rf_annual, default=0.04)
    if rf is None:
        rf = 0.04
    if rf > 1.0:  # se vier em %
        rf = rf / 100.0
    rf_daily = (1.0 + rf) ** (1.0 / TRADING_DAYS) - 1.0
    excess = r - rf_daily
    s = excess.std(ddof=1)
    if not np.isfinite(s) or s == 0.0:
        return np.nan
    m = excess.mean()
    if not np.isfinite(m):
        return np.nan
    return (m / s) * np.sqrt(TRADING_DAYS)

def max_drawdown(prices) -> float:
    p = _to_series1d(prices)
    if p.size == 0:
        return np.nan
    roll_max = p.cummax()
    dd = p / roll_max - 1.0
    return dd.min()

def beta_vs_bench(daily_ret_asset, daily_ret_bench) -> float:
    a = _to_series1d(daily_ret_asset)
    b = _to_series1d(daily_ret_bench)
    df = pd.concat([a, b], axis=1).dropna()
    if df.shape[0] < 3:
        return np.nan
    cov = np.cov(df.iloc[:, 0], df.iloc[:, 1])[0, 1]
    var = np.var(df.iloc[:, 1])
    return cov / var if var != 0 else np.nan

def growth_of_10k(cum_ret: pd.Series, initial=10_000.0) -> pd.Series:
    s = _to_series1d(cum_ret)
    if s.size == 0:
        return s
    return initial * s

def _fmt_pct(x):
    return "-" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x*100:,.2f}%"

def _fmt_num(x):
    return "-" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:,.2f}"

def _fmt_usd(x):
    return "-" if x is None or (isinstance(x, float) and np.isnan(x)) else f"${x:,.2f}"

# ---------------- Parâmetros & Download ----------------
ticker = symbol.upper()  # 'symbol' deve existir (ex.: "MLI")
end = datetime.today()
start = end - timedelta(days=365*10 + 30)

df_tk = yf.download(ticker, start=start, end=end, progress=False)
df_sp = yf.download("^GSPC", start=start, end=end, progress=False)

col_tk = "Adj Close" if "Adj Close" in df_tk.columns else "Close"
col_sp = "Adj Close" if "Adj Close" in df_sp.columns else "Close"

px = pd.concat(
    [
        df_tk[[col_tk]].rename(columns={col_tk: ticker}),
        df_sp[[col_sp]].rename(columns={col_sp: "SP500"}),
    ],
    axis=1,
).dropna(how="any")

if px.empty:
    raise ValueError("Sem dados para comparação. Verifique o ticker e/ou período.")

# ---------------- Cálculos ----------------
ret = daily_returns(px)
cr = cumulative_returns(ret)

# Rf: usa Rf_val do seu pipeline se existir, senão 4% a.a.
Rf = _coerce_float(globals().get("Rf_val", 0.04), default=0.04)
if Rf > 1.0:
    Rf = Rf / 100.0  # garante fração

# Métricas do ticker
CAGR_tk = cagr(px[ticker])
VOL_tk  = annual_vol(ret[ticker])
SH_tk   = sharpe_excess(ret[ticker], Rf)
MDD_tk  = max_drawdown(px[ticker])

# Métricas do S&P
CAGR_sp = cagr(px["SP500"])
VOL_sp  = annual_vol(ret["SP500"])
SH_sp   = sharpe_excess(ret["SP500"], Rf)
MDD_sp  = max_drawdown(px["SP500"])

# Beta & correlação
BETA = beta_vs_bench(ret[ticker], ret["SP500"])
CORR = ret[[ticker, "SP500"]].corr().iloc[0, 1]

# $10k
val_10k_tk = growth_of_10k(cr[ticker])
val_10k_sp = growth_of_10k(cr["SP500"])

# ---------------- Tabela de resumo ----------------
summary = pd.DataFrame(
    [
        ["CAGR (10y)",     _fmt_pct(CAGR_tk), _fmt_pct(CAGR_sp)],
        ["Vol anual",      _fmt_pct(VOL_tk),  _fmt_pct(VOL_sp)],
        ["Sharpe (ex-Rf)", _fmt_num(SH_tk),   _fmt_num(SH_sp)],
        ["Max Drawdown",   _fmt_pct(MDD_tk),  _fmt_pct(MDD_sp)],
        ["Beta vs S&P",    _fmt_num(BETA),    "-"],
        ["Correlação",     _fmt_num(CORR),    "-"],
        ["$10k → final",   _fmt_usd(val_10k_tk.iloc[-1]), _fmt_usd(val_10k_sp.iloc[-1])],
    ],
    columns=["Métrica", ticker, "SP500"],
)

print(f"📊 {ticker} vs S&P 500 — ~10 anos até {px.index[-1].date()}")
display(summary)

# ---------------- Gráficos interativos ----------------
# 1) Crescimento de $10k
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=val_10k_tk.index, y=val_10k_tk.values, name=ticker, mode="lines"))
fig1.add_trace(go.Scatter(x=val_10k_sp.index, y=val_10k_sp.values, name="S&P 500", mode="lines"))
fig1.update_layout(
    title=f"Crescimento de $10.000 — {ticker} vs S&P 500 (10 anos)",
    xaxis_title="Data", yaxis_title="Valor (USD)",
    template="plotly_white", hovermode="x unified"
)
fig1.show()

# 2) Retorno acumulado
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=cr.index, y=_to_series1d(cr[ticker]).values, name=ticker, mode="lines"))
fig2.add_trace(go.Scatter(x=cr.index, y=_to_series1d(cr["SP500"]).values, name="S&P 500", mode="lines"))
fig2.update_layout(
    title=f"Retorno Acumulado (base=1.0) — {ticker} vs S&P 500",
    xaxis_title="Data", yaxis_title="Retorno acumulado (x)",
    template="plotly_white", hovermode="x unified"
)
fig2.show()

# 3) Beta móvel (≈252 dias)
win = TRADING_DAYS
roll_beta = (
    _to_series1d(ret[ticker]).rolling(win).cov(_to_series1d(ret["SP500"])) /
    _to_series1d(ret["SP500"]).rolling(win).var()
).dropna()

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=roll_beta.index, y=roll_beta.values, name="Beta 1y ~252d", mode="lines"))
fig3.update_layout(
    title=f"Beta Móvel (≈1 ano) — {ticker} vs S&P 500",
    xaxis_title="Data", yaxis_title="Beta",
    template="plotly_white", hovermode="x unified",
    shapes=[dict(type="line", xref="paper", x0=0, x1=1, y0=1, y1=1, line=dict(dash="dot", width=1))]
)
fig3.show()

# 4) Drawdown
def drawdown_series(prices):
    p = _to_series1d(prices)
    rm = p.cummax()
    return p / rm - 1.0

dd_tk = drawdown_series(px[ticker])
dd_sp = drawdown_series(px["SP500"])

fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=dd_tk.index, y=dd_tk.values, name=f"DD {ticker}", mode="lines"))
fig4.add_trace(go.Scatter(x=dd_sp.index, y=dd_sp.values, name="DD S&P 500", mode="lines"))
fig4.update_layout(
    title=f"Drawdowns — {ticker} vs S&P 500",
    xaxis_title="Data", yaxis_title="Drawdown",
    template="plotly_white", hovermode="x unified",
    yaxis=dict(tickformat=".0%")
)
fig4.show()

# ---------------- Comentários automáticos ----------------
alerts = []

# spread de CAGR
if np.isfinite(CAGR_tk) and np.isfinite(CAGR_sp):
    spread = CAGR_tk - CAGR_sp
    if spread > 0.03:
        alerts.append("✅ CAGR do ticker supera o S&P por >3 p.p./ano — forte geração de valor.")
    elif spread < -0.03:
        alerts.append("⚠️ CAGR do ticker abaixo do S&P por >3 p.p./ano — atenção à subperformance.")

# Sharpe
if np.isfinite(SH_tk):
    if SH_tk > 1.0:
        alerts.append("✅ Sharpe (ex-Rf) > 1 — ótimo perfil risco/retorno histórico.")
    elif SH_tk < 0.3:
        alerts.append("⚠️ Sharpe baixo — retorno pouco compensador pelo risco.")

# Beta
if np.isfinite(BETA):
    if BETA > 1.2:
        alerts.append("⚠️ Beta > 1.2 — sensibilidade alta ao mercado (mais volátil que o S&P).")
    elif BETA < 0.8:
        alerts.append("ℹ️ Beta < 0.8 — perfil mais defensivo vs. mercado.")

# Drawdown
if np.isfinite(MDD_tk):
    if MDD_tk < -0.50:
        alerts.append("⚠️ Drawdown histórico < −50% — tolerância a quedas severas necessária.")
    elif MDD_tk > -0.30:
        alerts.append("✅ Drawdown histórico controlado (>-30%).")

print("🧠 Leituras automáticas:")
if alerts:
    for a in alerts:
        print("• " + a)
else:
    print("• Sem sinais fortes; use o DCF/RI e métricas operacionais para decisão.")


📊 MLI vs S&P 500 — ~10 anos até 2026-06-22


,Métrica,MLI,SP500
0,CAGR (10y),27.01%,13.48%
1,Vol anual,35.67%,18.06%
2,Sharpe (ex-Rf),0.73,0.55
3,Max Drawdown,-52.95%,-33.92%
4,Beta vs S&P,1.20,-
5,Correlação,0.61,-
6,$10k → final,"$111,157.47","$35,753.26"


🧠 Leituras automáticas:
• ✅ CAGR do ticker supera o S&P por >3 p.p./ano — forte geração de valor.
• ⚠️ Beta > 1.2 — sensibilidade alta ao mercado (mais volátil que o S&P).
• ⚠️ Drawdown histórico < −50% — tolerância a quedas severas necessária.
